# Wave62: N32/M32 production gate, exported ptxas path

The candidate and six-file snapshot are byte-identical to Wave61. The only harness change exports the resolved `ptxas` path from the retained compile phase to the candidate screen.


## 1 - Reconstruct production and run hard kernel gates


In [ ]:
import ast
import base64
import datetime as dt
import gzip
import hashlib
import json
import math
import os
from pathlib import Path
import re
import shutil
import statistics
import subprocess
import traceback
import time
import urllib.request
import zipfile

NOTEBOOK_BUILD = "wave62-n32-production-v1"
REPO_URL = "https://github.com/gwenland-org/gwenland-ai.git"
BASE_REV = "3bce8dd7b8aaa2765855ab927c611b54981f9241"
WAVE3_PATCH_SHA256 = "5f09f6147636c36db4e23b9d5f16a3384ca4c0b69679d5443d7c9a396387a508"
WAVE3_PATCH_GZIP_B64 = """H4sIAAAAAAACCu19eXPbSJLv//oU1dqwmzRJiDh4ia3Zlm11j8NHeyTt80zoaSmQBEm0cNAAKInb44n3Id4nfJ/k5VG4CFAG17MRzVE7ukUKqEpUZWX+8qgScmrPZqLVmtuRMI/mzmQ1NY/CYHJ0awWe5YTy0ih0ex1lGT2IcYVGB7Y3tR5Ee2zqE9VUlP50PB1Mp0Jtt7uGcdBqtSo966DRaFR73o8/ipbWbfZEg37++OOBODoS1p0VrMW9GSzhRygCqxVY5tT25iJaWOLsw+Wb8zNhTiL7zoxs3xOhY47FLPBd8U4T8LsdhcK/90SLqM38gPrNzcgSP5+9f98Umt7tE/1QPAhNG4i3L8WJ+EfHaIv3L4U/QzpRYM5m9kQsrUA45sqbLBJqZtzHNaPAflDEqeOImCD0NsXY8Se3IlyYgUXPDk0Xvvi3lhc2RejjtYMWUAPirdsWt85NyJ5YotY1RODf4xh1TeAINXhoXZjelKYIt2e6dtDYgY7RJzp6GR2cXbbvxHSsUOAMPP9eXFye/nz2Wrz5IC7+fHoO396fvf/l/G/AbngC8kg+3JsSoYnvhSvXmorxOrOcx+I04es08IFb/Yem+OmnD7QuoZj74vX56fvW2F95UwUJkYToJCF6RkLw31/6o7a48E/F51CEEUiIK95cAGftENZr7a8iUfuE82655q+waCcn4uV/XsLAHL5QVySl16LWf+jDsuhaXcT/TmA9eb2Aa7C0lj1fRMRF7vYRGB4tUCxB8My5a3nAyNo88FfLN6+ht2N61pHRFJE9l789M+rHtOZCmG24FrqWOzKvalFbNITsWH8BC90QC9OZvVC78A26vzCEojSMayFqvmeJFYySlvxRMrDOFcjw/MdI5tOVlyFwfXU7fnQookhm2m6KqQqk/taKbMdCXgkYVSMeUxNZHwrtBRBq8kdDBTLRvU+sT1a7ozZVVTTwQ4vX27HmplNXxEVkzhEHYI3nKzOYgoBFPjwKxKVf82DB6rRGpGOeD7IAbXGRQjG21r6UTVTIpTklQAE5BYGcRCxsuDzn1ueVHVi0oscCxegZCABID8zP9uAX4Av/hs8TP5yg/tSQZrDyPFCEyWLl3YZEy4zEx/Ozn968ezd6eXr56s8CG9eH4mH0OTx6GEkVI910V2EEgxSn79798ur0EhRstYS50VJXHdJQWOZkIdatV5enYuKYLigYKLjlzJBLZkTEXB8eJBGhKe4XuFY7jAcZmON3zXoADnInBAtkN/AGlV7cB3YUWR6oGnLjHaHoMQiZPRUTy3ZqOJUjUD/xAphpT49ADOqIyJ2uYOUKuecFAun0WGhtArACwjWoy8sifCk05dyDa1uf3ORB0bIe4UIVxoLEzlIWg40BvAc16BotAguyRAo9R1mfqCimKBh+YIPUmo60I0wnnpPe7mk7zgnXoPXP+3cglDs7tMcgCAqIGGD13BnNLdcdua45+tyvHZDCK0szAIxVViA8y9H957BJ6trTEZx7RlM1SFupaWDNhTIGsXwW/GD0/zTMXp7h5dkPupa/PAayz4LpD4YB11t0neznVCimY889ASikjPsAeAB3KAjXQ2x0hCvFUJ0xlm/B7kRWeNCQTT6Zd5bQj4Vnok2C59lhZAWhmJgewISAB5i8VN7KHVv40AVaR9MBo0a8V5hUdmKj+XoUgQbyN28cfwNFHeZa88Tw1oPfjL+G6de1P0wG+uHsr5ew6LFNOZIqGfswoT+LwJBaraW9tBzbg4HethzfXxbHN0Yaba8pv6leblC0CPfhhBrA58btJbLg2dID3Y6vb18MlOCti0EKi2gLfZd2hF5UCTkjpvYQXoHYMzVJDodaUIIDwXScKYslSSXwU22KKxLO6yFJZ7/TBOM10JtaN5HOtA+to0ZdbO96WHpbp9sICtigIFGuf8cu3gJw9XvwRKzABk0HREADSJ4prg/gNEAwA4MCgD42QzA/CTUkgCgjncQW+eHIMR/6oaCiNBKio9EihHLNWxAM8q0SMrYHjtBqQnwaWw4OAEhaD8A+Z52HIny8Y7VogKkRTMbTRuwywQo4kb0EYACvFs0cDgPIBBYIpCX69diZhWeARpG1zdiGhFy6fK1NwwvK6JowbhGu0C20Qeiljw1CO0f7btqOALSVgwN+y4VJ9G8CTYCtUlDDhVOqoPitO0wcPDnLEyF7gyHoGvIRK0e5t6eWfE5Gc1OK2lAyPOqDkONS+7NZaEVMAeaohCyQejP9yYSScQaZiTB8gCh2MkOMx0kizw52SL42PXH7YMONwfIDsvxJISnMQ5I2zDwXNY+f/cgEjWb6k6kMt49svTEytXxQ6yw4lgwKrPdyFT02qk4z/ZnD2HA1VsKE7SB8TdbxeFTx4M2HYrPka9LK9mQrPXe/aww315DgEBwGDokoSgGhPmgw5qRSjYwEh0R5kGiUiglNxiiREKKP4dXInnInUFOpA13ZSVeHJZ0wLiGYVI0BWnG115Mx1gY7NVqtab/Jv8ixTe4ixN14gTWVZLg33NY9/siMBQYxd/wxAhK49uCXRwGgzPTOhLAyhJhPO7od19kdSBVbo3FoHeRFKmuOHwN6ctvoZ8eSvdnPDTVlvzZoklHb1m/AvYebGsrWjOcAnWASMxsAvM7c7XabA+DuwICPHHdZdohvukY/1S2kH8JJnrA0gDIWpWSJbHp+dvpagIGxIFiIHQkEcIrI3Bf9TKCobLJWJ0FWtwBRHN++EJh3KDA+7WwUOJR2NfoFBlA/2bu72VOGnjzSzU68ILpe4BqExeKUghHJNheTBABH6FbXM3jxbcT0vlGPFTXhoRGzQSuIEd+Ta22UPghXuvRRGCOzPIE/A16NBgon3ZqCGLcRpETZP3jM7VjU3sqEDXka9Yxj8zGwXUu8+o/z87MP21zR2/FJW9zZJpl+clkTl1oRlwsrIWY9LCGYsRGqI0y/3VrWkh1taGxO162QLrOHNIWn2PAQCulhlY4pnE2dGyDPLiB4pzMLXEkYze24ocogFgLNEMY2Ac8kjPNPcD/jN+T94gTEc3fUjTtZR7k9a8t/JQ3UsgZgmpbKPOJVIXeaRMNIn/Hjd3xdjCGIfv/+dPT23S+/fIxvPluqKjqkjJFMJpnAFfq76vVX26pJ24ba3docPPpnC1W21PPNEOhhnsoMG8VRw0J9lJQmSRmPk1KJlLa5FoHWbSZTLd7sFcKaZClmWlOOsXBHz8Q6Bymzj6UC01pJd2XJQM9OiiHVGGYwSNbp0/mbyzMOMjRVbao6KGRn0FQ7oJDUgDKmo5en58cJUmO0TblaguuJ74IXY4laeGsvlwDc9wsf/bw1+jctf9YKTA/MCaWV64Ij9VhkgJOJwFz87cMrDE1acWjCycoE+hsyZjoGPTMxDvvlw6uzJqjsKswmaF3O21HcIO3ChjTRosRSlwWUl5QnFG1RuxVtRQEegLkhfKlvIdTLieQmIRUJqV1F0dUigYKgtmJfJJErkoFFqcsD63/lTdqY4bwuJ50V3BLSOsvrVtIqk07hlLFqW2Q/5lCK8Sre4pisggBXjliobBoroxfLppqFGWcTZrgdSm+sfwQ0j8MJuFzXFTpkMAUzMVs75BerUaD9VWQpJZgs0dcJJviS+kvnK89DTrO71AKb64P4o8kF+4V6AexEGyQVos1Z1dt4NTZMrd4pcRoSmTilBSdTzlBhaLTbNdDI0S4ARR4q5AjUTeXf1P1N14iHBD/Big4Lvk5yF5yXooNCEQP+1LU0IcKsipXXIPbrnettDTqyAdsb3uSjeevqLvPWNuat7eG8+zxvfZd56xvz1vdv3nqb593ZZd7GxryNPZy3xvPu7TLvzsa8O3s4b8Y1fSdc627Mu7uH82ZcM3bCtd7GvHt7OO8+haAGBM5qu2ziyWxS7/cM3UzcLqtRPhiT0bhJWxeuZUWhwLRy7A7RVgyac86o+NAT9xItaX3HZqCEa29SjOH+p+OGzcyWSrkb+tC1zXCeZiAD7WJPzgnjh1ZClxOq+KExxw1A1D5wvNNNJY1iEMlg15xiJiaXUuLwhX7EmbDNrGycdmoKozgKvjlVOW+HAsTpoih2x5A/V3TrGprM1PYjLRoGt0mc1qTNnZYn9BtS4rZfhixKn9TjgmS34znGc8tHcGo75YEueQjr1EGx7RpNY5BwUf09sVCrwEK9Egs1bvtl+I2ci2NfUPQYp17/8uFs+DtimlGBaZ1KTDO47VNgWrcC03qVmNbltk+Baf0KTBtUYlqf2z4BpmkVzIJWySxQQg7aPgWmVTAEWiVDoGnc9ikwrYIh0CoZAs3gtl8ocxRPVToKgRVJh4JPP3XUQeqUyXa458sZVnMS+GHI5+foUFdyRNEUHVWTR8fo2CCedeazrKE8cxrykcV7PBKGxztwY8ifCU7xRrZrhYp4Jc9S0NnAZ/2TEzoX+EzX6BsexfjhBJ7bJFL5s37yhFPxHKUifj77cHZOJ/9qlmtHI9ySUZbrevb8sq61Zr4zFSsv8B0Hz4wsA//OHDuYT5uvHPDTxXly2uofEB3yHtdAU/FsW+tP4h98gPm/S0jVdVVLKOH8+JjE0cV7UftHp/1M+JPJaml6k3V9SGHF2PImC3GF59tay4UZWuNrcXr0UkytCUglsH1h4QkXeVIUBuJZUWtsmRHvW/U5hOMD7XjYHC++/MRHltOjzP/8Q3okbgPapu8YnYK4ff30l3j89JfYcvrra0fx+uogPf1FpyUfO4v3CCFV0/p8qK+M0O7nyNS2ZuQG9k0HyTqDHu7hdw29OShhfSYfTKIJAxj+sUX/dLfoe5pKx2K7vVRTs5uNm1ZQo9kZtKeTYywdxP/agRo9TdzE+ySyHad2oGG8mZXywOBHlsgRH1ciERI1KbF0mpFmWS8Rq5RWXqwSSqC+ZZQ2WC/J8M/OcFPIEEfiA1+lPVk6jJgdYODjPBaw4wrvXDPDjKE8u9yjVerrm9m70/YfC/V7WqgOnzIfDAoLpf6xUL+nheqSh9JXO4WF0v5YqN/RQvX5j+r6/W/aiG7v4VZ0nzLojUH7W7aiwbncw5n3eObat2xGq+093I7uD3jmxrdsR6vtPdyQHqg88+63bEir7T3ckh4wwg3637Ilrbb3cFN6wAinttvfsiuttvdwX3rQk1PfCeP6han393DqAzn1nUBuUJj6YP+mDn6InPtOMKe2C6dq2/s4eV1OfjdfruDMqeo+Tl5inboT1qkFf07V9nHyEu3UndBOLbh0qr6Pk5d4p+6Ed2rBq1P30KsDXZWT3w3wCo6d2tnHyUvAU3cDvIJvp3b3cfIS8LTdAK/g3am9fZy8BDxtN8Ar+Hdqfx8nLwFP2w3wCh6euo8eniYBT9sJ8LSCh6fto4enScDTdgI8reDhafvo4WkS8PTdEnYFD0/bRw9Pk4Cn7wR4WsHD0/bRw9Mk4Ok7AZ5W8PC0ffTwdAl4+m6AV/DwtH308HQJePpugFfw8LR99PB0CXjGboBX8PC0ffTwdAl4xm6AV/DwtH308HQJeMZugFfw8LR99PAMCXjGToCnFzw8fR89PEMCnrET4OkFD0/fRw/P6KtNPKDXUDtar6lpxtP6+6Z/9pnsT7+3v5nYiz9p+vTH3zT9K3BtL/6o6dMff9X0L8C1/fizpk9//F3TvwLX/kf+sOlfnmsVrIFWyRrQawWg7ZPgWgVroFWyBvTSTGj7FLimV7AGeiVrQEOHtk+CaxWsgV7JGtCToe2T4FoFa6BXsgb0x07Q9klwrYI10CtZA0qEQNsnwbUK1kCvZA30Prd9ClwzKlgDo5I1MNrc9klwrYI1MCpZA0Pjtk+CaxWsgVHJGhgGt30SXKtgDYxK1sDoctsnwbUK1sCoZA2MPrd9ClzrVLAGnUrWoNPmtk+CaxWsQaeSNeho3PZJcK2CNehUsgYdg9s+Ca5VsAadStag0+W2T4JrFaxBp5I16PS57VPgWreCNehWsgbdNrd9ElyrYA26laxBV+O2T4JrFaxBt5I16Brc9klwrYI16FayBt0ut30SXKtgDbqVrEG3z22fAtd6FaxBr5I16LW57ZPgWgVr0KtkDXoat03eNC3f6lb2ysepPZuJVmtuR8IsK2/v+lMlCMV4+70DqrcltJk+0bq6oljjabc96wi13e4axgGe7XuE8kGj0XiUOr0ksEtvZu/KF7MvV2PBVUHFW2p8YUXit/g44VFcqdT3LHzDIr5dEiu6mxEVCxKzwHfFzc/vXv3H69PRudbp3gyxpmja/eomoXp8jK+IHFmeOXas6c21fFk7l/Ia+77TjKviHIlP/JZF+UpFN/dSxXDhr5ypmOBb3/EFlSKtrRxivWt8hbzWep0Sk2WVM8PPDfvn8zevtddy4I3SgWNhVm2aHTq148vx4Gk2sxFI7bFkZXIttJ3VCAS+cCPwl1Z6kf+MS6NigCosEJ30tN2lU1wa/GfPiHnZS/jPWga2Fzned7XDKxaFa26HddFs4Be+gVLIuYga3KEVjmvY40tJ64f1YUr0C882V9kWSyNP7akZWVjRzQ4zpdvMwI4WrhXZk3jFkP1Yq2qKlSdzxAJr6ZhYynFriVxePC4Bdw+EkyK5b+jt/+swR89fRi0QgZUHAsOF3AN/Kivezh1+t+ccS8rBg+mOhRW6xaWhpGQc4DKvrDgRrmsqdjgKfdeq1cXz5/DM6fGx5d0dH9+ZwcgPa4c5MTqsp82HKU1YKknyt/TitsUC4ZV1fqkiedmyHWaJf0mX6pfbWqmoEOC6ZjN/JaC3vubGw6PcaCeFGkBk5VhgFqLRDBYFmVo7nDt4cwSweVj/90K/VPC3dY5bbKPAGrKtN95NetJ7ODtU0bnT7TVV7XH1wcL3CrLgIOZiqv0XjDEx68PV2EVQd3KVm7Nog/ZAyuq/A9B8XtlBWl+cCziH7qjXkTOhgs0bAITUqFayd2cHvofv/JRSiRg980QehWrPcQJ1fLUt4k9WsGhm3JovfokLyeJYXppYG3HK8nTztyt8n28TXw18DfL+V/mr7V2LH8WnK7hMv/zn5U38Nts3Hy77KbVcWcXae01RxaUZ3oqXTZ5wo66IC9O1QDWtAGYdCjPMmocLGCsALJaT/9wftUF1zJtr8f/+z/+lZwGrW675KzzgL3BTXPin4nMo+BXIVILawfdsrnnpwS0Em9bo9GPbVr72NGpUrzD7xmSyC1g8uS9q+GTHMUH1J8ulWECLVuS38BONz701TQklqCnfy4yFw+5lsfhYeiaW7cD1upJ0i2vxHSWiIm6A06Op7YpnMIKTE9G+aYob25OXwCmJr1F17x9OQBBvaKQprWDleTABelN0iCb65uP52U9v3r0bvTy9fPXnm3qTxQ5f8XxzdBO/5PkmfctzSgvLeNKjbvi1zyAv/ILqfNFvYJb1EIETSK/hNQOL3nvaFB7WDEypYXEWEBQQhbMg8MEFAjxEDkllwLco+1FcfhDXnjDa91IF2plPiaXB3o9MufJMU2qPTDkz01OCBUZwM4hsxCxsP7ahc1qU/Sh+OFWAR/dvaXlY6jRGmleXp+HOfCso2MIMR2AAEq/r366QA/e1iWMvl+vj48j3R67prUdmMF8h9IT1a24Zgw9qKFAALa2xuvX7+Ccp3bYsOFOubIRP51YInPyhBgL4s0Mz+VMWiKfWeDUfmSEY/mhkff6ullnipgCX/zDzcOC2lIRNQcj5LAWaGRF5hCaX2NS11ttcVXUk3Soj/V0tVccNmuSfkhq64NfAGNmLBZCEkCZ9d3U9Z8pLaP9p+2jRC7dMII5GKCGZ4wM6M7VRU8yaYlQHfEfTkDexClDduAKyAtBV4+rU3XYHCwt1NV0WL91uUfHfc3cVCRtAXrzAb6Pk22R059ubngW19qq0vs6sQPwC8AVqXgiz6nMVVPomi78zrwmiSFER2mWFxoQOulsKG+3aC+BQDZEaxOQulkAs1Q5CCzGpCh/oKie/wKLQ4Omt4GE954V+w+Cke7vdSZbeMfrK5ISQZzwkFEtFUYCzAuBgPZiTyFkLNePdZuecd/2AA/kL29iRXGZXgVlU3+ycYVb+Tnvj9wwbM3fqqVeWgNlHfB+/eCmNbotCFXJjjjPVoSey2gFCeM69YO25uY5fCTZAR1FtG0azo6JcA6iCNxNGYVakMwjy8fKvo4v3vY7ikgMV1r7/7fu6MgGbESGyFW9/SW9nVTLW7qQDjtgEV6R2qEQAv6BT5DkdVu2FMQqWO+OX3FtTxe17/Vu1q4BsQTMHUzFK2Kf/0FXOAk4a0GUijsAamyF6Mljs15tjiARSFPll4biSF1UUfemUxQgYl7RACwddwO6syQ9xVyGHcGBpVTCVMkzPB2IyZRAFa9CfeO55+VFmYDWBd3d2aINjLBRuDtFBZtmpPEXtsL7R1XpYWpOodkjjpn45OMYR4Gv4RzKQPRHP4zFcKUo6tuthyahL+qQ9FCXbJyNlmcclknT4bAJLDbBwmJE3tV5OIPPsxwm0iwTyj89ImO3F+TxMwI3mazAQBAcFaaIqfgJXE2zS0rQDctOnvwKAgU+D695v0duKpbBSoZOxD4G9zFXlqJGMBBYOg4zcHawWIF0YYRVARE0pi0jX88lmmwE/ltqESimPCrp6uJkDzDIKyy3oWhm3vivRxtKcJLMpN7VTYUhOnJKfR1UrkFnEDKA3c+xJ1JoFlkXojrrHQSbwwTWXOWI0fzNaBaxdzNqkm3QYpuhSgvtJVROpwjGzSFwurBy1yHKXETbQuzxCHltoRnY4sy2seWJTem6J8XcQrZUia8o4g2U89HYvZkfVLlz5o6zP9sVM326d0xlYSWNTwdH8jE0MmU7E/7Imx8eYZhpNzKU5saN1Lb/yyFIqhDECnAShbCtKfzOnQ2y35/K2sXmbK1TC85TlKlzUarWEHr0MnIpl4Ne6OMIfzzZkbyMjl/nKRCEaj0YrCGwxS5BLQvF9CBVWy1p9G/pgE8fykFvoKdMKJDLH+hj5K5AGrgqPHSh2PtzkEso0c4BzlAUWoWSN+J3l2PAKtx9gicBr0IzrMqZlxllL3p3eyNKpy5DhcX7lZP1lBnooiQnWyDNdUJaNiuykY8u4XPvtuKHKaqR5A+hHJqkOemCeNJlxvXZU3zQNEnG258EOqcPYDALbCqoC1qPFfLIyr21Z6jyHyk0rxwfxI79eGX7TvlL/eCT5W1rG1Stxc0qQ9fv/3f6+Xrlt8D3M/fDV+bufxD3l8saIg7+C/YClHa/FMnoww1yoBLE1wsEF5m4d27PqGY9DwQthrQ6+BUTHAUTZ8J38jr9jdOXU/y6+czDza4YT2wZ/D+R3224Qp2kKG0HJZbkHNB4P9IFqKEq7PxkY2mzrHlDasbD9k97i4vL0Egn+gAsc1d+N7kcg1rUDcrEpl29zqMHBsM+7DN+jFb8zPSy4SzabQ4xcO/wF62SFtu8pTO8T58diJUoTJmnCDaOYpqwaJkhYuNUNxhk17wjiixtK53C2RF7FGkA3Q1DWyAZoskwvZOcTgACdTg/UGrUKldbxQ07zyzILruX6wRoLlB2JGpUaowpUwpyj7ERC09sY0twv7AkaYowIOA1Aw714r2BJMujKxcXyfXVd6z/el/J/NJXXlmOPSZjAZoP3kpQiAwMcuIo49dCHtmYze2IjfLA5B9xIBNr27uCGJVkDlHjPzPSQI+EKJjtEsTaz2yBYzAyzmJJfwBxyBUDq76nEsjkGt53lZaBhiIQfwKNEYNwRevO1g83CGBg7IYuXVlgnnyJNn08CUzo3uYxkUumuSIudHRA6lJUWFvbCUQKe9rN1vygDfG+uKRNPJdZCJU8LZn+ryOQXb+BgR9TUOLkHpqPWr5fZm5LNrmPyQuWeFAQPJFlsCcG3SjciS4lFCzD083QHC1kkgwSKuIDUwxEnKI/WJC+vLk/zW2YZasBcgL87mRbwAxtcOgh/cTUdq0WpAbQ3Yx8iMXD6WiA96PwppcTeouGjcnbmQibEeV8WaUf3Pimk6VD8RrW5iemRXz4yE91QEO7/so44RicqxBzzAZ8RR38gwiCO30+BNG4wF6nR8m3se5SuldyDXwUedsjmKkubxpkQ0NTPIIH34aRJlQ/xJ69AU6ybUj5tD/DJa5ZT2nQ1NjbkMmyhN/N8ImDAhcG8BSfkQXbvUa1pzULgWuatPXkapXQ5dkU3AldETP0k70NpQYTBY4Zcy4Rnc/KEq7S3dUzt6Sp+PK7fjzz/0ry1JPZimgBQxBMUZAQ2jQJQaMXZKzMCUMN8HyWpSomBpKwJ0oFXFAjRwMnKcL1LthtoA4FzjP5z3yJZbJXTixPprGoxNq9Q3RDpJYCDxG+YhxJqMcpLg0i2A8eBuMuV6Wc2DNHFb5smY+fBoSnZNrjG1wfH1qni6ErXFUY3M13bsSGkPsR8JpgNh7ABoiq45TiUhI8tf7rvX0ous9c2tgjM10sLhb/06bIgTQ/ls9/vNdV+knr+ebl6708tZzPzjNX3ItrSMUPmW8xOO2QFwz0bf4auHCI6FntcUbz862o6t4qG4x68HXCN/BlmGH4rjhK9Rc8DL5ES2Cs8lVS2Lu/8OSUmk61CTAsT/M183IORzHPZtieDgihB2W6TNDC3jgXcDtaxCZwsMHk4ze1QS0+NbX251MCDKU/HrKNNYRY6cJKNBwxTPCoPq8QnGCBcAZsONrWUHtlHOWPSXRQxOaR1Cw1aDbz0OCJ6p4mFLf09s5TeBDALoiTw6iZRk7a7ZtY9kJSGE/e+wvqQ8Yd1WqYbS5iHC4ZsIYCDdYM1/q7EvICTcKtkzypJxwGvLIHTNc/baoNK2fxFQHBpbesCSFjsMSwXN8xVgC8EY/97GB3DhMFaNIV7LJ6jVpjR3+EhrbKHYMrHGtHCXoXRtWicSHnka7XnrnJfh4haXiQhpYvAc49y/MPy0X8j4WQ56iUzTum75iQh77G6dQ2g6irxLmDmktzq4yslZMuYi761Y65BsMAa0/kJ+g08RfB5atvCOdouOFoi8q1zAd3GDRnSdTqTbteEkG5qdLu9cfmxvs2uuaBu8yYd6DNUOtEHH1rOTZeZbxm/j6bW5xUgGkjMFLf3wCaAJaWpxU4LHYE5Stze49Ljdx5iyxLjrey5u7xvG4M9U/MxAsnuW+GWFbm/IQIxKi2XwM7YQwQKcujNYI0e2MRZTS0mB6EkwhwE+k7IDtRSnpzxfK+19AEdWv6shV4riAHW9sqdArN8Qk2Ah3+7QnZeHzQ2GCYB4et8a8RqSTmDGjuUt3VMHMyXK2jCah87pl+kCjHspEFJBhcyx8Eu3r75eAzh9B2W9saNmHt5ggj3+SFa5IM3LTx4I88D5FJw/Mxhcv4nFvJ0L4/VhAuA1ykDl/auqV2JLV0j/tKhL9k8Tg2TdbjxrbXb5dchSM9epz1BLLaW7ZHL9SXrgHJRe84sfc7nk4rDbqIXElr1dJp00ulIei8cjHJcY3lztPupeV15ID3Ikb/cW56mdFptpfNS1BA5cIRMhbasXGtq47kKow8wKAN4UrxOu6kaoHk9+NT/e5qHDzEdOiGQpCzG6FRlRHZsLcw7218FtPSmmJtLGDjWE08dbkrz01k2pChVdQYqwocu6T7+ugJpQcGAy/gUiBcgBlt5eBwStNGLOLqT6X1wK+R0WzFLyW/MZmnooJMPfOJ4lvYPgd03pXvHN3yE0/OZYLh0AE3pHO9bgckNOpVFPkt8zOeGQ8Ok/PoC1MDF9HOSbkL9YHLwWPAA+CBMrxu7uHS+B8/F3BzdrJY3TSzxQtem/r0H13z4/zNc1vjqLfx6d9NkisB8Uxjt1sV7PCIqTpO0SpLaiN1D3NVnig2giF5Tt/sMBykbMj1wR6Etj6bBo3H9O3KG71HN0zgndmAj378FM/FM5pyQxQ8tCqJzKxFYLm6RgX/sBxGI2jEyYMvMmdLm9FFIcizYnLxMhci0iU165YcSk8FXvCkeC7jZMAtDnCAgIjY4AWPN2D/NpMGYGqXhucuJmkP4JM0hUzBg400Qp5a/TKxFVmLjo1VgYMwlRhu4LhDxWROT0xImJTFvsucDIHSKjxAMWa0oicoE5dk63BXn7S93GYUKpU9k0l+m9eF5EIZbwcTGKeLBUV5WeQhILmfcj05umHyOZ7aCxmj+Huk+lEdBMNMSySWQ3jll4eTOO1nVlkwiSbvZQrsZQ0oAzILJQo/zs9N3o4s/n348uzgWVzWJ+dkPcLINPCF6xe5CjWCcftABEHLWP9ODfAyafj1GcWMdBI0GAAzknwXUVHzv10bPW+p5dwwyKPU2PgLqmQE4BDgreXhGioKLMeDB/weFdsE857AAAA=="""
WAVE4_PATCH_SHA256 = "8fd9de6b6e2a41b84e73835530b3018bf037e621a6110737bbdeb26139021bab"
WAVE4_PATCH_GZIP_B64 = """H4sIAAAAAAACCu1be3fbNpb/358C8Z40VEXRIkW9k05ebjeT5nFst8kejw8NkZDEEUUyBCXb03rPfIj5hPtJ9l4ApEiKUuKJp+3OWZ+mtAngArjPH+4FPX86Ja3WzE8JPZoF7sqjR+yaLuOA8aMJC925kXAy2dl04IceuyZuhw3brm0YQ9O0aHtIzHa7Z9sHrVZrD92DZrO5j/bTp6RldnpdvU+a6gmvpiFZUj/UGqT1HTlhfBWkj7WGTp5H14+9m5Dw1BuNWJJEyWh0jI/vviO/HJDsJ2ApiZNowsgT8itPo3hEVh3r12IX/FkYNE1DJ4muuCO6a+V2/PkG162TTzpZuDpZw79oleokdOaMelwn+HA8f0koxzl0EkfqLdBkibNY6wetKlExiKeJ7zGdcJcG8AjTaJETwTXrB807j1vSa4ezTxU6JRqNzZ+3483vR0fkA02WhK1ZckPWNPFpmBLtz6/OSJO41J0zDvyPgxUn6ZyRhNGALFgSssBQMuzbUobi+fUyvD2oysljbuQxIa6KoO5dSJ9jdFk0dxXLZvGNPxVEgJsw+A0YRhKF/t+YJloVd4fKQob3YyHTKCEO8UPSNgw/ZWD/9baxm+e/m3F8vVlUDKIog9vPiEMIw7Y7utkGaYhfzHsQB1hfCtOsZvN4lRrkRRTyFO0voCuYn0RglLhJMMTQZQFHwaEVeuDZWQKulHGjRAz4DP/8ECRLoumUgz9MIzHEna/CBVn6nhcwwiNp2p4TMFwjveHkv63hwKi40og7MAKcKdJtEk0T3D0iVoN8S+yG4GvPHm8kg6O4H84C5rg0pq6f3sBobUI5g/HZaPjVbCihjMu6iY4Ld3luCgHCf+2Le3TfZr1u4i7/Kf007+SysXeZOfWe+u6K2e0JL2F3+/fiJaqeQq261mF8qUT+RVK5u2S+UjpbEqpIaaekvBo4xhP3SEZTrl4ZcXq9wU317QqY0WnH6/d6AMz69mAwtGuB2Q4KJXS2ow+qltXTe6QJ/we/9/TpATFSmszQyJdOvw1/Us9LGOcOh12Sng1hqwlu6F3IlAtrYYNH+Jwm+GCzJQtB4jFNUj/1oxBeTm6Eg4oTNvWDgIAmQRdoEpTUskAPwY9xN0oKjmXaseQrDsoaBNGVJNbpkeeAU7yVi1SgR0JTwJwHTYNdgwKHxFCrMWjgz0JiE2MywA3lOnx+AS7tAP1p6/5+BL1Z4ADHHFj5iNyc+xek+YRcw9Mg5EfBrxGZgVISl/mBFh5Z3V6DXBN4YJhAQzDufVlCyv2urQ9Azv3uQB+iA3n984cT5+W7t8cjMWEhWohggFH1PL1AR65jNAGukVSKzuMYcgDwu2kgJTv1E54eqPCkRjZNSRNHcqL5KSfRVQjTClJX4AWuEh/JFdTDZR7YZ8bHxdrBLhKjKG0bwxNDn3jFrn2ekskK1pUwEiLARfjqkRZMveI0aBgHLaBVYryWYxgMVsh80xrkzCendMkyZY4DGqK7gtUJOlHiz/wQ4LFm9shr/3mmm80dGgnTN+80vU7AddOl72YrmNykQP+JIFO2jW9tMa1RNRlfovgAjRiYUxCrQBY+F7QkM4Vg+TxKxJlqGYOIwogEUTgTjOQsWTNit4e9jREm6hjx4uyZVNTTlM4YsVzDHJH3lHNigusFacA61hbRwP7Jh2cn7wm4dzD1GxScTtwI/DEHacNCQiYNR/pqeBMBI2BvfD4NWmnCmOQrAzgResRjgT9hwF0Gqvfy1c/HJz8cn5IprF5ISRASLr4F3IVNS+9C6ASAlk6u5j5grgVjseSSZHsL1tZCpVybxItS0B4kJa2mb+pmF82mDw7SrtoN/hjg6eiSGACTSOwo/dcrbeDHYqcc53Z0yGKWjINZ+1S0izAmw11Ng15uWamWonYc4BlRRXcjYTPojFr2MH48+G5cfD2BwQ+Tx3blNc72cPq4Y1V6w9YfJt5j24b3ct07HDAYxrnZ6wzsi7FAs1KvzgGAoJpd7B8M75LzTg+HisFXNIllnKEBaswkAdVxKXoiubrAk8wQsoEFgic7j51PF+P6Zks0Ly7GSvbDtoiM/aE69lYlvxkvuNURw4tC3p5IdLTzjkrYW/0kn+VyhWwvFAKvUuqpLkUhI7WD7LhwJqxAOW+wH3TD0jcCkg9nKbilPZ5fnRaW0TqbEHj00E2p7xk3ik2Ddk8EloFpbgcW/OHzRI0ewuAEltwdb+GtTJy+JwdBDDW4GGS2cVRfJx1zXCUIU6oeWySBYOggSS6VarMHE7MnQhXHNatQnk6cZzSpiI0tCh0908eaaTdhoEykWbuMHJKMS0TyOLBZjiTA54GyTruvKxWwxjv2sQkL3xKEadFUA9VSSymwuKOrBSHV8c79qLAmV6N0zF2n1EgjYxZEExpkptTVhb2NN1Q+Edlj36ieGGUVRi2yUULVhm2RiLKGIHFhkc/Ozt6eOKcv3p0cO2+PP56BxhVfFZRwQhMB10kbraOlyCM8klEL0BqPpumSXosjeearwDFtzONC9JfM2zu4LLxaIrldSVMHfrWn338/aOPPuCrLlh9Oq5ZoCWn1xtmO3zz7OBJc6rT7HeQSPIdFLr376awA98BpiHM4JzSLkTJuaGgzrVXoQ6xfNgRCwEjJaAIBN4FzAQxRrehQFHYMQ4ibHni0IHIXQBLYnfgsMcgxbBxxWooHLdRkBASCL4gh4Wx4Dqd/AR/lugBwRRMEHnQSMCV8ARRFSgPOLJQXp2thDqsFmGAJuCxlYqmITwRci1E2EwZwgKT+knkK3KnMC5hlzIRP3EZ7OdKTAO5LCVXAWz3Bez90GGuf+8gtA05VAM3wBFJ7Vi8jFcAoUl0GXQxznfbQ3KEuX4tkyD8PWGqJo+YqJCN3MLQwAsFzCDvZuYMSsBneO7CJvwrYxL8dsOmY4EZR4mZ7uEfif2B0U9d3IPuCatwr/pEMs0yhYKbVlxCnlmF/WJwTfz3Oif8f5/x2OKdjWaaI4JY12Bjoe+fUdE6Oz5S2QShWsV80nL5+9X705cgm/j+DbN5LZPO51Ooy8kql7u02lVJlE6/XnnYNo98Zumb/8ylVNXpnOlW1C4DaxZpREx6WEFu8mhAXCz7k/dlH5/RNvzsi34BTBD/jh26w8hi6yAfaoSTr8GW/i3nZwwZsXg58/uO7F69FjRsGWV3kypEs6MZC08n//P0fMrE0Y9GSYfzHJI2AHCFrCTCXpTd+OH7zs5ERxqRMRhdrNE0k+320SirRR9SFEbTlQYjwIEoNcgZTwLayPBHHHJhMV/FIUsNFcARKG8sK6E2E6boAM4oMCxicrCEagmOeiv4TASBnIr2HSUZ/Nk8lNeHjZPYPUJncBCqIc/LuwykYxsufXpy9evfWef5fZ8en+c56IhhMQ5FvdTx/rYUjWX7wxFOUT7CvyoqEBvRxRHIWXBlm/eXsL8vQbsmWkWL1VlZ7JNiVp65F5kuFHNiWJJdX26f+Nab5cNet7Yw2OWHpKgkReF6+BaqXZEkXwGGK2TzBN0nubyyJjtBApwHIOXeGNIOoog5EgOacoW4gLA0BxXsskcn7yQqrjcBWYNQGO8q9OkKoWtnPbnj3LsYVP4Y/sb4kPAiIsuKVnzwh7axVOS/YFsEtqdB/q7x+OV2VDzDAybgLWMxyFWh24081LeDmtZ36AKHgVib8VekCdBnFGHkrAM5CGqDHUbCGl8AeLyglO9VtCPKfqgXLqZKW1N2ruQ9ULh1J7pIEPtoDWqYwgTQBsWLymwP8kGTz3DlNwSLRT6her8VUp3DOkuDWtDqI1cyOgmr+Mg5KnXJvClxHP2B51fIdixM/TIMQ/My5dDQXxGq9JCJB2hK+I9Nh8BBvCAvx7OUJJ7SpfjX3kVNhoZUdfKomUUvz3UKr3YiIBktaKcol4PzUiaU7FNGx35NgrJ4lKJ3LPdDuUhxtGR5P0TfWFDYAKm5IYelCbOuaaMI50SmWIUCesgihyvay4rFdkWgYG1In7NPKR7CTF+0fcVUWEfNPWF4ZEQdjopULIVGYRfmCiwX1YXSJBYedTeSybF2X27WCAi6GHRVp+VkBhmiXYs1OoeJ/2RgjJ9CX8OohWHnKNCqvTNwykkWHjK8QC1Is3kRTFVPQ7e2rPuDPf5xTrAtqbuDH8c1olEaRs6ThjQN7WmElkjcuZE80ssy91d99+YazYKp0rN8V90A6Q1s3rf2GVzjqjCqXUMTpZoSxoPASOTaq3Hep8a5qRLW+/0Mgq/nVS3HaEpTuk07wsZCPtXxEDbyggRV5ITOdrNUTInHRHnMqc08OjLl8zmNFcK5ecBdIls1Ty8r7emZnlbp+TUG+UblWgqSBBbDaMjsqaxRLw6Mfh57nletigguYdPkWf3Py31xnHfmeXtN7sae3UIXBEA9/Npy+rPZ+TchpAsfutIY5v1N37u7p3tzqjjz9AvIX483dC3FDQBq8hlZhTJ2q4eiVUqZOTFBPzYQzC5aM8Q84r36zkVajcomogC5AkF8EOxpGtHCixAHczbRffy1CCvxRtjEaHYczP2QaJilp+kDbviFyqOBTTaiSoC2HLr+UV3B7WCbWKOzqFu+ANOtYWB6zi6HlXjXcrXQosLoyQYGDdQoh5VFoUTdHbzeJE0C8Pp2FEU/BlWP+s8XjwN9cW5vW3+jIcsmauAUs7Mc25anItge/sysVrdmV4T+kexWk7uJJGwWbzSfgrnLTqaCnLj/hzqset9QdWFY7rpyQzej8ni7Z7rQB+jXtnq1b5r+zS96QT+/Lg2+qAv8eDnzjeuRnAb+t+y5w8w/mvUVcESmfEbnEK7ofyFNyDaeeyw+XeNRoLelfAU9fnoObkt7GD/F5calu/bdFPb9ptrt93RRlFTjeElhTyqvOMYpZiCrw/uyjscTEBcj90S+PGoYbrcJUq3oLN4g42+p/W9sfIg9LUod9eqCJWXQ1WieHq3BCA7w17ZFJQl0m7k4DycOiU5TjH2g4FRzbk5Q7V3461w6Pjg5BwQ8xg7VcYVILGwm2iRoBCg/zNtESDxGHRceZkSyLpn6Ccp8vna0g2i1WyK24UZhi8kw7NNaYxAH17xvtw8YX9C9e7xQDmp8ZcIc7lXX0hOyKkj7cV/A7zJVAJ1bd4h6UV1e6U1S/nboRcXEE2T/i0V/aj1BV3v70I7mKVgEYQQJuAcvN7uqNSPn8GFHvJU1pKb+BaT5Rr07YX5mbYtbuhoRR2Hp2+uLVK5ErlRpAyRTGBuTwp5Bdx9AVuJz3K9FzQQQU2pNDMmFTdHS+8AdoTlRdfBPn6EQlEjEHRTEDQG8IW7Y8yufKulVeqY332ro7rft241qa8riNXS7kH9MwA4BO7oUz5y+P/Q5M7C64g6d9BxkKuqQ1il6/oCM7Aohl28D902jJNNNpm9YuFdsxvAsj1HDLaQ/sOw63nfawl8/fc2yrfUcKbRiNic67jQLbGI3ePPtYGSySmJmPP1XZHZD1KgH9maFKZtl/tO8uCDTkUdISMVElO1ElQGv+/OpMpJJyalOakCm7Aje0pO4coi/X5VckKf41y9VI6K244rlEqSccCCcsT8dI/dhVrgHDCVmyVanJX6siTXvgDm1rahjt4WTQaU92Fmk2A7fqM5smVPgBRFtM6ODT6uSY8YcYLRgOLDWfPMRzgPEPtNShAJBPz5xnL/S6bso0X8jk2eRG1leUAY5U6Vnc7t2qNDfNneTkCRv4X7jurNEA6dzk6UFxq7VRSDVyxkKj/nsIrIWLXB9EH1zTnBbX8wSPLM0wuyErE4OJcDE7yYlrQZgrLVcFW7KYsIFgpbTdTmrqdrI6YBr1jKn7dK32e5EMwcEZaurAAUpgtsXaQP45Cy0AaNuovF1nb2OJ7fTdlBW2y79kqf20RR4CP5cD2/H1Sr7+3c1iY7uba3f8hd3Xn+2e86j5hTza3bHKvD2TZlzdT2xzMWn/nNn1pN08kQIiO7f4+SXn38U16r9Q3Pm5kTyQYPGyzpWKOH0EgR/sq+RMKw3KnXp2r9efdA3DG1i0123XutPq0JJDrTaiSx12+ggh8JF9kVa1TkdhPiERUW8CKKC+axQ4oE4i25yqCqL+m1EEIoCcSqPV0chYhVcJjfOTxfbXW4Ue/wtn9QMiVT8AAA=="""
WAVE11_PATCH_SHA256 = "0e644ae9c37ecdf73fd354a8c59100ed31bb3edce5c76e13b5f08ef4dd3292ca"
WAVE11_PATCH_GZIP_B64 = """..."""
WAVE12_PATCH_SHA256 = "9920afe7f160cebcb51ae0d0dfeca41ebfaa8532285e1efcb92245cd8fd2fee0"
WAVE12_PATCH_GZIP_B64 = """..."""
WAVE13_PATCH_SHA256 = "a94945d37f379119ddade5ee46380df8bdc31601995cae7bc02f31c253a2e415"
WAVE13_PATCH_GZIP_B64 = """H4sIAAAAAAACCr0923LbRpbv+oqOp8YhLRAiKUqiqJFnZEfjycZOHEtJasrlAUGgSWIFAhQAkuIkqdqP2C/cL9lz6QYaF1Jykl1VYpFA46D79LlfWn4wnYpOZxZkwj2ahd7Kd4/kg7tYhjI9msjIm9tJKiY7bx0EkS8fxHmv13e757bdPfXOjv2p6HW7p4PBQafT2QP34PDwcB/sv/1NdHrHp0Or1xOH/GEg4OI0Egs3iFpt0XkpPsh0FWZ/abUt8Sp++Iu/jUSa+aORTJI4GY2u8dfLl+LnA1H9mUvXd9IsCXxpidRzQ/gVZfGdcFOxOu5b8JIHJ5X3+fc0i5dWGU67+PrrxUFHfz46Ej+5yULItUy2Yu0mgRtlovUfX9+KQ+G53lymMN9luEpFNpcikW4o7mQSydA+OPwDgIjbuSwBkg/LMPBglxOZAeqkL2SUJVtYcJzR036QLt0MYCYj4WZZ5CTxJnWWSTyRIkhLsFzhxcutiKfi9h9f36hXAnZiuL5YwjSjmUB6msGLUvgt3nx/dSZCdwXbWgK0iVehL+AFxXPZJoapTKcygfkp0KldIHkaJ7QNIojExx7tCvzX/VTdX5p4C0e2/3ph7FGxRXc2LdOXXuxLWm2rmFz9phPKmettW+XXPEfKtcS9Je48S6zh/3iVAVIdpK3UYhLzg0VOQstYXQXcysS5W1fo6XNpsnga18kMc35snSG/nJ/C79/PLrBRr5M4TTtAHN4dEIvLFMO0QaTgxWnGRJi6C4lzq1F1Di6U8Li4FF8DbQA9j0ZRvGm1L8pb7OD+dm07yCSIn5+LXXt05z5n9/5PdvD37aLeSb9BLqeJdwQLA8YI4uhoEfsl0dx09yCSGzENQingu9RCmUV2l39s+6Q/nHhdj2T1kS/XR9EqDMuiuRE2ElvX6gKlWce9UyC0g8Ojoy/ElR6HiwuALOJNJCaudycjX4CImds0jgf/5K5hWn2xkG66SkAqbeZboJ0gFb5MZbKWKYiPSbyKfDfZ2uJ9ImE1ocjnIuZx6KcMy4VxmejZZ39G0YQEiIsOvwQQSZDNFzILPOHCJHjgcZfGLRXIjQv/ZMFCjmDSDDBxYQ0A8bgr3ry7en2UwuwQlwj673//Vry5fvcOpFcKu4zyTg0/6Z32bXEFXOXOJHELQ5u6CbxvqjgD5KKapIxmII9BxsJKgaCA+hdu5IFAjpFpWUAD5BjuuVmcmNi7iRnaKookiEV3izOEKcBkNsBaKcnVVQZoxWUTWgElK1gCsAbMOgX0bZSuQHjZPJEw5e0S7qSgLzJmdpjtMo7SYBLApe3ImMAL8XH849t8x18DDsefcCmphEUAvlOAHsPOa5Yh9glwbKoYwxZfKwQJXgGsd46agHcpA4TD0I2bLOEXCv+5i3SykIsYKKKYxvW3+TTeA5HxNEgiZYA3OQNV583jVEakOcapDKWXwSjEDEgg0HtSTwM0DvAr3gBluUpQWaJug8WhMLqD0Yxm18tWcHGLO283IwTkIs9ko6UmghFzF7YjFguQxMESAOC7FvEakKEYQk/FIw3JJJBuYJOfAQl6c+CjhAgoZ4RnQIieu0qlWjPSXiKXcQK7r4H9WyaxAEJO6X30bbLNZHrBpOGFgJ6UtxzYSpqUljMePMhiFZgKySgMJkiXEhaBs1xFwTQAfCE6bfHOjbbifiWTAPmYzQHFq2y/ELwYqB8HbQ2Lge/SXuOCYFk5RVlisiJUbnEKDK6wGOCRJa58BpzM61ysACLODZQVMJcnCfUM2lXqKScStY3fRbAeLRnCAMUQUI3JQFnsu9sLjYs0w4HAh4XEU5Lu4BD3BMRonMjR6E1I+vVCX/fwraORn8ArQPm+Bll7UboznQZw+QeQyoEnl1lSvqvMo9HoG/pwIzOCfAQa+7tI5gsopCUSn8VLKaQjqNhFSgs/4mev0d5kJsR9JH4mtCI6Dc7+imYlXN8HCYGkgyhduvDRZ0BuGEczJR7JPiQMonxBCgVeh012l7gdk1UAdA4Tc2E6qZI7cxelgXjz/geY3Z8++hLR1PpKTlYzS7wG2MClr8H8sMR7YIfADa/v258ODperCe7nystERTaBIXHIFs2R+J4oDm2DfGu9+Sq6U/Y3AomcjGTXiHV0/ug3P/JzbggWjr9lYkU7FshLmqDEBxAV42zMO+DD0uMCyhgkoTNxYTcPwR46FL2x0ggLSzCTKwG2QWYUyNxgTnkxmO0geIxpajjVafIKSeyWFkVXamuS26O1G66kekBc34Nsw60ZqyfGMBFpUA5OD9kKjPsS+Lt18xuuQ7mQqJRAkdFLjKe0kVV95qsAbUSgsYnMNhLfDu4BQddEDHtB6CeqluoVVchsiVWB38TTDOwwbZ8x4aIOjFezOch0AJ8gIUYgTnJFipMvOWpEbQhhJKYE/ldkwQA82cepj1eCogh5DeUgLIdQI3qPYpvYVcN7BUoTBDg40MyFhVkADF64fA9L5FSYz4hGKPkB4nONTDrZFgCRiollXZBxMxmRkEfqmwYzFqks80KW4DBfE2FdAzngfZjWcus5qN4puSKwHRon+NPCG7aiNnEk1FdNTzbsVavXbqvf/Bwhu8qXuLTQhSneax5nztJ8CZDJptBM5oYpG1LyATirgJei0O5MVqhbgMqWLvjPW+WNv7690iTIji0jBc3R8toJiKOf3r16Wq0hEtTqWQI1LPZmtcBVjVn4gH8TgfxY0wbodcLW5xhIm6jmFocqKaPkS4ZBhlmo7B+wAMl22Ro6BC5814r+1W+zNVxAY8SQUewmaLKrx2F0e0S7UBaFTRLwmx8LeDhrpTVQ32YuMcE4emE8FrWiw177qD8uI50Xg6RD3qGB9NOBiXT0RSPwReEy6NokXrRKaAevNB8agW1XGabn0caJwO1WhItoA+n28w37VSvkn4jaDDRirAullUtftQmpVPAHbXiaXAUCarWQvpWbeoBfD27MJG8hKAawFzwyn9BWJZJkeDAsmUnmeorGoHkiwhg5jeTOEQgeELZgU5OIS2EzI0U9ynAnkRJvGF7l8VykizcrFDAgzTZJjGEg9GrOHgRFFzTDoNEJoiRcLcCtmTJAhh5PQxBUT9b2lig0voyAISpOgClxb8ntUoKQrOyBkn8jWgIyNC6jheuwCsZpK8r6gNRYBse2ei8HQ5Z3FXFsymhzU8l3kZBRFBXw1BQ0vQcJbQl6P4hDxLREOeuRDUtvmgGPLdXk3ty7ZyXVswcRN3NwCkSEcRoCRUozS9gJCeNZWmYlHGgw0PMvU6RYDy0sk5MWGDskoWVexZ/KVEYjxKS4fCmeIW8+sx4ZjUuj0TP4YI7+tUEo/gOEzAKdDgxxLEHOoEtQ0HZAjAL4JPKFFUh3kRIWEP3INGrtsG5UO/ohXr9FI0bieUWvNwmW34COQrjQRJQebD8VP7XHtd5sN+Isl0tgtKAjG6xJDrHLBLIGLxe2FRAlS3dlYSFNTuNVwv4jIA0MA9B+12+v3zmv/nl7fTMihFyKwUUh/4ANEM1lb0T7wIH21kCHlZyRd2j9AO3j0tgNSUtCETzkCA2zydYwaSxW5eDEhcEdWUMMjJQTOcdgTcmHDLVQkNniFS6DDKcXwAEBTOwFS7eReNuHzdwKCkYBHoBrDJG1YGdU+6tzVnj6qwTDwctUBIfgGVOE9wNGGVIAgl36j4m9r+TUBWTtkH8NHk8MW2Iw/jsVbei4nrcCtLsZu7Xj78XfxDf/ujV16P2ds3C9lLbRegKI9wDiRxPAct0IgO1eohoMD5u2El107qtPgGNijLfYxmFi1AqrxtI1qHdVqD+Sq2PC5cC1R+RE4adUfBzfAAOPRgrG+FMN7roK97tVBn6CArxJAtwLdJQrYRIdvtFvwCk49FDDS+JVpl/T7FtU95n8BS3RvkwpOm+DtYVhMZRMGFO30AgunqnFz0YjUwCWZjVFrd1qloUW7cOoKqZIRO6YsrbD8kD7ZVWW6TumOYZPKPMOHqBhFaOvOtqPMxyqH3qxS+DCHf3CCgSmLoBBsZ2SgvBID7wwoOfLeWGIxQpAU+5eluV8bXLKIn3SvHe9c88O4E/O9Iiqit7J+bnhXs65xnoahwAbMg4b76733iUuaH5BkyVwS/7CokFelSkZF7XbPyAzX6FF+2QKEzvfaQg38RNIAKVY5IMXrnwOJXMgkbKtFIJnsxx0k+F7spLiEK6yCJV9X15AIToeWYbaJb0MtSPlr+u6ffA+oJyfrDosbIaiq4lESPYweQKh63HWg3IMGGvgAKrNago9Ygq9t3TwUjzPo5d7LawdJm0wzbPEYCE6ZBU7MnInYDS02gUOnj8XJrvYQeoo6pAOiLOzhqGlsEVbXF6Ks+ZBLLYuxemgdN+Y11keAXDS1ZLD8sy+lfhAW83D2L1Gk09tFOUq9g5G87K+qx9WEW9XY4hY5T6I9HS+ATDuB+jSm8bZ+H7M8YPGkIMlxncOmULjo/FafcrDU6G7pegUXWZw6EzjU+hYp1LDxjQDBZp16khkuLulCBalNDlmMQZJMWZ4ifQkBe/HH7UAzTO7hrj8NCbTCxYeb1peGCyX29Eoi2MHXQkHHOcVRRW1pQUkrNCm0s6YIgWixei9kkpNtM13QEoa4Xw9nHHTdGu9+xZJxPplhbymWzt09sGhWRlQoSBLqKTFS01mqLmIHi4rrMzsq/UuDlOhsUvlDy0Nvt3rzORYG41qmfwZ+7qP+lP7YHA1gILyq54wTdYsAlHLKq54xRbTZhqf1R4aV9a1K1hZYAAzxFH1siZN47ra16aRhZBqhKMKESq3OOxdm45ik9rgspRS97FSgT58d9fCzW0rEfOnj9501sJMDjLNIvYpq5PqvcfYFYhBTHm9uDA16PcbGfXtk07XPnmlIoUUGee4JkiAIMLwTX8w6NA8sQZlsQS9gbKMBErhwN8DLAcn36pZn0YwvmoWVW5R5C9PA8F7q2ZRnnvpWtWnVBakN6jdKVIk/cq9IhFyOmi6pTMZtbsqBdG1e/2TPabRnz7iRnzKkVS2mR1QioBmRweBHfBV6EJ6vwKxDYis2OykLC5NVBvWJuZTksyR91+0Gu1zC/EJYrg/OMGQqfkokMKVQGstURF7z01U9KbIh6GLDb4nWMIROVNBZpdnl7kBzu7J27tnd/vd6v7aNi7KwHTzynEODSvnhXe7YH7Rx0YUvMalI/3nKWyO9jIW4uROqXBkAS7Bsusu1ecgoL7K3fT9xPXXabiGDYz+N2Kp/GwjDRVDNO7KFjkgahrMVonMK42KigpwgK09VRNFuUSROjGKm6jwSLgiRes22V1xRMEzyuUFapOEW0qzJb5ZhpRRUQSVegRpQ8BcF2wGFHi3m7ma+DE36chdcUgDEzMDDfmOTmg6G8BLr+8wlho4HP3Ey6rnOBqByfzcYHurWZfvEgcYkCDPC4UAcsK5MzzvwoceMsPpoCoLADH9AQt4lYlgIcC0bz/2EhIylug5/eGJc9I9d3r9biO9vEHvIdB5C51PyR0ZleHXUb50KaWf3x1x7L8Adt+Q4y0yAOm8IESOKwSZyh5gFVq6Y2vJl1BhLgcowKFHsjn4MLS5wBYOauIUeYTdws+T2jhARVUaN51LSBqtriqYGVlznwVmD9FQElMTDYLOvzzj2F1R5CcDig9v3O2zfcCUC6ygaYf4hdg1g7I8MkDcl0DcVwTXs7wCKqGaWJXHScAGTfxQpumzuhRThH/LFQ6KQPx4EUQUciUycaeoH2eKZMHQOLGPxbtXeRVV1x6ewfcSQJ3rIFbSQktZVTuY6YtWCT0vS0tFlj2rsVKVaIsYhaPCIGxnxBQtdTBO+v8leio6BWWEEUGp6Ry8XwROjK935a/r5i2siqM8lGWVQD+KQNPId9JVskav1nEdUjcORlvQdENrDTZnJ8M/agcoo3S4x2Ad1swAYysetQaaPBbRe9JWfbYwsn7780Q/VS5GtiHJjVliLcUXEpQ1J983Mfm3KaWwRICpR0wkcTzwWaOh8uvOMm/lejYWeZfvqWru4Vl36Pp92x50T/tD329swGl4ulbmXbmPRd4nYBif9cQh/j4+wY4CSkHkDrbZLKD6YX49EFyzT8VJFOn0dYkJO+KdIuRE7t0Y6+PHgt06XZbHoor6RnJorXd9kHPKXmqPxCSMvTvRmoNd0OYSTNqZOUo3LDrJxAbUQfG8UTGDoXz2qj9mn6gAhVOdMIAeNWpVdD4xyNICFNV54kQeRIsrAkkkw2bKhwCElzLf2FWhRCFoAMr7kUhs2wWoD/J+FSTKw9GhtLygKcbiVZVOAoM2Adgt4EkSnJxwiKMCVt6HwXqjbYtx2X/PA2shVoWkmd4ZQMksm2NdQsesA9NhnNaYJpbX3uCGtS/Ibgn+DVP3t5G7wNIAs0Qbax5LM6OuEOW0KOShuiK9OMVacwniDUylQff8lKunOEWf110prD0pYNcxIuX1ppG8q4Tz+8X3UkQvv1qNqRWtRMpHar673nu3HsXr1AQy1S126jGC8vXmwF/pMS12G0GWqiQ7lcDCtHQRt74+0CQx826178gIKOZPB1POQ+Qx/HLLz/Pnedz2z+IMY+3d2gBzgRysbxphxOqrt4u2N7ayiYyVZVBeXRvTB1gMADovh1LpUuLeAV5VYxCzVR5fxBdrl+8brjHNNdxY77pB0cfa1TwA2WlsYqIYZO1WHotsfKhojOrsa4yq31Uhyfocgdwah5t7UhnQNroxja6/5u3Q3WGdRzajuuLGTWjcgDrymxG/A+nNCN+D7N2IbkJyA4J3IrddeLg/zSU5WqoQDjQWsa70O2++vzJyShs3VakCbEXKdA8FVsQ0l6WaVUJqcKKK6QJSh948jlMsqKNUPBeyUl+B2ZFSQIxiga0IFE6MqcsFLQvQ5Vh9gSXOefMrdzxQEZR5NeP+IA0vKNK21LCaBNOsnJat5yKL7OwkjsNaerYQfPWwxGs3Kio3VfkQxp/qel3pSzA8Skr4rw1YvqGWDpWN5IAG6/nOTMZcETh1Pe7bQBVPDRtoShlFzwU0/cwFR7Cqu+emd2mlpL5DpU7U84vWERYM11DYmDZVxXgN+qYRu79Nqtet2OtaB7WqIi0Wu6R8mXwAbsWKNTBa4GX+Bq2+q6NXRhjxD7GID3+nRXz4h1nEaAgX4H6XRZwbwmaM9DdaxLkhbJZV/EaL2DSEDcz/Ros4N4RNrP0ei3h/bX+VYllovlYNjQntLjV7UuuLSRjcZ5YT5mikqGys2ydg57XPEgdmow6+HsSoWWLGtW6NthAWmIknW/Vij1Vf7/RWxv0uVxv0Rc3F5mvKtQYpeHbal7Z9fjaV55PhTtdaPVVzqdV1as4/t07FIfyLjfkHQreMcqvO+JP4n//6b52KwEAmhek6s8RdzrFB+Y4TnapKij9vI++IREQBLaVzNDQ47i/uwEZh8RHfEq2rrz50ut2e8Fco1bikB4htFgIdel+mIAC52AFzt/n2XxwIfY27YowLxNLGdx/42I2y3Z3svOYa6vPLCvvecXcoz3u2PZ0OphPX24n94sHaBhS3aA+GtAdDPhyBctF4AAL1fo/UqQSwkAPR2LkpSh2YJmp2NnSKvQ2d4pGGTppyd0Bz7p5Ygz7OGrDcomfayAMbGczmmUMmUmsDjuub5eonumaJaKTOLgndCTFIoSm5SkxrOlR2VDXDxdB8wgMnpWZYLMsGWCWAzHrKVFJmeRAacqpifBOkOsFGIgMsMp41loYkwUPe6KOumpW4E3iNamPhnAnpi1SiBTfKG/Z0go80tErhndnnpSZ/3fesyo1woWYfmJjAgE3gZ/MO2SNqwkG0DtKASo5ic5DZsGKsGWv7AiQWRiLVyY9vbp3rt7djsZy7qUyLAiiwS+MlKWndlxsAWhOwjSzVQwvTwEZY1nh3QDNk3WVymRtjC9Sa3IoTbi2utaHm3VJlYUNVIee3uHGFuBWUVsJ1y7jW2FLnBqCpPclL5KM4y3eDSu+p2ZvEhUGTxr47RoSd9UFk1k1zKCP/eu9ULswD35elJ6bAMv6ILD1dvGRUPGLEe3rcxwzWLzgLLkL/hWaEZV+DC62URM8W36t1V+xWTk3kzTZxB+Tif4IDA/fs4jWLvNb7kl7Zwjawe7NWOZjybE2LFF7cxy1LAx97bF2fz2f48O7m2zhZ0Ge9G5a2+bA7ldntQW1K+XwfPvJGzdA4FChCiMxTZnUAT/pQzRpTsDhzDI0cikqlNQA/tsVN8PaH1ow294VYLSuTxA1GiptgG+rcDdfkYXny0Tfy3uYh8Vo5I6PqVgc0qeiBTpfBgyqoYxd2FxBo8UJbtG613LaVz/ApK8c6hHwPC0xUrtYxkwbhCktJmW204AS3B32nDBP/LsgqL6bTT9I0n1NaLljZix9zEhWU5bQ8qJCUSu7hQSWKmPEQFqAnZX6y+AkiuKOmoaZQxwxXdXEBRF5ITYmQDktILV8xqTXisy5mIR3jJTAIlhZvnay8O5kdqAZPJT7FeJGSBT7+Fmh9TFznKuGrjwNB9ewLVPqTVGL+lItiqImGoaG9DLSB8t0mrXnWH+BBSGenPes0z1qAauQGiZ/LRxIxWrGXwOaP3ABQGqNDkDhIfa6N0KkyNUh/NaJRujKniFWKVtH1vLPhWZ0VVH2hjB/oVUm8lE6abUFLXV6KD/DtBr+MRt/CiMpDySJ15JKnqD5XRhjhIy7znNp3a7OasFXMB1E9HPbRQDkHq6o33ItrRa1+Ah4J7qsXLzE3vsxLvZR0E3jIl0lXbbsMhMpQdSdw7rYC+bboCsh5nuLpgIy2yvuNQh7XOLmKz7nAowTItzLS5LrCGQ2VGrT8YBcYhUdZZJzi10sxND2q6CKwVJEBeV8I+jQqaWv4Xo/kb8u1XFF+ttRhU/SWUxKEqZ3j8vyDps49I4sMscECO8cXKYzq8Vo7xtZKWCvRzNJ1TD6XaQUFSqiOFCNi5lIiG9x7ypbXDvDi4mXctUv2H9UTH8NPSE1skx+fW308/qo7PLN6J49SfaWug04owJgKEmARgbnEPTmMymeAJFR4tBNcfnZBEXBoPLqgFDTYCa0c7LDrcfcnncFmhN8qsYNW85sb6qtrL7V2P7qcYuHNzttahJEpetcKLdFtP3X4+tHh8HJEh9WMK4OXcjLfObDGELtfqtj4EWB7sjFPy8o8lp0pTkh4dHEtHShr7+b3PGyTS8EdOCgd7Jjzfjvnz975CTnN/eOh1e8/iT1zA7TxhaeDhicu6ivIhUeqdWdJ3GildNjAe8bheUp3mP4z+eFYdJB7z/kBGY3QSONMZAhCxnMjVfOqPEbtLpPZRP1a2rBy00ZgVA6LsyIjX3mSG2wG993kTkd2tU03ovP1cPjwz43gzKP30I+NYu1i0oFs6pwyVcqm+t/ABLSb0c20QgVjj4qhxkqggtjMobq/xTKosd1u2DpCBxXafgRP/+r1J7Tk80kVZahMCDsBEAYaIZhFap8B5/rtLQHa6403M3KtxUD/aOG0c8C98+gQZW3vHXNno+fs3A8dpPuip69huImSRl2Ojjo6RZfiF/RQVim5rQsOlb1zs1928byJzzQjXKpom+q5XNgbfHspBIcXwSONjKhb+2IffCIcBT7SaAGoCxtcNwObdCmIjCsqPjgY8pmvJ4M+hzVrou5AlJpyhNmU06k05fw84Wnh8SNOPJ2mEgUNnvGwWmC94tChxGDSPzl1lu421ULw80E002UZtNCgVZCUrZrR6Cu5vgkDT9ZHUFnkaJRHQXOVcNLl2O/J6ZlSCSYahE5V0Bmc2PqIgS9Seyk1sfIBdalcunQaFmYqVOicg6MoGSlmVoCig3fUgchk+6MNNr7H09QoljkNqR8/z6ahDL2Jr9QUOOFdQFNnhaLgRH3SdBzN8RW6zHISx3diRsXCnAhLtYmomglWUaE8uG2hgKZ1z4zO/GLNQTlu4dLJBSlsagSztcXXU0DIFO7MixBn0RdCsJZL6XLVPtqymFubukGYqqSdflU+ZzpJxOUjRuiGfMhK/RIYC4jFaunD5HYUyZtUxSW2biKp4piKVUmcqOAnXcV1OvJhCbK+Ui5f6UmzHu9CYye29f7D9d+/fvvWeXV1+/ofJXiwrpNeP0e9yvupmDYeupq2K208aL9pZ2SHIMfONDE8P1X/DIYgVUWWrOSO2tocpAUU4wzPus7Jef+xsaqN4vi47wzPB06/O2woV19FHO/Eg8v9WBq+NQXvAiwJ0MdEprq4gGI5uJwSOLRIiSnwCYwC4gkyGBpXHRqEvjiI8qN04c1FfNasYP8crE3dEI/0elmg3SzPFia9CUVvpkZwmLUc8rNUp4ZuzmCmLnuchsw0wTTnzUhSHeFR6dm2lDmr3NC5s6E7OHX7tn026J7Ls35j7qz6aCl7Vr3JB4wP1AHj8HuoDhinTXdMn1Cdb0DeBzkYJLTQwiwQoKiMTp5tPZ/FwEPP8eRiS1y/v3HeXd2+++GtJZ7VgGNbB1dn0IEnWP5wQ9HC3oSzGRMusOhgxJyZNDWC+63xLHSq/uu4rQ4c+mOgWboFnQ8H1VmqcUMuvc0BTGICycduzulMNSVecsSR8rj+8frDPzFwYLHhzBUdRbNsFjO4Sn2GrmjQ1RmNNR8Ue+DCj7bORTM0VXVEjgLIpNQoRMF0Ep36oNbzJRZ7JQG+K18CeiK2Pp7+FHUvnk8PSlj/PYdaGt8kH4LRRD/EQFHqTsFn4yxuihYBH4/iJC72QiUg03F9a+Br2wWAWdJq256Llv9fVsOXWJGa32cPjWKH2vJAPrDnWey3fDr6ndxuZNG2vYo2ibukw+vZhmqKinTKJ80X332znM836+9801n3y6WDOwMJtcBBAQGm3BQX3VEcvAO0igaX/HaS+UzYRtoDmYL+ugRLZ+PkX6qsQS5KS39zIgfmokeJedrUON3BFn3VRWd2XqjzUPOyOIrcFsHZvNhcVb4gl9DxA8RlWDtEJ5u70YUSQspGw78YoRo6ZJobHur0MWX2iFnC6S639qcquLgNWY4O2PNCN8iPozI6cVi6lgpr9gR3jaCu8TcEyicD7IvnPh7HrcZv6yOeErEtVZE30czOUFJpXflVFCbSL19X9dYG65XOcKj2GtXOLSgDNc+q4CMtitU07JEOYKq/GfH8zkImRt5FlmVO3c1tgtqN2pX+qcpCTLuLqgkbprGr+bIQV1gVBHwZgZfbKoupkkcMCheIcS29Lz52p8f9C3HPAvDTxcH/AoQnuJMYaAAA"""
WAVE13B_PATCH_SHA256 = "51aed3b9a31adb8d902257c605b79181bc1973ef5ab123a5036148a0e6ca878c"
WAVE13B_PATCH_GZIP_B64 = """H4sIAAAAAAACCu19/XfbNrLo7/4r0NxTrxTRtL4ly032Jm22tzfbrySbfedk82hKhCxeUaREUpL9Ev/vb2YAkgAJynKabrt769NGEgEMgZnBfAEYeP58zs7Orv2UuefXwWzruef8xl2tA56cT3k4W9hxwqa1RSd+6PEb1h7ORj1vbtsd3r+Yuxes024P+/2Ts7OzA3BPWq3WIdj/+Z/srNMbDq0ha9HniMGjechWrh82muzsKXvFk22QftVoWux5dPOVdxuyJPUmEx7HUTyZvMCPp0/ZhxOm/i1tN01DJ472ibOOoylv6OX4d4qdstjGYsuZxXbwf7RNLRY6C+56icXww/H8FXMTtu11LbaO5FOAyWNnubOqQKlRksa+xy2WzNwAPsI0WuZAVu6Nk/BN/jtJo7V10irDkb1gjyvd0N/ZLH7eXRbfz8/Z3914xfiOx7ds58a+G6as8d/fvWEtNnNnC54AQtfBNmHpgrOYuwFb8jjkgc3eLLgkzGgEBAHCjMafhTB3J2X6eHwWeVyQKeDX7uy2RKfPQaOzX0IfnTIPpu4xZGz+WSEcjtdObmF+xFHo/z/eoFJJj4uxoMfFxWehxzyKmcP8kLVt2085iAHzLLqXSr8KpX45tX41ilWodncPAYl8/UGbyNcfdD4b+VB6IAXfdajz8F/7/WeUhR0z3ZyV730S7QBe4ofXAXdm7tqd+eltrQB8UPPPT6uhpNWw89l0kjrd5DCMs+5Ymv1KdPsMtPvc9KvQsETHWlp6h4yfvbvjnY7Z+snLpPnTHbmzoefZ9sWoc9Ee8sPmT9HabP8U5cRs3THZP93xZ2M1gW/9GfKSGb1aiUbMRlUaNg0KTHy9y0kUcMA38u/1xh2xJyz1V9xrSF79+BF6KwUijbs//HcY9xG81pseYDYqPAn5ns39gLMVaNyMyaQFLv5su9cedGfugHjv3OO783AbBIc4TYBGlLetNiDc6rZ7gPCT1vn5F2Am7uBFveeAu5jzEKbnhO0Xbsq8iCeAc3e2hGfs5/OX529BeKURi0LOvn3x/ffMnaVbNwhu2XR7+2eCJkB+47vXYZSk/gzqBrc2+ymOvO0s9aOQeX6ydtPZgvmATZisbnjNPTa9BTMUnvAbPtum7jTgl8xPBTR+4ycp9CQiSzVwb0HCMdEQgaxjf4YQOIhXeJDSM9f3UNzaSqfAqmUr7ibbGGrPAhdkDVROOcCesKvlFXNDj13t4BPAwFw8A+4A4NH/cOo42CrQAw6WswCHSAhcHAFg6WrG/aABjc6H/Sbwe/eKff3mGfSDx2zYPwNzgwP7Bu7UYty/XqSiNI1SsLqjUAB0Wb999vp79qZvs9cZ1mHEK7b30wW72lyxa38HL8M3dzqDLnVQdIFFc9YZ5+8UAPF9AMpdcbant+IQ8Bc4A+lixYE88kEUe9BR7Cygdg3YTX2U1hLj6T4SAIFui4Sttgkw+DXwCpsCAlGp4SdiD4ktiUS9uuYRvAb8D5waoOh0ciAEcFGgV1M+c7cJx6a3bAaMIxtwcFncVbQNgfwwQh5e+zB2oOA1ODUTAQfmPF+7sZtyxs4ApZ57OwFACDwjjyV/p4gE6CFoniXwgBhrkoFZ7hxidgQD1mCAaCb/CL90B0NC98uPbyXYrNkmb3dmooxF+JjDBJUse9I6aeFYSZihXJxMvgsBQJheZkViDk8m0y0iYTJ5jv0Nvef081Kv48XAE1DnA/503J3rBzh7LPY1/L4rVRYOXjKZvKQvr7l45wy4O2V/f/bq+7/9NGHbBJQn8PDgMiv57s2LV6+Lgm77Egd/zn7e87BrD87a9uD5hHX6bLNFZ1MITuSHLnv5Vv4E8g37FvANCHii74qNL4Z29oqfnW+++754BRTlb3/5Vi+DWVb07AdTO+wbzva1H4ZAZ5jDq3UqPWGUh2wBknwKsq6QB1GYd+WHNz++VAbb7xOOQDeREvvqL08bhEx2+jVpsxVMF0HqCfuLqrnmOF7APEilyeTbQCisk9Z+AXwtlM1foEX4/TatKLxyK1BOH0QT1YQUBGMfCs0l+kGGT0tYSK1ay6iV6WngvTiFkUounEzCaN9oXlbfR2zwi17347JBb7N54K4TsAiatpuA8zVLHMAWoOEx6/AhOxcch/oWHjdPWndIAaTqj5nmiYCY7Ar191Ums0iZE3gUnKC/sLh9hZyHU5DmKFDaX7nXHIgNliloJPY6AO2RDQobSNJb+ZOkeHInOeETrBSC5s/ZF/o8BSgKQmOebuOQQavGo1wte7kyTaDCZuuDenLZ13/75hkD3Q+9f2SjVm40mzoZkLb4MiDt1zT7hRuREwM6Q5wMreeRnayclfs/UWwx/ZkfRnGTffGENUYWG9T1Fthk5aZfNHQjqhjDm74yjGwUQK2VMxoAu0cpfv1w9+HuUckOO66DRZtm8bUeK1IMAmJyOTiZBJErLVTCUFGbtDQ5YtCA5BRrgWR7LAWTMpPCKU4jEknAw72uWgQmgLN2PaiA8sX2/J1DNsMYmX4sa0oOz1hVaG3JsexVtD8jDLDX0TPQkuEMdF4I/yekrK6JVy0w3XwyrnKIyPz7RRRQkC+JwsKkA0PJzewpj8/8BO0zlNshEMSVaJJqyy7GsoEZ8ZbPvtqOn8JwGiAZFBQ9lghQCbFy142P/kfWaPj2PnbXIJevndU2aHQvmsUD1/MaPuDtYtRssi9B4w6ExT1WQc2iIAB7LJdPML4XN2CGsnlnyLr/92wE/bnpgpUMEzLw0zTgZ6A5fTeEX0suQ50pTF9AL6Ixkwa5MET/4fD4wmnTngdu6tCwnI/sXfum3bYYvvd9s9JDhHojAM573Qxizg8PwldvJDDTIcwAOLA5Ot2B3W4C3jrdEXypwVXRl+ktcgz0YpPYAQcxVrRoyeFXn9+IR9Dbvvq4MoqaQpgYpZY9eIATAT405GqV+qgPnHZ/7AxGQ8EMw76CVlS9YCPBYDQLCTQY32fOJg1X03f7DQ4f2tngukSzHA8SfvPPtrdOY7VBMtMaqEgyNiIBtQBTtAHvstjppuiAUpSA9X0qYGn9u5mrb3OAyg2J/pq3iCpzgHajAyoNtFFlutox3+hjpldoBNWbSIlqb7ZgQYCudDbjjADYsRtEww0OuK4Pva4idWFSv85selzDWHK+Tg7a8FYuM9GhYHtwcDha4zm8vQ/eNPCYzZ7heFyyEaYR+FUwWrEEMgMrETwjlJAgKD3heiXYDpxT/FTFBF8Det4VzKpjSnI2KQuJKOueukKZ/ILK73WTjpDxxNy8mHEFFXNMPcucN9AO6LaA88aTWexPhZfekEoJPZxoPk94aknFA9UAqy76yk0FVRLaE/ZRta40w4qBxwZc+PEeK1i1PTKGu+YrsAFWLjCcU1gYTogObMkaEfyoP0NZ0GIN7FllUpSrwowoqpJE69ZURXYvPQHe15/cln7Xh6KkQWEsI4IaS7ROSVl+pymCzG12EjSBE52fNbNY6SMQsG3o+UTwulJyZx0FrNysACh4/BMhAqUq7Y8Arc6h1MnjCqb4ZUsN6jd8EEBNdJNKSKVVNfByeLhdcXyu2/uKE5XY2H0AY4vZBIDe+e9zea5YsNKRAnUgqaoJ/dTZOBSGqXZavgleIrBO7wCTpdR+eU972VgikYB0qkCyoMhB3NX1SB21/tbC7s5Ffh0aDnSheG8hCgt4uiYi8zLkCfBNEfIT/ieGesTEpZhYzNcU5uSl8BNV1sxxCrqJl+7cYMuJieibDQ6AjKEp8jQ30kUYDSNzOTw3vGUrP1lRTHXF3TAp3F1UclmEFR8WgT+7cO8/lXmPY9zM7TqIcc06qg1UoLknWzioUHZ89sW7Nui1y6op+V6F6KXRgvTfqQai+v7sLZxcCuCbeMtLRXM/TsD0zxAO7hxwxeVnwGY2QL6uGZxA8HtlapgGh82N0gM7l8pIDkEsySEsn2WLk/Sqco18YQe6R+95l+bdwk0latfU+ugV50inNorJ3xIA2nUA/Dlzwf9ypn6aNCgQMVV+fjAvPWb0m7tBwi/NdXRCgiGROlHs+GHC47SBVATDBgxWdCHyGIK++tiq+XmncT5aYHMQB0psQY4LcfM6WvFGgzhEfyP2Xme1T4u8qM4+uvWEmglLhAKljzsy5j6kd2ACB+zD7G7CPriTP9+xHbDAFL48OjK+ksukfO0jk5ahsrpRhCiU0MTidg3GOE/8RDEdZ6mbSMMxsxU/ktgswie43PK4FFKBZ7JP6xj6GIQqdh69kytiZ2guvmcfPvzj0R6chu36H48mH0RI9c76xyNamMdHFIrEJ+i54AN8Gfw+af3jUT7Ft1S1MBkmdg+bCFWcl4qfsmyply2VMgSdaa+ignyQga5U2Og1EAgi0NlgMTahX0v91077ucl/Y+viFcmac09gaGL3qff1RcRh+Ju+3N1pMT18jdDlzfJT6dSUHxcav1Kk+DFKkWK4nat2QH2VUo1mEa0mI+vOuLKbxLNzN015iAr6fBV52tKuqVQu4w664+msPbPti3F72pvNjTsJjO21FV5jDVpVH3etTp+16JPW1dfbKa6tJ5zCQdkiEDvNo58gd8BJhCdv//osA/k1PKEQ94sf8mc/uSnKIXbCMBj/ahuSSbSO+dwPApb3hqBRIBFMoihO5YTH5UMwYTzmpzZBODlDMLiwSbIA/pkttuHyT4lcSBJGwtXSof2KV+dXO/mNlmjlYjCPoT49FuCmboIrfle42yXhGWwuY+W4CcanPtIiM5Vk4WAijyWWgcHauhLwYg5CBZddr96FDi3kJqY9Yu+vbLFEcXg4AvoGmVbujSk66OP6C3RxytM956EAh1EIRWJeVd98RarbzSIiYtGQ3oNBjxja8hB7JeBRzBfk/HYVSiUQgYbNIwbFerd9CPEClob9T0e87FgJ+wXi86HdTwH2H+8w2rFvzAJ/vb6dTNIoclZgIjtufA12V5gmzfcn2YSQjCt3VqmreidqkEGbKaJkM2Ff/00MrIjAqFSdKDtFJBpLTahoV18EOCg9xsnd7dBW6W5HbgA1joRGA3iwy7th6GGOw0rlKC72ZTVUkaqOTDbLt/qAlMSJ3Txh9XJSIlI+stfpjS7HquVSVo5G08Ecntgjb+CNhrNaWWmAUJGWhjqE0vFgaHU6gFT6MkS0vvr+9Svnmx9/eDERg7RRW6yYvR322dq5tUxPp+WnvS48BcaUW+z056S7BIYNBZappKDBCe48kjuf7JhfQ0VcxP5y/VX36aX6eAoNv4y/6rSfFuslmWk4ka6rewsSxygUhPwQAiXfDZKJqRyeFFcCmi+mK/SagwHFcK0fY6lyDRbtTACaLmz2k5vgTkCBoRxW7kIL0SHnvfR+yZ4IbqWBqA4w32YIX+Ed9BnN5xoqcLXky/lX/RKCgHRfxp7AEJP8MKTteN3xqCcmWZkbAk8QhggPrbsWewf0f39ZLqbOdagUBZS5XLQmsufxJ71GPrx3KhNk4FbRTtbrwcDBMPI9+6ZSBmb3l6CcTUUDKJIlYvxi1393PO5ZF6bxz3Yp+mT2dRBN3SDDAr499jqXh+r0qU5X1ol5NsIRPh/iP51L3c1CBvNdqZzBGfDZl7h5REyp1TawkUklFG8gwWSLROASZCiUbFF9B3IdsCVCPmc5L65czw4iOxGNgZc03irYbSTfZOyJaNeXo3U9z04EGqgXEmGDjB4lCISSkbH52CpwOZJEu2h3hRC76LTR/IOHz56bZRgJE7Epv65EiC1DYcijm7I8I+kntX5JbqlFnyzTekaZ1h0aZBqRUsC7BOmRJGL30eN8dzFFQcqSpV6imCRIp2sWIV1FhFx0RiRCLrItvSVaGEVAbsiY5USPKiEBqhVEFzp9qiLx/UnCJFnEisDAjpUnJOF74Qa4Spmj9Zx1TVKlThgNFWEk8NXvWWPE12AgRe6PP7145Xz3zf+5X+7IySRfgn4J8ADYTjdkkbLWE5ZKK7Hgg0s0R2EAElXv0vd2uY/ddj6A28tC3ORCoduxhHCHr5c6ckqGqalx18pBdNuXJVkEfhqaqnJ9rRAwpeaKGBIwzM2N0qVDWEM4BvnSV+RLZ3hQoHcGolY/I+R4JMXQxViKIUHKWmMqlw4PFkXF6QVDKylUWgeLjSIJTMVPNLO6JjML7KZtPOO6VMK+f1ahdL9M6rXbHZRJvXZHm2P1UqlzjFQSogvHc1BsKUg/Qi4VJHhvNGTq5Mqg3sgZ6kYOYAGNO8BGJqEr2HiYtIEua/JGo/AR4oYm5AFx0yF7gDQ/TslWTQVFJuQztzL3heQy2xZiPstZ3amD0LVkl/sZOoddFOC9NtiOnS7i8+Xbv99jfugOZVY+F5NQ8VxL01NzU0/Omict4zRWfVaaxlX/Z3Os83OJ3jgLcQcxyyPU2szcKB7IBl58qdURqJUFBnEyNoqTfumxnOo9nOsC6eCg0Iy+6AoeLuPc7Kgs6+wHMVV3dcXCugBM1M3gjTKFNwbj4oHyRXaqXTJqDguZ7Fzbe4mjTrtvdXqApE4XvoxNWDJP9JGc6IVq3jFR41AraZorrZBzsnZiXkuhQfEyYMJzrJFLjZK9mp9DesKuAZXf+Cv7Jttdo8KxZdAM/e9g795q/vM7XLqwKsDfs2kQ5VFvdLSgIxi+PHJWsIa+ygRv+vnl22YBL4nwhI3ggQQ3DymhgSICUBmzr3Y+Bya0oqWeOakI0T66VGFJO6hCsk9GDVWLO8YKJPlktW6N+OtRe6zal6K6RoD2enXO2zgvr0pykhNWSar0uzWeplKbvmS+b02XMiGU66/ljvCP1jw7146GsksolFYkLgxLuxJ5UTZ6rJpaTPe4O4RDemmv6tNLAGKO9kYXpDz6OFdJeTx78+aHV86Pf3vzhwY5ToN0hkYVMuoaVQhpFoF6UB2oQvpdeczZiPk/9AggCHhzgJgaZHrEiKrDyuQY1aGoCaEb4kLIG7QBaI1RoRCObiH1BQju20SK2stM+Cuh3ePiwvVSvmKhdxTzvU4olNt0ValeaoN2thiiQZoPSFBj+9FluVltGyG36lXEgGTa4BgVMfiXVhHVkZPzgyQcHSgeljFX9RwGiucw7o5I+I+HbSn8v/352cgk9x8m181V0mj97yHwL8wuw8U9LsO4R2sbvfFIBgHKyP5D1ANu2sJluGiPpagvY+kPl+EPl+EPl+G3cRnu22Rg2oqll8nNBf32sDvGlC4wbpCf/N7NBXXbsErlKETISuyzFn3SlgJ/tQ6K86Zq3pKpafuHwMxqUso4QsvEk9J5D/O2k/L5ZOUYjbY9t4H7hm/FQfYp7jtt3OIO1MsiMVBeyxO1UqqFcXnRH3V7bKWu+IiTUhtL6XXzUu8RNqBZLw6l6PmOqLd45uUxfnPybzNnF/meZag9fVBt70G10wO1W5XacXIM8PflbHBy83wp0w0P5vYcT806uEAudtQL3huPrAtgvfEFLsUcZD1cTZywaRQFyvul3qzjStQMvwIH3ghGmWFSONrJ7ofEMvCcnuHvS0O7hai+kPwWSjhralxJPmfRkLWUWDDUMgNLWj1RxvVQFr15EBvBCA/UJ6JekOE2aMtQbz1Nc6CLh7Fy+LAur39lzkcsr12fyFBs9muoS87NuomCCBt0cfdKa9Ad3z8LQHOVBa2ipspFpglQrFh98gSQDJyo/OslqYPGmoUvUNnVMA+yCbCUE0JIXIXxcb1UHZeJ6xNEdzGYh3L9wzjuPp4f9Gh9Y9Bv/2o8v3xQ9UXyIKZPPonpQ2J4MKOKHQQH+VyI+dbgIlt5q0cT+c4TTBxgZmeloupKl5m9bqPrg9h9I9hUcutOfETEs5jMULL9Tn6CHX+Q69dy2izWOfvL6TQDkDoplTmRbT7QUyIaUheWzSGipIuHwHVUmSwhKBN92UgrqBSo0PD5YFto8yAOXt4z5YZtSsk3BFP86CmXPGzSzx6mGd2HKZrNp2maZOHGHCwpmRyjyL2pFpQI17SjJZ4ZA9OfN+hgq9Z9yf2TyQtKW1ac1CI8izjMsD++D8+/gznrz3MN/CUb4Rm8Np6i1Rw/eDjKHpJ+hgfgSpZwohxcO4yeAZo5reGwdx96/pAnv295MuwTn2fhxv8N8uR+n62cUxuzpEp84ba5NmuNOu17tfnRQqAoTaP1v6YyL9my988z4wswA402g5JUTiDKTlyeR4guk5G8IadERdxvO8lGHZpko+746En2WWeNAXz676a066KOmKY3qMQbs6cy0tjrD3tTd27bo0FnOJ6OaiONebtKjDEvEavRQ7Fqn51XwmNgMqHht+vtX/FcXjYyKGrM8Khrk+03Eyz+3s2OsallywNlu6JMxtZFgk3KY/GzxV7Ssb23MgUjHgZyRdrUq3cbMgZarPt4uRNz1w/Fab0CFGA49m8scVKRcsDueczZdo05+biXAZtFYepfb6NtoqVtw4OkOahipY4y11IeDiWHb57ZSSbChac7kWEDByDSaxTAKA+KH6oJJmm5O9KyUOJZV3G+KV+HQOAFmCLrBgHLk4jYetZhSjhbk3u4AJZiZmY6bqskEtbO11+yNLrm0I24SBxMSKVWo26BNnaFySuucrwzwe14Jsyj9FkCzyhPoiTxKa2sOImaZeTiascA12fywD8d9XUFMuaAoCnmYQFmwk7BCLC7c9cPtnFGRJXb8Mj2hP24xpF/JfjuqYEpo3qGnW7y9t/wHSULMkCYLk21KNjWIS+o3x2gMhbzK2slp9nfo3iZgEjh2TxDHDSuFr7ncWLwq6Y8Ab1NOOWilglosxRpyAspHZtNZws8eesmiZ33ka3nzs2EZf2y9IJQLckp8CNNuJ9evfjLd3/9q/P82Zuv/wttQX3+vb+ivND5nnNlGa6ApJ4CVqY3Mqe6vkdZff1UJMamwWEGXUxvUoCKyW0o0li7yhEd9XigqxwOzFkCB0vMYMbDpq5gWVeggSJK98kP6w/VsGqFtkqCRLTgNmNH5gAkDWOsg2S6pziU5S1zOQz8MIDN4eLl4eIMOMWQBnQ0YKDcv7QD60UcSmwsoiSdsNP/go/vUQtZuBSnmJiYl4OsxW3h8AHln2eMLdMDJDiBQDY0NA4VOgPTqjftEy1ljFZNan6RMb31BN3hpIE52GY2peZ/zHqX+FLEqyXmCH0gJwsEV5s28rkBP8TsaDYzKBu0J9UJgrOmrhMbR3aimzent6NFUtdGvlBttKRGu/qhFtJFa3cNgomabteCnCOyBwcX4EN0LiRBt2vMEuRE61Q6JTvMvQy0aTRFwtHdR1mnITNnbudgsjebdhq7YQIagDfokDcT5/X/RtpZEw9mNS1vmdr5fI8VUpFEX2RLKHJ1Z2i+EjL9KstXgzsK1sBlPt4AkII0AZv62asXZvX+JwFP1rNAz/oe49A5fGnokaBy5Q0kJQ0ujycnlH4eX834DQxcAHz76tn3NkgO9Nik7tsmMj0/5rk5K/ogx5nfbKCO8xVFP5JM7ZZFMLYsdHEq8+NzgQbo4s9jp01Ja0U+sXM1mZgfooVC4lNLp0+w/FgOzQ0R1lTodmF12ZTQGggPDN8ocmbliRCyHEYTYalriVZlKZqVQjrkBiJak+VHO+2R5l5KPc7Kn7oNoDuhRaq32I/QzdrDnNvDFNrvtFSYkZtreczB2ziVqX0tln1rPn1Kub78SE1uRNm6ykl5Vx+ZSMZ0urL35exTOLi/k1U2mSCtXkcu+8Aw9anQFuwOI2krWxjA7MkTwFz+46lMBVVUbzZLrpKDtZB11MyFh5Ij5wmmAAno58IHQ7/HkEoKc93obwOWgKGqsgARjBnRmn+27q26PL7qrqZqaaRFoikti2WGvwKXl+XU3ZRJI8vnj2TOUrHlJLWhCklVG6wXDX3I8zD3HEqlC/wymaAVU2TJyN8A8li83tRa0v9eCA05GkweThJeTcOvsgYaj6dIToWSops2iC2YoM48jlYi8xy0U4MYRX8MVSX4asp0b6OlEpbv0lMgV1IcexuRoJadivpanj0vqUA0JlauQk1UqEoCZS2H+vMzEKvXmQdDYl6VtZRFjL0go1UR3tNoG3ouPAtApRUp1Mk/ZD+AY0Y3o7DG+GIIvR89xiubwJnv9vHuCfzVzPNkmNRCsc2Ntsf5eU4LkQnNFZ/xNgxJK6HtfJuZ2qjkz7drJT3aHhwJZyqG+UReOxDuJpOdGztR0nj07V/xkgDn+es3z7598ahp+4mToDRoVjMFF2BAQqlwSxkKGynKAIrGgRHcBniug9n4qHJDklmjjFVMwsz119N7IiukOnelGzMPlPggzdkr3Rhg6vyVJsfBzJkrTbQ6JEdBEf08fv66jBkKoyYTHEdJhmXp5AGuQWjfVaTxD3nOal3K0c0CunQwZkMn790oZnoq1UXMNhH5KUQ4sPiO2RgpKtrGzMYs+5esfEUcrciBRd1b4hGAm+eYzyVrIR71jhd5982NqhKx2p6SVpPxRNZl5bIPS1hQIXNl6m+YjuS4gjm6D229I8p8UmdGyVaefqyk8eXi1iQcAOWzxIT8Y0NvKcpYz0cZL2Vual2KS+TSCZvaG8Gw4MXQ22VmapRU4BJ1a67mI1RPFEwXyROp0xkAnWQGYHeGZxnPHzsCKfVrR/FLB1HLQkeMR83qqSpRnGf2epsspOlaHuOeYlMVY/CXkFoqU0CRnLgHsZJNwl+FZl5OLSk3DnYlm9rHoLv0WxJ2Ulho5mTngsKTzBg01hHz2KohqMAputsZ5pRCMUoszMZyqW19bGNR3sWq8VSk469ySw2nCH2ysQoNkmjYyTFTmI+VUd+HFYGRSUnSWYX+UTWLj/l0ie0x7asjLGhVpeA6H8YLSFGldgjGJUjLbYgXmOD64XHP1NybEqIlfZgsO7e8D4opQQh0Z+W9BNlawjrmCY93lPk2TSixGDwIUxL/9omMiKAvYsj+Ryuxox7tqxpjYqVeHhtECmIQTKTBLC/6UUbEGg2MYTRbVJDXeVzqV/kGZNUr1Uw6Btx0AC/9deFGncp7UOBbYKOThh9L8bGr3Joqe5AJL22tSP2j1bQwinELuhoW0t6WV0LvrXKRLAYGNIev0tPM8as2XR5sujzUdHewaZ27mXUYw65tU9lSlHVMZTtRZlJVcjkDi3uGW21xGeNAb6Oit1X5upmoUT1D8+nG3HB5b8MlNhQbEuiA8Ljbs7od4yxQuVOslBFHCC418g2xuFJV41EZD9yBpTbdBq7un/kJhu3entHKXCNwkzQPL+PyQYuJXRVNS4NHUb+sXuYYets1aDpMAbyPtoGXLV1QrI+WKkimZkt1mrX5I0XPZgs/5Gfoo1FC04CupsQ3TTm0WLnxki6vmbmYnhMcULl0li+0SGC0VAbSNYZ6ynqIH855HAvxxdBDB8Cpv4IH4Lau/DSlSwtnXFz3qcATq6aWPHoYiWDiIkrpRSVjN5P8udiSwkHGRwBf+OVj8JEBK9KCRO5D4kVX27CINZVByvvSBEB9VtiU2FwNrtFT8LgdjFhq78vM7nLlauimVEEoFCeKG221i9yQElykBRdr7WfwUpEUXMdNliZb/4WDpO93lYvr9OZW+c6PQgvUtEPQSvDr0jTPxBQzig5lilkiDb64o1iDgnmvydp7IraR9Ns01cdDZS3MNNPJYMrWwCalO4bUVQphc5dFkFja1Fvpa0LZso6pZXh001apKcnhw22NS0KGXmzuAyRWhAwtl/e1lG81NN0d37Rut4oILVW2q+SP5X6V+bw/n7oz256OR73Z2K3dr1I0rGxYKYqIt0a4h7+FH73+QebCQ6VRTIvEwpfnqyn3PJSEQRStSZ4Cn5PgTmwqdIjbdd4m6XNKi537pLwXTFklJHsVANF6rQj9yF8h/aycg8sX9+RqnQJgowFYar92GTRVVhe7VapbSsoL96oaJOGKK0UaNLEET7KFQh2Yjje1cfdY6LHd4Q0mcmOHBg+U7di4vyS/nxIX9DdicwnCHXVF/m6kmp9EAdDI0yDSCPM7d7t2p3cjdkbkUVA3lbdUahpXA/ITj8/o+OnmfMnQfEjydPDXWzf2JuwKnpFh4YgLYvdusEyUdUkNHIESW3uw736oZEOnnRKkP7M7jEEJ013eYiuEESAgUuY3F3m2szU8G+yaKBEXuwcJLSMWC33qxhldneNVZgIx8j7OgLuxvOQn3UdnAZA9AHOEboks6ff87gNyPnEylLSkUT2Xl76Edg4CsyHATk/h4YbwjU9D4LLs6bL0tGwsYP9ozZK2UxZOqiaA5bPyTmjTPPTn2pAN7hOl6Hsi5ySOQ7nEMN/PLXapgloVX4QsLz+APmYKQokpVCK6BPBo8aBA0iXFN6XLAchc5bNt6u8knwmhgLeku8hHyg4eWuXQoGXbdFjqLrm8q0DdneXx2N+ptqeMlRZ7dKwyy4vrCfbIp7R7TN7OgEZ0thZOZvKUw4TnueQrMSzo32yb/DL/tsu29N5L4UbOUBYzfb2fUioXWjoDln7qtKroF/SM8x0oio7A5xrlsTzTDRUoys4OBQY+1WBs17UQcDkGPrJ1w0JXJRoEJQjbFGkwLoYdqw8z8WI8tgade+1B9a8Sc1D/ltolocJwLfbraN3F60JBDOQhaHFJqBHqXTUOgH90PSROjIa5PIv8WPXFywNlpyQ/MZJRX6d9oEzjtvpqAj0Hy/HGyUPlEqGHYRwoDg/3EunyWxJg+YkE0Cf4b06B5b8uBXb/HhTY/TIKGNdBxI4hiSctqpFHWJyPqlqrvddOiT8Je9Eix0G9KRJN1klp0+5BYJUNvaTDQfHTJcqJXd+YwvI0oCbuTFIYrbZJzmwY2d+LTZZtTVfXqgJh74XmxR2dDgcKcYEdO/vhcB+PHUsxee6ttjyijqJP7q3bPqLOpn457MAcO6beJjmuXjbnjoJ5RLXwuNHUzsXfCYGXn4nAJen5e6fw8n8PhXf/Oym8+ydR+M5cdFf1DnCZn52YNJ9w0ZJ14OM20jSiYBJ4/76btDbLM/zViqM1+J2YfOyM9CFDhYmnluimxbZMLt5p9++Na+rrURQio21+eN1xwrwo/FMK/27BRz6jlRW72nS9cBP+RSN1XOj36zfOsx+sOjdMvV6WknpJvpxuoEW9MWZr6aQK7ww071T6i7qHTA7axim5aK1jX3CEpXC4yhFqQ3b8cKUHaMrqkO+Frd028iCP+e6B5F1+EnmXBXlLIQ8abHZU5Tci8PJzEPghctQw6Pug/xNJvPskEu9+1yTe/f5JvPv8JDYuZkyYy96FFjMmXsUYq1wsqQUZPharFco5rCw0nEHL1Jc4Rt7ptHtHrPhqKpyugaunaJSIVA8nv8QI2Ojp/Y/G9tJGrS2Z8x7GFHgYjAQeOoPfIx6Wn4CHu+blAfsB3W6wH16+RftBoGB4IVCg5ZA4CgV5AvZ8JLV1q3c2/BNRIlgDIw1oyD2EPUYSN72LfwXc7D47u7gzYW5+XbDLRUegZNB9IEpmxfXKNS7Wp9p6m/uxR9sBljsbN205y0ZgsXbz2Oq7e6tnqz7G3RcpT9LkfI25IG61/Relguzi4377go+6tj3odEYXNRcfl5tqezDKhWKHDwk6/Oj0e/JgMDnKq5Xr4Eozd0i9ODMgvZwdyBr5qV89v8Z+EQV0s3eqHnWaLVwf92gkKR28mOH6934hMlL8fP7yXMYkKdGAemc5qKi4Go7EgJ26Hq8eqH0ORojFXkU/vRBntV++FVFLCn0W19HjdgrKwgA+Hi1t8tJNxufKNcbKYiotfYrzs/ri5+EFTwFPXfW02QtclA3xqji54grNsvfjxgBcgN3KVX1gt4CJe5Cx+wJcFGMW9XRPadXpUgEPRweuOdu5wRYjpGI7oMAUIoB+izmB2Cjubf95z8OuPThr24PnymYOGhnlQeErPP6G+dLPitcU+BT7yoAQ//EOuew9HSGWL3IKRhCbyxKH4t+4tc9Z53vicNdeuoij7fXCyQE3muq5XnEeVkaLm2gBX6+3UEWs22YHV9V96lqyU5n4V2Q9FdnsV+6NM0tvaBV02KcTTRbrZF+62WdbfpFV9F3uAnxmORYphpVsnVY5fafhwF2+4UHd6aBU03LrKW8RF5409T0RWQMpAGmjPo1USyFadAHjOVCpY+MppSKjLsCcg+1vJ5tY7OssmmzwSkbMhwx8sWsgNjMfGBCIN6UCLHWUS1P9LAFQB+89LTXYHW7QrTTYkAeSt8j70gMhbQ/0vmg1C6D9StVZVNSrohBTDmO7QaUziR8e0W6Yt8vPgX4XpjwOuCslEh3lF9NZRKToYAOSVCzrvNuwj2wJ/+/e475flEi2fkyDdiw9YTs+++JdG6h5ySQuJd+9V07wUYb+tm1TjdIhPnEl9mkG8l1agLDtRgps22lWoeIftHxn20SP9zbmblDPF58SIxE0qqHCEk2aJVCymr7lxgB4mQMWVVTIspERdA7Ttg1Qd8dDvVPnS36WsaGjv+hAKyOMacupWg0fqxNOLevLi0YzZlMvvRU1qtyollYAVt7/mPXV8py7JQ60QnHcDk9D15xOBTUPONFSSEwmId83TmU+AMRacRZIaY2OFMwLcI4pZ0Ov+5QOqNr2k7xHMnLRNGZB8MT9q8Vx4EZlJPnJ4Pz96t6wrBOSsMWwQWm7c9RHdP6a2GYyIRaK3T0ovDhNGlnvcd0XQAJkMK/SyQSTT+AxqLxcnFCmztxVz72vMQ1+3gkjnjwhwcQZgAytyrZwKNaqC8FVW13k3FeSAEwP1RYiWWuwPNhgmTcobqfZigMLhfkjLR8UWCJ1Gu0pE6m1yALA6gbbRGU8MK2gHx/xvAweZl4u5Zed/MTDdnKHV/atJuXJx7KgXM5UvqJt2dUZaeapXPf9chARHQYowVCERn3rcjRPUmljIbWr0X4VZv54k5hPX2iHQ2petAS7DNjEEJVUbYCmsgfvuJcpISA932QxPrpWAqdAbi7mb1FtInpARygsJuYgdc3oh2tLampqlk/pI6FG6yQwxS/tIeDvIV0shUuM3ZzJrtKbK/2q9lpaqXmFf2JfdzM6UPqbd7WSATjg1+7s1tjpEoeVXzwrPdjNqoeoS09K3K4XlrGilxLmqg3yFLwGWDoODfkxyr2rRfLnmXekUXHVFdzPEPy7RtNUK1MdeLTEaFBv5A10OlwvjRYkfU+zxiLdsekVWGpK9kG5dg5oWTBJdSV7WMeWq+8OVd+Vq68zdxV0qMKensqHnrpC5+205NJGTqpbqck7Ih7pPqgnfZt6awXKdVdbhD4qfceKVuk3GrcbmXSkb8gQkVdqlHb/19TP/PzKGGsLDIOnpS9Q6vOYczHgCiO5ScLjFPRqmjh8o4zyVA5ffYMgpvLkEcbzlNv4zmiUIoC3y+6FFb8fKf3Kz99T5r6Fu84OEXDKmox5dbdgiNxSTMsi202LOeE9YGx8MbQEFB+94BX3fDzU1R+jDU4FItbc79BdAJ1B/8LqZOkpKwKUgkxcyCEK4eEOSx7zcMYprMTkPGMK0vjmiwYIoMDWU0SjSY5HhXMiZcFxMR/wtEqxACuirZNJboFOJvKkcc6hqBvR3sBPVD6UCok8kpY4a6KxEEh96JIiww684X51cVpePPfKCsQQwKpZEfUqsMry3qvqmtpxlvsOg9ZkOKux8coERIJYJiS9+OFZ9uMnqDOZvAI+yQAcVAKC7UZiiWPYGedJUfc4WzqUrt/xE5x0MlyeRsBuKcwS7hXhTPiG0dDVGspdPyjYUNF8Zt2WP9XYUEHPQaKpCCwjzmB64GgacsxjsRA67A3+jcZ8kNb/H+/FVZoewQAA"""
WAVE15_PATCH_SHA256 = "45bc801fc6a33b5aa0b8a0d95284713a3d7ab01fcec9b07083c3281cdc75fd17"
WAVE15_PATCH_GZIP_B64 = """..."""

WAVE19_PATCH_SHA256 = "7f2116e3e4e44ae5124f590457ee005f218ef546558b5b268e43a77162db198e"
WAVE19_PATCH_GZIP_B64 = """H4sIAKlAlGoC/+08a3PbRpLf/Ssm2ooOtCiIAEFKIi1vZFtJXI4fsbWPnKJDgQQoIgQJCgApKY7++3X3PDB4UZLXG11dbapCgZienp7unn4O7YeTCdvdvQgz5u1dROOV7+0F1958GQXp3pW3DqxDd7GaB0k4NpOUje6GebIIrtgkjAI2j/2AWZ1O33GehAs/uGYd/p9pHnr7tjUeP9nd3WV7frDeW6yi6MnOzs69VvjuO7bbaXfYjtW29g/Zd9892dnb+4b9AwCZdcgEJLvwsoBN4oSN4/kyWKTw1WcTq89O4Qu8fhknAfv5jUmzOYpXoXexiNMMZseL6MZkp9OALZPYX42zMF4wL8uCBT0tvWzK0sy7SdlqkcWr8TTwOSaakwSZFy5gPW+UxtEKCMmnZnEUJN5iHLAwZVaw2xuybAqPsM5F4s1ZcB1mKUe1iBe7vwdJDGsAK1JAwqLASzOYEMD/SRDsCuqYH+A24zSkNdJVsg7XQcrCDKh6srNKA8YZOxgoSgaDv/90LL+89KJo2AiYBJMgCYDmweDzEr6EUdRGAuLVxdQFnt4OcZW/nPnA+XVgvIziRdAGDi9vWudPdgIQCXA9mafs85MdBv+dIu1t/vx9vErg8RYxTID0eJLNvWsjHYN80gHbnq8ydjbp2uctOTsKMgYg7IhxIDPMgsRomeN4GQY+PEziyDdgymDw7uQH9/W771+/e336S5vRK5jZGmqIAH0K9B2xjtkBADGEigPsW+hLuABrKCrwv6cpzDPgcxcJapnB9dKQyPE/RLxzBGDi3W0t8hLGvSOcl89QnFlGYWZcD3AbLbb7nOEW2/ybxplpCDRpsjEK243i8ijQPg0liDEN2wDTylfVTo8rRG9w2EsQDglGCHLmjj04B6W369q38SrTRSvejkEJ4XVJLcVghgo04HoErwp7XrhZPAMqYW+Iw1TfPTifafh7MNRhp4Hna6D8aw0kDrh+OJeg6nsTbJolIRg9HVy8apiRuktQq9lanyLfgabVzLoKfbA8R2oXTxWVdASldmWoXWBpFSM0FUM0JBPfjYKFXBpshzvy0pxUtgNYdpilqTNinirMnAANsUQ+W9MYYJ6yvcI+h1XgSxesGqgk7ITvbQdmFXZVmYHs2r48w5mmSfN3FPz5sJYeWkPS9VSXVg08mQR+Mo/Y34PxYHAVZlPQ46U3DrMbI2deSzFdZ9FvgkUak0tcUpR5nCzawm/N2853glsXB+1s5pkmYGjePP63t8fegieOWBosvQR94tu3x+C/0hS2h/sCVxegEWHeeLyaryIvi5MB+/HHdi0u8DwL7n6uYjZOYvBKdDDbfCRe4qn1IvbTT2b9FpC5fpyV7W2Zh8a212bboxay8nItLfzv4dKYrVt17JT4DW/aZm4LnQPZS6813AA8KgKPmoCRYjDkHqrmaFoDdPuo23DbbBQ9dBvRn7ENoMyL/l3SiB4gjXACLjqDY5N+YwiNJVcyGGAA0riJL979gznwYGFWOBE1AN7ei0Ei0Fmu0qmBaJ9yv5DCZ1Am4bZs9mTUtp3bzlaNZY29h5n6mOwd4oSA4SwGexdvsnfx2qT4BCxLeXWS428gxqsgvJhmrVJoB/IMKHEAA2m0msz1+gHmmhYEkwpLrr1oFdCK8VoLJEmDtkV8dLaGza0Lm2tUraeAFqXOtwJk0AJ3HwPt620luvQgxVl7aL1TYzHgIUAbnEbgw5e+QwEn+MJnYLKfFwJxFHiGfuWIoNkfKmQwMExo5Yuac29p/OH+Ud4Xn/4/R+Lh+XNmld1CCeTZM2b3hndhsfdLIIZBY+ZV4i2X4eIC5ADKcm33nJ77vXNouc73/Zfuq1fWqxbOdzoUgqFv3GOGBVyghR31dhd8WK9V5ftTZpsdjdk6D8ZxFAVjEH/O+nkAkcg4NSDnHWcqUIbjt/BDH+iVr4pBP3ygWEqigEPoQtJZdq/6cALBQMNwkCS2GOs7pTGI/StjZetIO9ANpNpDqxyBwlKYO3nAxFHLBJILiVO+DfFkonWBOToQmHSPz2TPMZPulhUr3614klhAnmKmjk87H8QIOGKw1cFgksRzWhsEW3yhTSb2FGZ4RXhl+MUyhthZWxLXZgYtu0e4WmZ6mYCa6HrihYvcOI3hpGbs5MMnSghhj1RLyGVmyNi/LZMFfJit28rCoI8xbMcRZ92SD7b42+cvWo1pjt3rV43gnVlKnoNEKJpStqfLUO5gwAoJHeh+DiPzlgHrtPWZtPCA6eldYR6yIgeCwLoCIUkesELiV4Xh/BiwcsJXgCQ/OmCW2UFjomPEBF4IW8Dfaly6BBYVbLPkhHCiIDdLl9CsAg97K2Q8MMPWZ6zvM6ObpzrKWOBhh7nrYPzNmTAnrEzeecmKUNXqoZMmEKHdd46sUWxftiWDtmfohNs8kCCq4Rm1T3KhrsaB8wsTiXIxUcWOVMh6EBrcSxkLRaAV/iYBHfFkNcbKJVjeci0gizMPT9ClCSkmGIYdNuNPmnXevqaoVZXIpmhDtmcVcywLPyrsvC4aN50UsHJEijBsMBWWhrlmFrujMEN7fHTErvOvrWr5y4i8EZo8zUEAnWfGFrF5CzhEDy0wilvIMnyDf1vnFcqrdhSYig+4Gelbt4Xc8/W07S2TcJFFi2+MovvY+vz51621l4TeIvt1a/Dr1mci+vbXrfavW2JVeP9ZPA7Mw+BWjMDqcgQe5YigC0fEoxzBfNy1gh4O3d7ebrWrjgzd4bMjNPj5WImxNfugPRTERzIhEvS3SES4WK74EGkWUYw2Ad8AoSW6ijqBdgzioz2hlPwrB9ZtzShIsxqpCGGbHXMO6lkZJcHnMT34fcLzHJmhq0Oa+YPBMonHQQqnCgvpht3SC6n+Pfod87nnXs42tjsUyL26HSPHmhyMuw/pduQLaM0Ou7dfaXacOmwSeGk4CuHM3lDPY8AuZw6FBOsgSVcpWRzVJyg3Q96+PdY7IC9i8Nt+3gaZBckiiFJMnQM4B1GYBlT9Sb15AGdplYKsKY1CiCS8ppYJR6VOGvY5VEME+yoowIy6JAUNUh0SyPWWPKu4EY0QrFOFvO4EUgXKFhcU7uT9lbyrAsuyn9/goqpdYpm9azaBpyBR7RDSlckkHAzG7joO/aH2OgvnYNxeA21w8odfuYEiPEMJdrSaAMRg8MIbg1fzX9DXEoyPzRWA+fwSvrbZG5INmDT45nprLwTrFAW8FcODww+n/4TcATgMRy5cjKOVH6BH/8bYMs29NBnvCfFKtQOVW2bXW3hkOIJ/HH98+7cPIhvEELOjhl6fnnz8pI/kQx9PPpwcn2qDPTX07vT9m/w9xJ75iPvjyfErHaM+9ubv2iz1nma4H04+FsYFJrBFOK8A7L56/TYH7OcrfHr5/uOJ+/L4w/HL16e/aLiA3iKGT6cfX7860aihGFgiV7DHp6cn705fv3/nViN0DnF6/PGHk1P35zfupw8nJ6+QzWhBj1Bbh/9Jyh81Kf/LGZzm+MoYR+FyeQMGIY7dube4cb3kYjWHQ51iXxMEFHmrxXgq/C2dUrZNp1O0xOh8DeRRVX0zoFpC6F+oK5Z/BS8wYKM4jqjdBaL+GKSrKHsGeQLaBLC6g8EP0UmSxElB+gaKH+JO/DPjfwAzZXvwFt7gt2EJXmSIXHN40I/PC4iy+dPYW/KH6XJGuHL+yRNXSXq0E1MdxLNVfVs8ifUo5Zkvjla6uphx0XnqCG2Rp7SUdZXmYZtkjqWHs3xVCt0vcd5TfHLVE3ce7RLk7N6QIIp7w6pM9l7QeSJ6L3AU9L2BQRfuT/Xy/ni5yO6APs/DQDghut3CA2gWjqT8b6YfQGV5UANNP1y74yCMDKvfkgpV1mhmtcpz6W3NQKf0fTtXKW1EGJ9bBt43+Je2UCJUP1T/NqrzkhRGSv6z758buu3jOfuAfa8bLfBtVav1ZOdqCnGRuAgCMxZv6YrFZlsHFH3OM0lXdFt5rKLzcmK0/lpMkIi76Q0wN4kX4EtzADz84IcSzE5E4DcYLMAF6Mm0XIpiny9d6f3MoIXMIPKWKd1X8VI3DcapCzwysGhoBX0wV3wVnkkVqsUQny/IZlPdPx3wEKDvPCfGYRAhSOPjZhonmTu6MagWSfmZO54v5cY40JmA5ZWEPWafaxdPqE/iQjqvItr8eo4oVq8LReqmmAStXamOQwr7VB24PJQ6H1avMxDw5/oLCRJD7YWEqbiMoHuPmn4T7YpfFjCwNUXr7bAMhVJ0TA3XBnhLJlVXk84kRtNUuMW9inOsjwArjLpGGb/eJLB98f2myqWG6j0nnZNXyMltfRs1DainV3Tl6arpylP56tPVsLEJ9dB1CxejGvBR13CVCRlmmmbtsGmrGKnX9wepEFXi/H16g/KOyVQuIuIe3ilsWFYu7QslVhFKQ+uP2qB8ezvMP6cuIDYAz9by1Re2AQVyvdlAJSfDy8/4qHDGMaYSVHqCT1pEjc2fUSXNMLavgcU3rT8Y3Tu7EY0fHY50m6uoptDafTQqOrj8ND2ESK0FVnfZb4MZ2WSBJOYvsBn6VaGjrE7inFT4pL6V4Z3xxvM59szkc6vax9ooZMBU6SgV3e2L+PqZf7PgpZAAPe5gwB2vsucQen1TLDsUD4YqRG6d8dLCLkRqu5ezc7aI2cu/vTpmfrAOx8EQvmNl5wK8mpeuksDfKpads1WyQI9ZqSFTAwkoAP68pOLIMolHJX8+j/0VZQA8qIo93+WvjA+n/yxAYsXsSMCbF0HmTiAAIz+3dRG5qjricLVLt4rLzL17zEaowvTmFk+jS7yr14P1jqL1uavXUzujq8/gB41fFMwpEiTW6jfVeW8yuj1nGJXmBDyt1ZMNSNQKeGIceNnvub2uCMYrbejRagJ4C2UyCNSCK2N7TLEnrVyQj4/8hWkm5vRjF469JKr1V9NfZokOO6vAzppgUR94QFOckO+nMgXrug+YQno7zWKfgHxsJV2qrZUGscc0K+wbb46n5XBLrXRei0bsCXDR7IbFxDYKUGJdnrkIWQC2NkOykTqFeuJBzpNTWpiBpySfotbJklVQpKUxiKdEXZ3UO7evvAIsdc8ptL6fxVPixXZxQbXPIrlF8HwxtceSSXhIb7y+jnJHX7yxZKP3xd9VaizFvrheT9nQF99YBSr2xRsqNDV9cXElhVfVyzIT57uuJ1xnVQsNWoW01CSWRx5yoIWK74tJ0bauB9vrVunkb5qpq0RxJtc/6lwd1Uc/xWVzTGXKBQ41Td+p3FmZ5k1z5J5aw83tx1m48Kl/uvSSMLuh9mlhB9hgzLcpe6KCJM4hHVTuRTVc0bXWQsodSMhgSa+LxXnTUi3YmvarYsKzo2JRv1h0xIhIgpohpPLhIqR04Y8/chTPixj044z+YJIEATeEynhoURCEYEYlgJx7GQRZb98eO4wzV/zsCaIy7BEUGQDr12x9q2WC2GJDLwlpwVaNZeW9yvq78aL306ozr18wj/rPD5qJUXUSLANPRuxivHxt4NIFyCsvwZsVVMWSLggEdn8vVhAUEX1vtJtcXQ3aJMB2bvB10V7eG+2DmHDprij4y3m8ky+FUV7H7FU2yKfMtSnz5imbbmwoc8PVgMyNfMRbD/gk7cuKGxT4OzC7ypbwt/P8bboMAn+15BchnJoLGrTjPdpF9RQRQH5w+BVrnFG48ZifEA4wLwEUjkKOQ6xakx7x/dFtC6oYaiSUrbwOphEiwfb22Mt4tYAcf+RF3mJMCZs3gxMZeOOpPG74O5KlFybY5R+lQcJzDP4jSj8Yhyl8U/iw5xfSFYP5iv+gETwL3iMY8QsGnBgW402BOFWIiQl4kTGmOQodvSfoq5jh/YslZAfgD6IbhtVPmMkRpqbmWLlQ850XOFx2niOI1WepTCbJx2AeFYgB+oFTOjfonGDpotMyVwvsabpxYuxaZY7fEx0/zRvw3elz0xXgSLjT1TSenmp0npShqvXiEZUfBjIvgTQX31eb2KZNEOEcW7nKp4rFC5uTdPCXipAKTM4uhAHbMOc08YI/XVzKgoQntPie6ua36tDTO+EESidXasDzo2o3vujeG9wzeH2J41kVhe5y7unEOd9KTlxj/jWsU8/06zv8uGg9iAvFdRehGi+E5PeUGkHudxHK3++Nnd6mi1DNC2gXobpOhy5Cmegc8MLQPjb2Ta6VLJ27+z346vk+xMCpS/c08C4aXiXSfuK9S1eR5EUqfolJLG3ye1BXEFBp150wp9mde7/FCWE6w+/gC1dBctOGmTfn9Rei3okKF1o4xn/nzX+xHaZL/EES3Ugy12AeR8A+E4xWAlFcTdVJKI5JbTlm4s2HpXvZrns7q32LLrv0vmvDe3k9vG5MvwVcP1dcMq8b5DG9jNQa8ctfh5YAJgIH/v5IdfzMJLgACPQF3y6fHTwf6q9HMOPb5JlTfg27/zbxn9lO8T0u8O3kmd15nufePl+dOAZzrDY7AyarakFp2KbhWdNwl4aB7VUAopRjF+xvgOFLaGJogOsKXFwcDUAOARXF0gDaU+tK8VQAOf8sgRPEpCok68wzs9i8iGIIGCQ3YHHk6EaYHsHYG2H6BNNVMpvHa0EzjowzL/TN62Hll6t0UOkQl6ftq2k3TdOS+Ko86wBmZTVL0SyIkgJmYEQE/mrJXp4eix5HGmRL8yLgOJZWWyyfdMWWv4OXbJR47Oc3jvvq/bsTtU2khf9C+4hdQnRsGEDUU/n7Bv67sdbTvtN6Ku7dzleRGcWC2kO1khQA2Ed9jD76YiydRuIwybGDXCQoBzHTJ4yHOkYuI3ovJL6vb2EmtzDDLSDNe7qOtZ5qmi434oeS51anLcScyGygsEs+zj/tyl7yUbt+Nwd8vLodGhDKeaC2k3PQkoK0hlVdEF4h/zW4oGs1krPttsBhDWs0KaVGf+ijjykroQXHvjOs/WkozKWbvzCL0bV/hpxH2lG13pz84v70/v2HQaNaImqkaoNilpjgyGlVFbN6csyujvXlmDwEEAMVcfJPuzrM0faahjnmvhpWaniyxlNNpzS+WqSUu/iQAC8onsBfsmFiM/WiCSYVr45ARyraRDp+0KhMdHKs/aoycTUUR+SwqqUHkh0NZ44rm3VQg9lq59pqWY0A4k/j2k7z2nbj2ra+tt0IIP40rt1rXrvbuHZXX7vbCCD+NK7db17baVzb0dd2GgHEn0M91BB+jftRdPSkHZqfLQB0BYDVBOAIALsJoCcAuk0AfQHgnGvWNVmIUdRYJBNIqRs+UMNO3fChGu7VDdOp4OP94R0c2rHsgzu5tAlIcWoTkOLWJiDFMR1oMvdquYYf+3UwGuvw46AORuMffhzWwehMbNN33ei9xgpOFEBe4UOutg520yxYQmYq/nmrNguxZBJCJMeyGIse/5UyWUkZ38Bx8ZaBmXsgfnTsDt8VDE8iasuZPphUOUpajzB4uLpgdzrXE/HfMMcy4RbJdoZ0coq8ow9lzbSVLc6rhpXJKSDMHSuTPbJ7pZVJIvSR+5V8ZZtLoGFl8jcIc8fKZI3sfmllkjN9KEumrdwVYm1YmpwKAt2xNBfLfmlprj7i0xl+qagPHkvSB48l6INHk/PB44nZeSwxO48lZufRxOw8npjtxxKz/Vhith9NzPbjidl6NPf8aN758Zzz1xCzlrgvZOJui5S0kyfttkra35388zRv6OXVbPl72yS+ohKBWVNYsQ9U4aVbSeD5IP/cHzbO5p9OdfphPqq4mudmYviwMdHm6b9dU4XiI6JgaPUacxquwnLlTI/wKbjvneuRe055t5NXPEolFFIfMQ6DIIhunSCqCRTX6mEtGTvOuZ4e1BFifw1CNC1vIOTgXM9B6gjpfg1CCkrfQIpln6tUR+IaVGpMXUmX1D+5tCyEidlY2RKzkwB/+01XYEzIfbDlYHpReLFgDjNHB0w2aNx0HszPnM5h/3y4uZej3QH+TzOn0syx7FLbxuqzb6cNPZ6+U/Pac6ch6gr+tehvFHfEX/w+cvFfOYE/UVzfIOrUNoisfs3rMSIeI9YxWt1x9z89pP83PaT6to7s60gHlWjtI5xm1864SOLV8vUrYZAWvlBg6U+7tZPwn0PxfDkrJ+6wsVO1ucNFFlR2xprbIk4N0kmYpFm5C1bXLujUtrHwVkOxj6V1cyzVeKrv5lht9VnTzVGj9obeVF0VfF/r5uwPS/t9QwwUvZISF+EcFYy+Xlx7GS/WQcL/aXarfw0r/cwyvBARy3+UhP8bKGwa7kUxthbo32dfMOFaAF2c3Jh1DR7elyNO/ux+Oj3+4aS2a2OrYKRjO+UAUMz+eHL86pdhWYPzzk2/RgOiGCuCSva5DudNnX632rjyw3lDp4frm56qqPpuZyJui8gIlrYYZXoMYRXC0O++kYGE2OJ/n3x8X6tN+1ozqNqE4qOFfma514Of/Y0zVYxZN3Vj03G/qenoqKbjhpI4VZ61/Q/ylTCWsvocfMrLwxod8BqHRXF6qri+GuVBmFMp+1fwWnrVv66VVcNw3h8r9pMgwOPHwVwBUWcIdV6gqzq+Y3ccikanNWuoqE/91E9TFDpJOd/oaAjGjbyE0jfWqWn22speWv1Ky5EP2sIYls8E/qBPtIOxLRtcj6NVGq4rlgbz4g1tXTLJiADMxwE1eNHSiJ1gQHv6+qd6G+HIrBtpzG2EU7HVhZOJsU7lZOaj1sZRe+Not2F0pVL5Zj68YePpajFrswM0Nqncv/vyx7+9ezNQlvmYTRLvAv/VlgFasVS6ZDLK4nkHNBWvjnr+b94YIPNmcFpp/FKan2i2QBtyRAJe1UVRaeGf1SY4Ryo+qycpH67B3MtPkt3NDYU8KVpkfIbQ580glgDZQRfSCEdBNYejw9cMp/B1O/v2ue4uX2hCweY73VKIo9V8wSBSlGIpCeUNyc+sMKCfa/V+xav0Nukftsnu8Dm8utDXaxfoc3r5kXmh+xxNZvtyZtVpiLH9JnGr4U2XC+wap3FYc7egobF5eEdP83DHUW7lxUa3UujqVka7ek+36HSQzmm9P0LXMe3W+aND1QPdrxvm1YK+XiipkOTo1YvKaE/VEwr1PZ49fubbnXZvq+NRTOOIfdq7zZOLOf+9Iq8ewPmYW/3FwezABG3Gf3pK7lqSkN/f/VxNM29xiUKiS2+QOHqoThj+mVRE8Z9PhZbkPyIvqlTcxQvNHNiqk1hrg1TNuq8F9v3cAgmfl2ucVkTjy3f0Emdh2JLU1Q/bku764W5bfOT109yKHTR7w0MtEDkY1gY4oiyNngF/1JWn3pVyY4dbfrYJDVa3czTKD407FBTkDoeiKZF+P7XrhLGfFzK7tdI6aKsdOgUAGYUd5gBdddsQ0ipRCaP90DJLYE1lgH8ear7I6uSq8OmUfpJv1aVA3Tzh7g51NtJ78QnxVs0k/uk0zbKp3SIFn480lew596xhzb04Ua3v1JZ7O+ekzNItyc1W6r1dW7LYahYRB3KGddLhY8iMrywY+88VDP2C/U8TjFUSjF0VjGoD1Fs6fri6ghMbztWGI9X96kLrfrHQuv/3T5NdElp38K+cmK/IfMxiy62hP4H7f+6R6Uruq91Wzoxwk1Z+ZnLHL5J9iaSmffW/q7Su4eRzAAA="""
WAVE20_PATCH_SHA256 = "26eee3ada2b71f3ad3505a1fff603caf48626676dc66555cc0fdb5af8de4fd64"
WAVE20_PATCH_GZIP_B64 = """H4sIAAAAAAACCs09a3fbNrLf/SsQ39OsVFO0SFEPS3W2TuJtexInzaPt9qa+LEVSFlcUKZOUH5v6v9+ZAUCCL9nubrubcyJLeAyAwcxgZgAMvGCxYL3eRZCxfefwInS3nnOYJu7hyk8iP0wP17GnJ+k+25/vyN0LIs+/YX3L911noevmYA5fPWb0+yPL2uv1eruh7x0cHNzTwtdfs97A1EbsAD6NPoPfbhylGTv5+PGN/f7tTx/s96cvf3jx8bu3b+znP388/TBl24HJjtlgNNtjh4eH7CfnymeGwb55dzJmG8ddpSz1r/yIXW795JYtfcdLWbZ0MpYuncRnceSzVz9SOguiLGYOe/HxROfAXvn+Bgr7zAuciyhOs8BlrhN5gedkPhSnvGfHptVL/DTw/CjrXTsJVAn8BCCzjxYA4iPA/thnJ3+3P7x4+/7UfnHy/cmL7z7+LPtvTiazvYN8AGafraBt6Htwk/nQ+9SNobNJfJ1is95t5KyhLzQEj639dZzc6uwj9GZhjNgyOAxj9o7Dy4IQuprCuFJ/4yTYcYu9Cp6zNHNwPE4Yxi58g+6KAaVrezxkfG40lsZsZPU5LOpFykIf+phSWTdeb0IfYALS2NwP42v2cZsE0cVfUub5C2cbZsyaYHuHc2hnxcJgHWT63gHHytnZiWXT5LajBlqHud1ji4i5fhDaXnDViShTYx797bLeMyr8eY/hv0iHMjYW7nhdIqrhEIlqeKQZYyQqAOVkWWQjPm2ORHt+m/lph0Zouw5QTpDdFtDfbhBDX8HPZ7IV/Ke7S99dQW3H8zqtRNrdY3cwAkLhy6apY4s4IXTKyTdGPU6vOHvTnAo20AxUWgxMZT6IKvjkh842cpfsygm3QLN+GMx9nPDwlvk3brj1xKTxqYUZWgQ3vsdBcZrA6fSjFOp4vXfUuAZVHTcDEE7KXvzw8gQnlo+hl0LvmZNcbNdA+SzxL7cBkAdMrsTveu1Yvwe/B4jbYMHKpdnxMeuz336rJj/bSUYSHP5L/GybROwNMP2Mp97xP0DCErnHFfClKR50/8qesicDUZtXyUust2HHGHX/WkqwunsH0AoXKB9IFImWitnTgMjZcxYv2MZPeiSLAGVbF3FyOE9ix3Md4BbkWJgQEEAcGsquxN+mzhxYfHIDMIAy2KvDH2nmkCSA75d+CEBRApQIAmZl4/fSje8Gi8CdcoDbKN1uNnGSIWXGUJ4XA6pxVj6RDmDQCSLIhvmFSUepsXGypU5cZvRJdhvGWJsgm6VZAmNgZ2vnLPa20EfBOXMYyIVvb5J4DsT9igsanlWI8PGU/coL/squg2xJzUf+TcZWPS5KNom/8DPAtZDciX8RpJmfAAWWmgk2SisHpVbM/rRE8yg9gZjYu1dssU0BMLXsOoBjwEW8yNbODeKfnfyoc1A5nSsDQWanNt7CBIUwexydiIFNuE2ho2kcXkHiEkCFgF5kf6AM4HfOmjr7VuTAEG45bq0B4XbU57jdbOcSv7zdD0DCnwskEtdxdk+dtc/84GKZ9YDY2Cu+JOAo8lzOoz0ujQogizjONiDNM4HRi0tnbLtLoIAUWHeiYBP4twfLh5RfHHkqZhGrBc3ki6jAIgmKPHvK5nGsUsRL/ypwffbhDEBuQdQQ9Pkt9f+NYU4gGbAHk80utk7iid7COkbF+VrB0xbIxsVUIWbN/hFi1jQnGi0OAaxodZzKfygpFpx0QVSsx0P9ws/sBchd7Hln/yK0L/z1GknCvpzwgvvdv86aoMzTB8NRWWYHOCD1x0CD4gTsoA4sp+s2eIWAp8m2QfTUO+YT9YTRk045Hf/tf+Ka4DmDaU/jpEfiEOnkm9OzM+ZHKNY81oFp/Hz3+a67r9VhYH09iBaxDqXWzj/iRKukBVGc8Gk2DZpmy3jINJN05/JJzHZD62U5ls/njpIkiuRMaWW8V6SJMgUVgHfd4vcdA/3Zr/Yel841KKasM9bYsAvZhIGhpRkWoGA81IzhbhwgEUQoJ5C7jpE/9QBUpXjtd7rs6VMQPd506kdX0+mVk9hx2tn/5jVqB/abj9+9PoVK+92iwqwMlyPjUVCff/h48s3pDpikyYEYdfkK3gYnV9HKoA5UUGVZ9KhuchUElJEdPVVkKMJ2YAErA8yhkcnw4tuT795gd53U9nxY8Trd6my/XXX2gffY8TNmavUsi2dVqcjGRENJvFM6qvBtr4lngVejLAG98Jx9/vzLPqmIKAcAYb/sTz/fab/swzjtiyTebvKEJPBMT/5KzOFIfpeEJn9zApG/Ck2dz6/SgEQkJd3dgYQo9RYFE4q7C1CHNZZ3CL5SVzSGndByOtcEZWoqNWnqhJUZ9r8ZGbIoEnOBnYM/FDtahXUKcuo2klaxAOQGBTeKULELwlDRF8RisK8CAtpWRRdJuLFFMv5o9BAZL8ZTTlQHV85RR1rGZHnYlWqFFlJamChNB3WyY3TRtCpUlHx+uKrCFceG9deTa26tXhqEW7Q92ivLEjkExN5oOEHsjSaGZkx2o68gQ8LKDhNPEYS8OlnCuRK+9EGH47ZvzFXI+1RHNDfIn3HlBCFSRQGNW9GQB8uw66dgFH9A84VU3ZKGy3LbEV1AwloTCmUBrmjTC9INCmroqXSihLdCdUVFHOzdMg3YgmA7T1M/XJCFiyqtaotihl5hGGmS1hGEpl7uFOCGI2nxiyBLS54D4QKQImlnJyUW7Nzq4/3VWJuZXh3Eg6x8lQTyEea6/XOHm3DCyPJ8FwzPXoF71OzZr2DfrX5l8AHms/SCIU74cAtonTNTH0gB0gVDgizFzlJjWZclYN9yT98SbW00hjKy8Ir6v7oO9sYOfVz5NzF03L/8BJL9gBm/kjPMhwJUlbqIzsQ4d5HgZMTXEXHTeEwG26QPTHWPMGrkDDGfQMrCEaAYTWRYgkzuZcsEh4PON2g3JZQUjk6FlIEDHCQg7LjqWSJ3ZZ/YbQszVzOGwRAmE3gRb5MCHPcQIl+K7oHsByKM0JlIRQku+peiIG2xn2EGFYBlR+YH7meM0CYGFWkFLMwuwnjuhJKBYaVC+zUJNoLC/+cTOjKvO24YbDa302kWx2AQRLe29E+l3fMSL1Tsl05B1JwHit8ot6fs6Qv4o6ReTtmLHzwyTDdZomSs7LmT+m25Vztz423WlhXZ5LsWxmyejIm2F6yr6YJ026ARLHvjJ/bqqhFkCrj1/GpW6jpINYtSIvJmvWBNgKi4w7WjoQmSMe/9dBtmX3W6GvsmPE2SOHmmihywbeSY2RP0DquZipMPanZE/en0NLoIIr8DCwQo3E86dcNrX/Ibau+K5JdOzbzN45E1Yxcx8K9Mudsvg+t2VYvirmZcoH0iZD8q9aTS6/HKjhMbbbnOb79VR1Qdxr2ddSLBonzq93X0j3W6XaVbJbMfe6YKcOjiAyX7o/r9ePTzlS5frdfbFMxH2nkxdP348y6/752YpnKP65O1Cyed9TZjl6DXwp8V/3PF/8RgU7EOZHFu1wRfa8jB3SYoS49X3KT873IjAC5FQuoCyHLnOpLENMnOWolxNZVVNc6cjY3D6Hkjl0lKHS8jRSsxZM0exl6DzrNGsvhU7uFTAor665f4zc6/ufZVHHhaQ+nVo0pfPap0/KjSS+9RxWE9exT0zeNGunwc+NR9VHGY6keVB0J5SPnzWXmZ1Lk+1qnZR7rizCrTeLGnB8uIBjoKyH2x1sGvbrU4WaJGQ44qphqnj0hYyWlSRk9vNmEALFHdRs6WfIMNOCxElQTlEKg+V8Au3IkuFegC1HhqcP7sAQP3YJmirZQZFVeUOsW0IDtEbE4X+jt7sHaDWufReEQ+PqM/NjWzf7+Tj5Tp44dth3b1bXSdOBsQ+Z1+k1+rBOh+y3AHtMyqw8qsx4DLPUFXvvvkU12kAdKSgJY6XqC0HnX2sdl9ja/UwhXLTRObb9jhZ1drqmVfrqxKTZncVg2H194YjZ57WpqrAmizUl0m31PNaq5m7ayWNdbKoFJmdRUX3HmznAAFjhamD2gRgkQAfqevfG679VWwpi1pD9ROlLW9opmIydc323TZ6ewXZsC+VpFWTHarWacTgAoDrrKLOXwxBfGRpr0UBEsmrXMwPlEMkHDhO36soxwxiaMQyJlv+xlDUzsCfkbn/QN8Wg8Q3kyZFHafBni/SKhqgL0HaYC9mgYYRFdOGHgMqI/vvFSVv7om12vT5HY1/Tsa6qobLqU9rtKaV3b7NbC/3EgiV+XA6Iv9qL4GhjakrWOwaP00S0uetjT1k8z2L590WiYEzKfpFPRfYAs82CAlae5N+J9PCFQwIxi+10CYZl9wD5jTdgD6JJ1IsOdoVeNRh8iz5YkRm58Tsi9p4+GgrV91ZjQtS7K2MbRHsLCrPPSQ+kOl/mRsPrL+yOrL+lbfPhr1H1m/XyD0Ua0arfV2dlUsX4DlA2bZ/aMR++oYj059yYy+aXVnFW1FzipTZtUwbemtt8l3YtMpEphDB6RbGOIKis59mw5q2YAVN3PS8oYSSK53135k6sNeXx8+Rw5Zb7JjmMwpuzyMD734GhqLt6HH/BuwSnySVsyc9AYm+qFSQd3DgTZB6h4NNMNspW5y/QQLPG8SR67PkPxAvUJjBvWw6EKccbnIPYHoYovwMAPQaqLD9JLPVgXIS6fk2gVoAR60SBzhZgbLGOYCVCZvit4pTxy7QDF+i+69BOzL2xI4GGNAbn3yW7moOaAHEPsFetr1MgC1DTtGINgiRj0tZXjGLRdP4gTg6dmZffrm4/vv8Njip6dgbc3Y6LxsVO0oOuZFy1KzvLHfqe7xQP9PWBRHPbGljl1MEYVzP7vGE2X8cGPiu5lciHCUWIJk1awGLohILEARXFgcOTckORyEHiswexImkT/OSg2e60QRmOmO69L5SVBzb6ENPJeUSR/oX1LFIzCPPel5V5HQcBqhUz0w0Hg2495SfJfogcVohx/LEhNYBj9yMDSHqJTvFvFPOk/4sQBbnAFCEwA3mzr7G9uZh07m73dVNVmRRaJxUZH2lEGq7H8B7B14+u1+V6ftJtScjBYQuNn3IAD9e/fLOUl0WqelRaDu2lMruoTaWXobuTqs4ReR7+lrYxRNVhOli2VAlmLxNTfc1tTcSagp1p+pCDCbBHsFRjF3YrW9pKX2kwVy/Xy/+ygI6doWQPjx2yYqABJw0gbKWQeRnoJg/iIZaPBhX9zCkrDG44YtUEqE9Hg4aEKTQH0Jqi5IyI0T0NFDx/uH4+IhUZQukx6uekxMIG2qzONsKQRQuufdd1ydJwFGx0N9k920nVuvFhMH2MewNs0Hrq4P+v7oyOrff4C9Bqj9JHutKCnxdPoYPlGB32M6YAgPB7Cx3odfGdjwqH2jwxZ+giYGkwxqV/BPH/C7d6D7NxlAZ7rYJOGUz0Bn1OcTVqWOT+czfvoQMPhv+0fwygJvyuZi446Wlp8/cfdNvM1wmfq7+BlE5+xr9tMnSKYf//cRFk5+Itugs6JDY6ANJuSnODs7sV++fXM63ZNu/Fl+bvrfOpgD9fRnfbtZngGtnULQsSbVftj2G993Fke3eTI/qv0T33SDllMChzJNiDHcTRSbcfzIpjRO4nC7RqUGWIhlwdrX2SmqIXjgN974Ca3aHFriQ89T6LJP/s1lcBDGdC1Anvj0YjpBmbJvvz349vXB628PXr+m/TvauusJ5xTBQqOJO6TQiAWA3Ko9giXZvwnAWAUNzfB7Q2WBvqANSnSWGSM8kvyOX0cgcPIuAp07L2/6cceY3CmkMYMgToIbOrlcOjTPcVbeKlzgzQOudJACAWisbDfiH76hmMVbwBzfTyzB+luxc0m7majq0m5mMXmk/BWE8JpMvykdmeGuTHJjHqpeTPjGd6GBYqhm5Q7AFFn5S76jaW83Nd+81YVclAbxogPT3eUwPvDbGxIELizsuUQ2P/9N5d7zPSHyEU6LnbNj3Dmr7vOjxQGmiP7v5zj9KkgDPKWucx20RTXgq6JOrlqmb4F6Nval1pS6aky9akxF4VNJB67Z2PnuSlMlueHSVjHfg2mFLLZlyvkLyqedmsaalclvKlLaq9k7EDa5DloXFESG+mLzldl/NlPT50BiXyy/mlRSaUX/atJvSHbsZdDX+F+D/oZxX/zF33NIpz9hXK4M2Psi8b4yrTJQHPgXi+ZkFwG7CNU18UNer2he8Ro0qpk8MhN6HFc0idANgPkJMCbdj9Vsk7JXbdkDyr5qy7YoG1e8WgFCLG9cUlkblCGVEuTWAsnMIUmyayk4KJrkBFIrx+eBd43IsAWSJUqo5NhSlI9AJctiQtyrzNGzWOfSVo55rNHkzHaVmVAZc2eZIyoz2FnG6FMha3chgwoN836v4ysxuhHkZGAE3QgA6TIROTQKyB7Oqs5MEJ+4jIgbIpEnOG0iKgyMWb0C6svVFmh8UMtsagF1FDqx+d3LakN8zFBxMGuuxxUXWbEYLOFBGH2zasXqwSKlmplXK9AUys5gJn1as1pnFkGSZsX6WgOM5kbU1CGoK9bXWh0g3ZKMmNVKAMVW9WZRCLRvWQgnHtlAkpeCXsrCz/7Ngv9zZ4XDSL3dtQlASS4O9NHlLDTVirOKBVZBqRICT1xIFX4OL8g7TiTHiVVyxnob6mFcLkCfg/pM5LkKWyELiNqemKqxigzBIGT3CaY0zIYCllZwJBXIx/wijkDtykjBkzohaYLo8gNV0QlCrlvxewH/9JM4pZN2ZElyUs2BJaiAiYtNdOfUSRK8bYvqnTzM5kR41I+fD/QvQJ9EkgfpuvWxwRo1cLaEHv8E1PDOpqsGU4E7P9voFz4vuSG8Y3H0yAoUfA3JbJ44jFd+f3ry8ueapDCOZM1RnQXw+m1YZYGC2Mx+XndQY0lYV8BsyY+OFtRrGlrBekezJkbGo4BBxNzlNloVOOHrA9TrL/ri3yxHPuq2qI/jHhRtk0N9Un0v/Hjtg2I3pROA6HBcBBGMCqcZTBAyGkY0yQUwMjioFPaEX+wC2BWVtJcfwuyf6/nyI6U3HysujyTE5QKVbudKHnExfp0pkxrKSTU1UaxfzKipkAdmGaWamahJLMERncP++gmkK/Twv6fv3zYxqqlUHtYY1bQkn49qcknU5J/W7ux+HXKe3SYChrxAA4dTjly7h7PqZOSUQ3MxPM8ZCnEwLRpLIh2sUV58icS9ULsCyZgNqdjTZV+ZUaxIbVgabwjKzFrgImIXVn34w5yF65jjUp3TSj4nmVBC9S106hOWOi/1q55/YPatCRWqt8FlDbVvTkRuQSwkeXK0kSgReFNckYpgfYs7L0Lokf8B5dwicS7I2uNGeEwbH9yovXbQa88FMbFZDklIUXGKlzY4irupYACj52zjR+ihl3b1IdjU5M5QWCOSTEVIHJd4yioG+sr+6eS7j7Vl2aQKrOFfriisfDTu2aSHX1C4cGS9Ov3ZxntnjVKbEzSRrSK0hzu6kxslNRlY5Bo7c82duYOWXIGGyQ40vCpEvpTchAD7xbc/vHk1zSe0oIMpy65j1YNBbiBIyn2yOcQUr5+TIqrXGIdWsKNCIBVZA75A9Zt4iufxz5zgFZYjqPxz0CCs8uwGyIbCrUeFMJLcqNiwn7D0eXsRQxQ5wGW9tRyZv7wcMXh7uRzeoD82z1WGfa7MCmonSMXcs0eYF9o8KKd0WCSLC+edXhu/WZD1UZXCFsNd9LcYNeSWlzYShbwJS1nZRjnTPFdXNoUWBrJifeESeYOWyS6y21Ym3qmmlWmkKTqqMWpZmoZiaRqdtxQYyQIHlly9nu9cvWgNGratQdidxahxbcNpWzYve7hCLAdNy96RxgcB1ZuyyejDMQCMti7RynnUlkuwDZVeaGq4q+czH+5ycFfPD2PKR+jL4V1hRLfs1ekgh3Sgezlq2QXluFbdJ3SHTZS8UpSCnaMv9QqzP7MXYfzn90LxyP0HcVHvxX24UBaAiSY+J41yaCyzR8rSPc6lkFj2CoIDFVvyA2+deMKYNWQbsnPN2absdnP2QBMfebYiyawdC6KijQysil2Gy0GfPAOFM0ipOipMuqOqL4TLMcWy6zfbSoNxu600GOe20mBcc7aA6aesZG5/KgxXbCxfuEgtE76lL029aUppxvlYBuPGSefqwFBdfBD314Hn52vBWEKxGpYDYaJwn17uzQDKmdTxMRH4GJ83Wofck8Y7M5goniDhaadGaEybo7Zc8dlXFlKj8Bl8+Ei3VnIqc7zCShxIrUv6nAZ1W5GXwU+zRm0WV814749qJguS8icsdU6sIpY82aFpDZohcWG0zxsvZO1ApgDzRyLTfBAyLeOPQ6ZRRqZZR6YpKbiuFlsD0dNmoUgcJOqPdzKIZf7LDGJZzQyyg1nbOciy/sBJH7RPOkfoH89CZnnWB9N/mU3+GIyhrWy/Of37x4eh7I9klIFAWd6lGqfwxRI/qx4TafbT+q+Y8judJnT44pr7S/IVLGXXGrs+wJ2Dgwl+GKYONjT5VvJIbOid9KN4e7HMgeFaLc8ZzH08veALtzUdf7p2wlWKi2JKvw8G5pdRzf9sEaLGAg/v3/5kv3779vtmBzS3yFACjxT/s5ljBM/P1OdDURysut9OFNqhGFiFYmC1KAYNfSUC4m1biogxCvcojlWlwqoMm0gITTJsUpJhk7ZhyfU9N9gVvHD45AysOVqKOAAN3lvrSMsnzqh7RER2I3MMFeawqo75PCIg7RGpe5HCxCOv09/+Nmn2HA0JGRNBSXgltp2SaDZ5BWuizE7hpsP6709f1kY3NGXVhtHRpPMxDs2ai0SMgvuIh4PzXPzcKAPMPxvAG7LpgVmRBXK0yuCh89OKqZoMLd5APqpFyA0iPNcuy9DkYEmDNmuVbb5F1fZdcLYaDmf3jeL39mHyn++C9Z/vgvmf74Lxp3Yh8G5qPejf3wND9KBBduz2Opdlx+nfv/+XZMeHH87+JNlRdoFxg90s4VS125US/cXgb88nJyeD56Kcf2PqzmaTxPnUWaVJq2kw0AkqoK5ORUPcOTColbhXkkncS12So7JNkg3+fZLMktR7/0B+bzcm/xW9sP4remH+V/TC+LN78TsF2yAXbHQ43N0oPRnmTRX7Xz9O+TYm6eGk7CvbXJSI+2BCK9e4jo7BdnF70UldP/KC6CIHFyce7o7KE9TxNtts6bQCRpj36RZZxK5x7/QdKP0UbJ0OwmEsu/gfviuOj1ck8mjnLg1a6bsE9qjYXlUE7Dg/Mta8iUKq6bDphI84usPP8RhHQvyc/LhjIeATOKotBMXmKlQnY7je0Yms27ASHCkrwaRtJeCa/fDovFHOT7Tis+2sAg6ZjIOjth0hsy9LgEWYmxCLtaO0NFJaoi2a3CIolxsX5Uzult4xC2g7DOt7aBzX+GlUVg0xT8WsEdqnDcbLqF8YZsag1gTP5p9G3fAvskc7q0521GyhTVMUqGOF58ijnKZy+EKZrU+UdV6agaYSNJM5/lU7tO6kG0o7z6qgW5rpAgK/t3KQ31vhEd8bTr3X7x/y2y7lw+fXl6nWmJ66Wvv1qPwORst7HrV8cSHKMOYj4+hI153BYDDyFu0XouoQ6jeh6mV4aHh+Bwr+HMnw5X60XbPTNyeywvcY2VEJYU7R9HJJOw+yHl0RDVwnnIGwdfDkwmLhJylm0yUVOqsiYuvi9d8gUmKZh3SdBkrhsUON39zlNfGYIF4M4ZdbANFxwG/PyPAr71ZWPXK8CFg55YH1+M5LSyx5HqqRbhrJOyoFOLqs0hxink7wpMtgs/FFvAJEUeKvaYR0gM4JwzmsQWJxOePRoikQBQWKqKOXR5WwZKD+IqZEy0zIf5Xs6fQ9RY55xkSolHtKf0PhYaA0j3VyT2lAOBWmCCXl+5y1smcUpBwK42WSnoji8YDevFuZeYcOKILKgypZaiVLrXTHcWtioA4UMpzS78OtiADth4sqxvEqPjTHH4Khxxfi6+Ltjjjyey8+nvTwxQgKFEbBZbo8zBBdqgyyGjzQSnwM5OWwZZBmGEgRz53QZTXlmQ+aWS/Gpzx695PAb63TB0JrOl0k8bqDcZP0/D7UPRO6C+pv7fP/uR5WrqUD5YJ3NSxRpKclf+dCidEijmdTUmUCaANUoL6OdLoSfyuQnyM6f3wBz4SL6dAfwkW/tdPzbzupVgRJNkkUm6Bx88gMIiAmkCBIpo4Y65Q9zYPMaBT3ClJ+fJ0DfwEpFK6xhbSDhcSaroRULseakJEamwggj2JyUIHWEmhXmdSnT6m/evmyWym/BVhDQFyCVD6TnAdSrL8s00iceeiOKmLymNvKKFh1FIJoMYYuqHFZsAl9O150xg1F1QtCgGkY9pjP+MTkYbEtzRwqMy5uuCqRa1rFdk4L02lzeKq9+xeKXTDwnL57uwuKoOEqkDy0VjEXD5UQchraI8GqYX4aXmu4bEhb2bRz0JBz1ZpTXFAsNarMflt25Qpj6fWK0gXGxpoFreyEX7rIWCui3mVs6D6FbE7bK5cZqwnJyr2ySnbt6RLBhW9XHVSUyiG7HiTg83eOKFdcSnAooKp/k4HuL+PtaXtNEp5CzsiAfoWuVwo/ky0T3ELk4p/fZy6dcxZh2oUSKkKji+qJL+rozVYAhRY5BJMB360qGwDVLKH7Hw3Hhn800fWFOxx6xqBZ969VLqv9tWzSgwajESmZ+Nfom+LdNYpTNKRoWK6TSvOnO1OeSMvv5ssH5qoatRL/HPT3CLQi5bEmrERLLAemxs8WMRVZR2jbMrRPquXumndaRQfvime0+OX6w/IjWg7L4hAv37s+3XtnzgVq5vyE/TKGLzE0iI+oUYwbGgxgKYXOZLF4nk/GbjeOxLiKQHCg1OfBu4rAXfy6dGXlEkFLbN4eZNjFwCmcV+YEoc1f8yoCeWH4FgqQ1SEJx1YYfY9dbLZQhL8rI1lKHt2D1evJSl86KXagHBFMedjhw6vvvp+K4MPiVUAej7gHiCg998MfKCiFhOEtlt9nw332Th4VNKLYtzzeLo8zcYkRzLrou1NiKXUMa4v35jVmir99+duySimq3Olw1yzusY/hfynOaMckhymkD/BdHZFzrmJBja6mKhZA6k85kp+ulACnbUMphk/PATwsunkluFuleeFzqQUzX5VUPZ5WBBsnNMlEihWupPBQ5qUyPBa4kkLjKZJKtFdoaKCgUYlZkYmqCb3mQ/clDwj0rFKXsy2U4YV1fB/O3sTXsKzFCzu7jjtdemNjREEUZHMVKHIZxMDIYuDskEbbFQ90KBVouYOSht6HQnmEZlwkKDhDeplkHbXCdeCBVDqWKG3uxmUxEl7+gOOtVIS/SUABJoCRDBxRXu+A11PLYyQm4PyrDtUEou73BRbpvREM/wCDULu6cvM6OHrZVRlf2uwbAEBMSwuIq/tAmPeCQMUApRCtLaTliVfhphXzQ2U7qWhMiwHWwuFzaiVqquXnBC+nqaHE6qootLqqlyhC8atUsSO2vkrCtZKqXnWnoAdDeV47oJaIwLV9ILsZH/WXnAikm7oBg/hiYoK3YKdTqfw/vdRyMkLh5MLHFX7IhuArTkm3CLKAb3jxIKGKaOxc6kBmFL5w5eZfr4qvCIv/QMq1IKUjCOGAEzMmjSwZ8FCQR2Xk8+0CBv4cd10i7/kW3X/TaeRfiwWMulWEUlQqe8gP2w2+xijKPhXw4NtlqeRqR8mVWyp6taPoVbkoKPkoy7YLnV7cpXgnCkryPuveJkuUiki5Vw4a5D/6Ln8gFYRAX9ePFTZCyePGIRnwpVahttpqpyPBiWn5kuWY3t2F2oRvQSlboIZAj5dhuEaf+1rsxLkG2ZVkadEY6AsAEiDjO6bT6VfbyTMMpdbUGUnrFFp1mcVeBwehFZ2oTe9qhwVXsdw81R7yVuqPK/VH2SJrlQntfE6IBwonaaMgueUJjp2CYrdskKtuvVSJPkqvbzTBFT6NKnJpGnDjFsyWCBboTiN3IdXjawslmVRQ97kKzMviJVH/U1FLI3zX4CLRLsDiIc6q5Yr4fW4Y56oNuUIQnvKTBJjiGvj+g3128vHsh9dqoeZ3KehNCvkkBTcwaMqOPwt6uGOrK/yxurqjVeX4M37e0WxAOnzesUtYxI8/01quPj0hVcfuTHkcmDv7+Xu7SzprioGLGVgsWwqLic8XaaQKlyK0dpAAJ0cjbU9uJ/jJ2gcrAl/6niBfU4ZqMeaGHRhK+H53eljojRSYCaw4Nn9AoT0QvUhnpMj70ngUD8Xzf7ruzvvzkeORTXkIRsFhtA1DMiEf1AQak30NdBdDGwwwgB7YTU/yTRh8S3ebuP5h7LrbDVhitxgADeO4yje1FQuwiFJ2cvicwlw94dDegBKd4MYS41YsCwN84RzsiV8rxu2vMxWgMOZxi0rW4AAvwjkss0thPPo3vrvN6JlmJ11xZ76HkWmTvJciMrgcjtjQygKfw8uWGGoXr+2inYmPTl3Qw2CavM+9CSL0ZoBl0+OvZBEl6bhm46FkqQvwVqfTzxSmMH9DTWNoEKDoVQtLV/A0Nw9IB8CHxMDMLaw/NArL8FotQ3zYEpaOvGDJ8qOVRLzdNp36N0HWMbv1h7qxKZA0L6iPfBO1q/s3GDJWNCCeyVVVW+HgOVYdiLRuP+XiRdYXEWvFc81YIodDtq/wGd5nAXNzt27pPmbsAzXos2IGOxgfmXdEk6HxRDxpMoEVv2buJKco9RSSu/KWEkLDd5SKpwvQnZynKpHxq+5TdKcE0dZvf+wIJBdQG+AcUVyujIaVzfNtio7HV8J03ZEDo4c+yqOjBYvs9QowOXmFDKDt2BKCixdNy1Ien+xcgcT6ZX/6y75kv1/2tV8EKVD6Z0TIHaXy6234yib28Jf9che5ioIvbpbT6UHOhuFiSZ6OL3TWQ9WSA+D/Aba+4fTAgwAA"""

WAVE21_PATCH_SHA256 = "7f01ed0e154cd76312b27af2a436499ab5559fe07c1eaba713717126c8c84d5b"
WAVE21_PATCH_GZIP_B64 = """H4sIAAAAAAACCu1963LbRtLofz3FWFtxyAiESPAqyvZGvq4rsR3b8mZPKT4QSIIS1iBAAaBExVHVeYjzhN+TfH2ZAQY3inLir/bUcSplgZienpnunu6ensH0
zJvPRat15iXC2Xei6bmXuNNkFbn7Z/50NXNakRu7+H7/yrl0rU5rvordWSu+8s78VWvmxt5Z0DpzEtdczMTkz2LYCdwrMfd8VyzCmSs67fag19vxgpm7Fm3+
zzSHTqfdm3d2Wq2W2J+5l/vByvd39vb2/oL2f/xRtNpGW+x1jIElfvxxZ+9v4leoJqzOWFBFsYxc6KEv3l95L37+IBiBQAQ7ewD+N/E8Cn93A3F+vQyTcyiN
8f3xuSsiN3G8AFAQxo4lHrfixDlzxYtnr14Jx49cZ3Yt6FUsnEC87lgj8XZkt8WV652dJyJByjjBbGcvcqEvsQCuhZduJLj41Ugk4SdoGuFiU3VcQC+gz2Hk
ilkIlYIwga4sXSdRHTB29rAtQ4SR7Nvw+xhg/g2EhO6Ggdua+OH0E7w68+IEGkQiuMn03BQvoVtunMTYigjcdbKztwq8xdJ3F26AtYEdCzGPwoU4XSbrlhe0
Fr4iu9W2Bq32qGX1gfinTGFCtHS8COo+f/56Zw9Ju79aQpshdsgLYYDnUbg6OxcO/Lx06NXFygkS73f6YSLFn8NgZm4ACBXHwsC/NnA0Ilwm0BERL+xhXzw5
PhLTcLFcwTjEoEesFFF4hTyYYW+8CF7v7C0cGLAXnAnoCxUDSRGeaA7kRhb/1LWEH4ZLQyRXoXj16kh4QZxEK+p2LJZAOqIkUHzeGUi+tuKpA4ydhgHwMgZA
Q8wBjzOdrhYrn4cXRjM3MqhHJBHMDyfykvOFm3hT6N4qTlB4roUHw4ZXjg9dI2qmgvfJjQLXh74ezZGLWDb3AgD8SXYLx3bp+CsXhHYahXGsUBBRrpxoGafE
j88d5NHCXYTR9aEO4SyX/nWu5Z09OV00cojYvVi5wdQ9JFBkA/wNkEhA2UWc7zq8bQFRcPA7ezqziSZXQAiXa7hrkAoRTyNkl/Cd63CVIG3j1QLQTK7F6Xwe
2LPwKjg11cycAg5vhgNYAP1mXrykyigw4gr75KD4zBE/yBMMW8Ag3DFWb0kpQl5Leamf6N4C/8X6zqXj+c7Edw8RxemLn598eHpkP//w/tlT+/2vL4FUDzun
wotReGNgJ4EpRuBsAGmMvDVC4Fs/dEA+YNJCO9NPrhQShuEWzr0ZiIU98xaEduZderEH7SNFUM9g3089CQDvuhbVU3OH1A22ZLWeirPIA4Y4yTkR8BlI7bVA
XRehTCxJRcHUm6IK/+S6ywInSVBQ5ZG08QPWIPWqBKUlWexKmTWlcn0SRhEogcCNeXoCdcJVNGW6kKbtmOIfIU0FlLNVhNOAFBRNEFAil6xigKgRqIOU2Wcr
J5oZ4vWg10LtQ4OEaSqEFPQWC7qYhKtgZsjG5ZDmIB8TIDz00gJF0CNV58SidXnKzYK+DaNE/O5GoYiXSFCs7wjfPYP++c4qgB5cwWQWoJV914EqoKeodRgh
TWiaIKhA3r8yxTuliKfQm4SQAedh0qtZObnG+YCSBuRC5SHHgsrfC+YuUHEGve2a4qmHBBVQbXaF8EtUKteAGJURMAX6mio5Q0TO2Rmgl3YGZDg2lNgTfqDv
TE7v3mjQW48OBqnIknCY4kk62d6OVD+R/TC7SRFKTk0Y4cQDVa0rtJwcIdo9cXrm27Hnr2xQl7YSHPtihBO8x4rZDWatJGzBH1ClzhTk3tPVRb/d4iEpnQRE
Ei4J9gK4sUKSOtGCpPAouCZ7AUbWkKwzJNXIgkr8cyAO1AO+hHIGoP9xiDbbYyM899Yo8KDsgPAzVI0wUIKMnYWrGkYzCjITeUGiJsEvGZGP9h+nvsfjEOUH
NSd0z2Wbi9ZuAioeBoJExp/oWvmo0cIpW0uYVIsF4Mp4B53ynTOWn1Q3vXn35Jn9dgR6yUhfvnj38qn1FF/t7OlKDOBsmMXPcsCvj1/+/Ax0Te7l4/fHRy8Y
jjwb9f7o+Pi1DUq1B0WHrImrtOSpmJ47AThMJivjpRegbBz3DPX44sWH5+L9P47SF1avJ5kNA14s5exx3RkpvDmoE+V+gM6LyW570gDHSG9DPN4/kv/AT6rl
OlMiPevJPjDABx3RR3u4WC15inTamSyBtYqI+DFVZ5nGuf/oYf87AXMMPC2cIUr9Qn/3Y1YQLJX5HoplGHvgCrE1mfoOvJ1fg64GZIVKKagA/cT9ccgMgJVk
z5Cnl+wXelwwFf3wSljf4TiuwcmDriECRGcoRQ8djmRnGBzapukgu4iTGfVFCxwamGDXegVngkp5r/8dNepgH3DCh0ErhokM1h1oE8HEhCGDTAJGMOegMOAX
zDzUIJqGgDG4kzAEX3UVpPYdRj/zpjTxJk5MLq2uqa7QvCfeApTTT2RtxMID72cC/TxfONGnGLTE0icOoesdr/yE6AVuQ0Yy4Km5M9OXU7z+2AcNg+5wLNcf
Ntk5m9cfJojUZEvArRZHvcFkbrlu5eJoy2a0NZDVGdIiaH//XrqaOJYeMnq/UgqVJkYfqPVIrYzgCb0Qk6ozDt16Ay+m5y45KxNenQAHwL1mfT29xUgwPjAO
0lqQGgcdQU563t9AT1xfaaHjDsrviqwk46k1IDglUdejjUBTuC/tH5mydIGQ05yI7+2VG1hmv9U2+4+l2UNRxNVNnMzGY5S18fgl+MIOuXZcxPwZjyerORjo
8fgxunLB7DH9PMzDzCKYgQDzGX/aqTdpiCfw+6YALN3+8Zil+72bFADAPYG2xuML4Jcdh46dhPaEKEadQ+85Eb8evXv14ZcxqDh0yx6K/qEqeXn87N37rMBq
U615QHNq9uD5owY1I+5j5wxgWCLt5lg8b6KcvKM59WA+ALUNfQJhGI9f+M+iKIwe7exd4fqVnAHxHGoEr1ZJQ6/WaJZrgT36zFVQTm3U4DA7eAhCluB/3I9G
8++H/O6G/2B/zfgaiqIwgFFlAL5Lqyxw5h4KycDxOAivGs3DcntEmD/V3JtPDWrNdH1nCfO00TSd2AYNGNtALSDDD6LjDsQ+8wB1Orxu7uzdSA6AFrRJABtZ
S4oR/EYJh7ifSocsAYlyfXgPnrR8w4sIyWn5jtcM+XcB2Kvsjc6r97AiCc7AeobrB7PrgKeDiywbj5lzjxS9kNKwdkPsKFQwUm5eYwS2Yy+dGZTjowmrGnvq
en5jhIQZ6ZATgOGuArFocaOKLmDs/3SnD1ajRwDTALapVn+QNZoZA82Fs2z84f0hGg3PvIpgpQvDQbXRsA6a2QtnNmt40NDBsNkU3wmr328ib1YjHRU4CT5o
w1RySLTIA67vUDDRMYCXltiqR5qY4X/7++II3HXwjXDlSvrRwNW+h7q7MwA/4AodmmyBJh1/5b6Cb1RCOHEx/uHQcholX0Yu4qU79cD2TsG2mvlKOCbS0g9F
e90ZtdsraHoPqQdkkUTpDJBb7XW7124f5qtjVROUke/aZAQa2uhvbqNlA+NgM/sClij8xMRtQl/Kiq5xH+HuM4ihBM9QEqBP/zWzB2yR4k8qh3eSmO6QZaND
ZEDT1oKleN9sN0FyOtYQHmpGqAkv00VOgKo5QiOyeZ1YgCrOBIUqJZzpu4FO8r0cIcula34FDfT01yXy1BTCJC3UxFmf73IFgKRB8bU+8IpaHbvdG9n94YCF
cNDTCIEWCqwwkCJng0HRu1dky9jtjHNikVxdIPGgnkmLqwIVZSvNv5uzZRLl6sXTino6jSvrktE4T8JZA1sG2VXtpb3SIeJpCiGnQU6k53oPbJDFhuRlTZsM
Mgec6zyiAg0a5alRS4h1ng7URE468lVUMFPz1yRvsF9rJMkaR13Xha5FPc/aJ7e2qgOp9JX7DF7vHWsov9QuUErN5DrqpPUqqKSJerki+/d3bE0uCu7Y1Gpp
0x7Fw0zBwHo/D3ARzlHYGgpWTnuMQf4A6qhiNgJkvKmSXqdML6j2R844Krk5cxcLe7FwQG6UDcgbn2nmJKn/aKYVX4GQ5V+tS0DrEgwKW+GVpBnLZqFMui6V
ZSRwpZJ0Wv4VYwadKVlXMXoujMuF29Bhtfyfp0LVUu92OlRwrNR5bXLXlZQoUFAYFWOQa4Xi9NxKsvXV/V8l319XIpS+qnxdgv6LxSUjNT8ptmUrMupHI29j
K5du4C+fzpLwHNUm7bk4gYw2o+oFH3p5bYoXUbhCh5Djjt5I7r5RxDFw0SlMkV2F0QyZj946bgytJr43FUe/vOQIReTKKJ8MmFDgD/e+vCBxoyUORO6N6l5O
KpaIHU3EpTu9d9KGdg9T33Jf9D4WnCNmxt3qaDOAovy5eppZqW5ry0rEDkX1xv3yEA19jua5mK+mDdFIZXJThcL4DP3F7Q2pSupXzqu6sBdezKGth4XxaCsE
DCrr/rj5u7ds3NfGoZdxGDgMGn800Jtt/iEcXGjhggv8zXtg9rOfpQVqZXd4BFv2R/qgf7JD3lwjjenFoHIWMANRLeb7qZVp+hK6vooC8SyKGrjj7CT3Chpy
VwU9P1Ms5EatkXFvx52NoXGk68PPWSfGf7/hph9+zvcACnZLeobJBKooG9JNlQ9jr1DyKZomXVxVkhMTJm0Jll6ngOjOzSPXpUI9zlQmwe7nz7/tUvTot93x
b7uSCL/tGr/tstqF15/56QbeefSblS7+RhWLb/Av/tZGg6+1n2OziwCq/1iqnmUR7Zbg+7HZw9/ov3iJTfoUXuN2PLxlkhcLbm52jRzTU5rupyTj8qYWOVsA
VCnMuEXYCmTyXj4oWyt0qXjNPOcsCOPEm8Zqo0M4Aje7xMy99KbubllKFNOxLWD4EwrkLnHXIjNB0BdiM9Seh2YMToHz7zAyRP6dF4QRzbDG0MCYzBfNkOOe
Nopst4YORxjiLEzw8fPN5xwvUo14a/+2nDqKKNITArqkQc3xGA9JNO7nxB6Zpbwm3VOy3QA5N6tnXQytFDZBRWFLU1Se6tjASnWYweYuQe9l37QhV2za2IV6
OZ0LS99Qk7T0vIRsYhWkYrqb0yV0tArmEh4LQjoi2XJBrbWdA7GXbmTHi0ahLyBQHcsQ7Q1dCqfT1dLBLcGLFfovlV0CRuV79EB0bptVcojn4OXRwQtGICQC
PkpRww3abveDnDI8kecH1Qg/ClSP3AjpxzPfrvO8SWcm53jCD7Ub0uS3XVAkzsKb2nxkgx0nKGxDURVpUfvl3t/cKJPSzMUJptItO9HIk24D3GebcF9KliF2
cV9r0Ns1BB0BxC0YDAgY21Xm/bDOUK/eGW5dPds4AwQ9e4TVRwcDAzfpUyTKt9vEFP1QJzOGt96RaLzrQxYKPBKyLrRbwjaJqAXvTj7ffCxaCioz/x2CJdg1
dpt5YoPJbLDBqNr1jaPpvhynfGXLqRsvhn1zmayzHdnbYbfa+521nW7/4GDT3u8WLWnbvyCmcvtXlC0V9mTlu+pYLB21bKnzCrfvCINiXoJNcMGjwrBbHFac
nLPa8nAdn5uLXLTJcWHHH7CZ8tykGJpt+JU40ZmbcFX46cxmeNDApt1JPMhJzb8J+MhheBXQqc/0dKdcUE5Df7UIeIvXgYUIHilT56Do+Ceer42WMSHDU8nq
DCnHCeRJyJFpdvppyWqZnRLNjm6qk0jZWUlFb+3koDqXpI5u/kSH5+iIJlpZ/Shm+cjmIaGUh2Vd3kWX60w6cUmnibODR52B+Ml7LGJQwbSbyYvW7NhpTCSX
xwdNOkcnNug+OV1N5PRCmCug4tLGzmVr/IqydKGfL8OYQ1xXUlNnXVNjXQN/XQN/XYYHQi5tlpjKIq/6NXrEO3vNdI/ajNwzAEEefrd80Bk8OtTfT4Ah350/
6BXeAqbvogeDwmvk7XfzB12rAA0j+C6aPegPqrDYZ9d20jbkUzBRTzDjqrBg0To01GOcPV5ob6dhVUuThM7N4VO84oYmMJ34IZzPK3s3wTlMIKvs8SzO3tJj
ZXNocmVz6SMeW8GJUTk26qA3W8uBXOCBIX5eac9nsfaenqsJdbFMIlU7fTyLs7fZY2n0Uh7syYVBf2L6s1qWu85jdZeqT/CY0sZdgiqrAk+c+BNBXIA08sO5
48/5Cerwg6SEfUFTv7J/F7MwcKlvF/Enb0lPvhO47SwQ9go0G56Ll+exx+JInLSNbntoNWmXRpzgs9HtWiM8zCGtx+O0/gmWGAc9gk/Pk3JI6ARfGwdDqCrP
ei49PzxbuUJ+JeGgFZSIOoPuqCfSvY8TpdI/4hNP5I+oH2WYzJSa1XR8/MoDJqI5GaFlmZ+tTgjXx2zTYcaTnLQFMLRjiJNU0aVBrQKQlQHF0zqgLgGR6qsD
6SmQeix9AlnX4xgwQD2GIQFc12MYMUAVBpI5JokkczUIE8SrK2ZSoP7MCK/MZZdZ2SLDFbkwGeioSRBeCS+Y+qsZmvcJHpJ9O8INfrDJeErcdRaxZPcivJTt
pBpxCkZ5Zl7L3sTnfqXKxKeBhFmsfPMK3BOJSdOWWR216w6z1IxT9qT/cpW0zUjrFCtnoFJ/Q3NxoTmuqI8hU+RxXpFX9GxgZP8y2IamLwpNd8r4hkb2L1cp
jfTAYGnZOMhpoamDmgFOc0apcogjI/s3b7zASJlxSn0wiAZLoWpYddBZl8HSxxTKCyRUN1eubV1mIghT+rsEZG9dog4JSQ+pI4r/4T4BmAf0SDt9OcxgJmV2
IOt1O4cV9VBlF1vCtTvVs6pqoHd/hhsYL58Wm+p0ZcVuXUVeDeP3YISi1HIvlfMKBOhBsy9anJedTjpp1zgn8/XmXoQfgfCx1ZynL/G4ydI8k2K2JFyR/Fdy
4Ud8LSaRI56/+GA/ffP62WFVVWmmDebWKKssC0iu5KxW7kgR+F4GrSmmHHQqNtPLBMPl5pkfTmBNJPUxTaOZ6no1TIfmzszaDEQ0mHU3A5GwzHqbgbqs5zYD
9VjlbAZiXTncDMRqa6Qbi2drDxawYAvUIouWl2NxQh9covv38QQE7OMJ+KUfT9BX+GiKF7iawsUgOB542j/FxodBnST9YpM+EHLwkzjap+PPC+nzs5j29F6j
pwGOFn+ggfbILCn61EtGwqsRZrMr9Z2xuGMN67Sk7stqOHtVilJzbrNqnQpdqYMe6F5yBajmMksRysOSdi3D6s+VWj2u6uyoqq8ZpBR0+aqqsxqsVQGb62zJ
/5dt6IJ2tL+WByRhDU8nJ2P87JW+6AHJqP6sSa74SzJhKa1vlcTBShV7yU2xyMxag6yMRsxllrRlZQHjIvy3vX7O/410PecnSkXSdOa+WVaNHFptDahMeC5W
msFqZxMaZ3GKhOQnlfVyffWno/XCDxX1RqoLvVGZEFw4yjWQqVwLpJxd/rqaB1y/mkJtyZz0TFNG5m5a1q2sS3gZyNLZLldehFr+e1BHe0s2UT3vuZj+VHGG
S6U6TruQiVe3UxTKlDRchP/i0q6mmCnX7eiz5jnZaKs/kC5CjKxwSXWiUietSd9boRLFFZ8XqxAaha967YOBtnSkr7vwPUYz2TjQ57L5L7Md1s3pLCRN/ljk
pmKONbwqp6H3B2VPXcU0KucrLfVrp2xaKh/0iSt9o3Y5gFJrAiQyrNO1aqZWtxh+0dWiwqA9Wd1Ks5AFOHRjUo11pUdGKqFz81ePATFtc/O4DKTD53tQD4tB
hk2ALK3yRRlwVYGRRb8g2xh0KIo2HslnY1EQ8Grhi0tKpU46ettIRqWZj7WA1e381GNaldD62jkN3eF2UQXJ41KUD4M8mwAVb+Ia3pQx8snUlDW/uFGLfG1k
jBvFpviwlFH82MWT96xh+NtDGLSMDNFWQGkR0jbyrnrdHJb/WuX1QRXkoFc5NdIYJ8Kk00LrDbkHaF4rKKgFSFM0gy3AqudKCibZQb/LHRqqDnUO/yzphnck
na5nY51wFWKohYzVY7UYZoBq3LEceFm8UqdQyVkFS7tdQy6+K7QcF0qQQU2xNKxl69LtKdTlcXAZ/lsw2XqxxNyrdnbAC6fiylJolGWxuhgFtFdfjN2y6ovR
tWvXFw+QlvXFePIl7VrmAPJuCopHey73WduHekknV1KqaNVW7G6u2Kut2N9ccVBbcbi54qi24sHGilYtcazNxLFqiWNtJo5VSxyrjjjSkW9zKG5nD2M3P/38
5s0v46rgDbvz7ZzPDPrlIA37/HL08p0Wpel00xI6+mP/630Wvc7HI7DvJ7RYUQHuOJGbDARyguuIj+xkYdM6zrHWYrvQ4uNSg0wR1ZxVbg4BTtD5xubm+cYe
a22Bp1ts7NbhsTNYO0TpH2njrMbWl9hWm7GtdGz9/EDe50YSF0dy9K7cOG23dtRA4uqmAegkdSaw6fNODSZLDWIzplWGycoP4eidHMPEieiouWjrm04KTxoK
OFGGV9tDyQMNNaC9zqACTqOBNGYftbUDbpChwwozrjDyXH1Lr79nVWPoqhHn52q3nzNgWsEgtT9ZRI/uP0sXb69G4EQ7ZxhnAT+ZLsBIV4cYOZ6H/owv9DBr
CMRi3O3XUrAvATTqaV1EXXqovzjQ9gAW/L0AbykCxsUoGH3qDEzwJPEjT1wfm/GI/len//G/z4wXkd0Y+Mvqyb8D/puW/uUN9eXfYXVDGXF0ndMd6PyOAmI5
rf3n5P51R4flEolZc3O5GIpQHkjcQFWJOYwtLSEvj5AyUKddVR/FhoStXB9DEdQ0A3WycATp2lRn/PT+f71+UnaLWFLh3+4IUGsFLKkDbeG9QdC+ydh/tIxZ
BRmz7ihj3YKMdXMyZn2TsW8yRmFWXcZ6d5SxfkHG+jkZ636TsW8yRutvXcYGd5SxYUHGhjkZ632TsW8y1hkVZGx0Rxk7KMjYQU7G+t9k7JuMWQWf37qjz28V
fH4r7/MPvsnYNxmzCj6/dUef3yr4/Fbe5x9+k7FvMmYVfH7rjj6/VfD5rX4W30apqoocbty4p0c8QnHbnv0mwNzWMT1mhyRqN43r4bQDPt36Uyr4p7wbLs8e
aRt86azD+H9KLQzpj9OAYrr1u3ASN/JACn93ZSqANNdCGmLEGzZCcZJ9RwACGn80y6c4VcsffrHfH7959+xpeU+Svta4dQdUA0uf0p3QSmzyyaqCUbv1GOMY
VbVV2GPnGpsA674/yaLel5baf5CwH3F6ylBep3NzWBV/K9GtHt2e1e6NJE6LAy46TuuLcKK0S5w9XmDrOLtfhHPQ6fUkTrmg0nH2vgjnqHNgSZwjdqB1nP0v
wtlpW702I2Xfy8oxafBlSC1rJLnExtbKcWn4ZUh73a5kE2tXi9iUQ1OnFn8sT9YXR8fPssZTDfGiKt0FfymZ7kNsyHlhimf6IeLs20y6oxvv3tdSWqTfpsoz
J5gMBcDM/4+0R9FaS7Z/lma1e2PoukSzoalZV9v4j58/Hh0ddR8fArcsE3gXhWvdTOsmRHtNO8vd5yO55zzzLgvmmVVX/1BrlV+xXa7pUuerdqlT7lIn7VL3
r1HK+flxBy6xiq6hi/VV6WKV6WLdyqruV+1St9yl7l1YtbXB+3J+kfmrIU7vqxKnVyZO71Z+9b9ql/rlLvXvxK9tnYkv5xe5FjXEGXxV4gzKxBncyq/hV+3S
sNyl4Z34ta2j9uX8Irethjijr0qcUZk4o1v5dfBVu3RQ7tLBnfi1rRP85fxil7iaOtZX9TWssq9h3eprWF/V17DKvoZ1J19j+xXGn+AYrTdqyPNVXQ6r7HJY
t7oc1ld1Oayyy2HdzeXYevn2JzhGi7ka8nxVp8MqOx3WrU6H9VWdDqvsdFh3czo2rI01HtUfN6QbNOSylzLE0RXcDQp9GeLtiPOENgXeLUKX2HK0LMJPYjlF
ZooHOshrYoO/e8E7LacekAp+GFyDrm+Y+s5iKTPchZFMajcNlx5eZJQmqQCENTl89OV3ax5GLe13+doHdSmKOkX6tvaQcHrtiaqU3XT+oyqt/E48+0gsvXhF
Iih/KJbdyJIHKXZF3ruSoezqXcHStCtvXz/713F5IZ/d+MJN9ssr9AxEPgyq4gHp0p27Mqr7wuMiF1K4LRDQaW8VWtA+/ilG9odVEQJnEsti3rNOOzv3ef8B
84Ryv+dqVxrdP/x6sb2ey/8O6foHDQ/9c7ANstFfiKv3F+Ky/kJcnS/F5c3WEhWHINAJLuHiSjllyUEO+qgHs6jc9lmGNqXcCwZZyjV0J3fWD+9+yBbSb0lZ
yqJouiweFqRjpBnguGJfh5f4w1wjvK/k0Q7XPJswfGIwf4lI9nmTdgdAehFJVtrKVBNi500znDad3NTKf1WW3fwkVUrFB1IZjHzobANEU7/6S9RuO4Us7/Pw
J8Pq3oduO7N36rz9iE7a8xcNkdUpcFYqS7qaSt5O0tbv2qCCGj2Z6WyZKLl0PY1GuRREu1pLvz5Lu+WkrW5xqNK3Gp6s1faWgKzIaz7+VHSW0L16Yg/qiC09
C0ltTdqRbuNSH9PbxqQhGxR2497mduPQXo7TG/gPt7lkcxHOctkUy2XyvkyrO+m5YBbMUbcz7A376jZNvDZzA+b6izRlOV6cCQuXgdiDfw8E/FyuJpwqS/xy
/C9OpEZZyPAuKteGX/cau4wPb97cbR7ugB+zTzdPyysv+SpLvka5TU4QJxbmxJxx4Rt2SqPpyIs5v48Z26m8D/NU/Nf/+b+cKHqmZbbG/QW+Y1omKZZ3Nu+Z
O/n+2+9fDfsbB5HeIYoj2dvPbg39Hj/QD32+e5JuC1X3hModkjTbI+WT9zjJp7wZlEfD+OpuCWViYfLz4u2g6IPSlV+YKBdz44LLlx/Wr0f/fGZ1thhd4aJU
YheT+Fh+zJ2md+drM33K3du68mJXJd9l/7iV+r5pyo4d2aPHP7958tMYs1JgzjvcDEex6rT7KFed9sgYoWDNMVdsEtj48bF9duEMc3cIN2LMhWhPnaUz9ZJr
wkYXmb9ZYpsPVpi47POOzJAtXi2cV0RjfMe+9L54c4X0BXpzTolTvkb7VOZ2ZUpfOr43o6GCRPoh3W/Cd97MQkzIS8hsdU0st2Eodz1tQvIZ5FFyt3R9czDz
s8tfqVHptyuWFJtIR3GsC8uME11jClDZN34zlneEa/WAULBMQPq3gMTQZuRhok4RIDXk5+CcTF7LIqsyiZvMss4BsQyWWcSyOmJzFh576S3dio5IAR/TFbJu
EOMUauF3U3iD1tuf5GyiL5ymzioGeY/DeQI+AAnc0T/lQElYFgunlzWxl2+iM65I5mrI+cn5i+kCWHltnkxP8uT4yNSyosgLX7Vx3KgpgtftSt3D3BJLfxXT
deT+Jd7FRVyOeeKQyMm87+IfsgRljik7sIiygwOmLE5nSd30sndFXZob03OUHJgGozKLlQZhQupURgoD3TgRsaahGAUS006Lx6BjQp2oOeydwuXI3FZJ9VUR
ktGmnX7Kivr9K5n3nRBNeO+ZbiyhNTnqQspmL/sK+pzAx5zshrPpYIrKjFFkuyyiqwWWH7P+glJd+mWKpnlAMCnntskFDvN1gXioZOeI5FGaduD+fZlWIbgc
j2HK2WHc2FWpw99gLvDdJubxCEDy6F5+UUw0CZpZ3lRvoqxJvdBQxiuXCyqtA+ypqaNZBqxaag4zkqExAONqz1cBaYFG4Rb43eqaeLvN7ZXtCOBqMMj0Xdtg
YUjCw3w+MGAZA4yGv73NnNabo9wS27dnE3x971Hj3QUbgNchS9VbHb4UQF5MDb7rbqU0zHM3VyM6q7O5ezp8uX9udmd96arG3RN2LD4K0CJxGLXQapNiRxMl
ZAYK0ZB5M5q7RhnHlskzcnWahT6+x7w4Jbuk/5caWCBuIXtHlRkGmlV0VRnbeUUZyvmYZkVFIQvAOBV5qa16rK36vdu0VR4Pi+U4k2ijHpJsspLVCjjNrGpC
WEGhvFLPC1kB700z+30DPmRc4khRb35megwHOJv3rBHM6tHt+jtTHwtHy460SQlzXhNWwgxe0OvkjoIFn1Juulo8R8fHr+13b359vwFV3r7eqZuEHqZRL49+
LzfPC7NcR55nH7R0Fnkza1Z6zRQsva7tl54GZsPINZcFe0ZZtnI4U4Qv3h4N7Sf/OHr5GinpxPbMBQ+jbBrffGrsgr4TDx8JyygX9bioVyiy8aU+kW+0jmp6
rVWl00CXBUnkTBPOxUHn4lDqgZeYfgMzb8A4bbr5NX1BZFa/UCWo5wCveAT3Rv1myqtf2UqIRU9rQBEyB4qiRS8o20erOE9RqYNUuIZIe2hIETBIURlC9ceQ
MmDokm/oHDQKglxInvT/ErXS/Fwqo42i4N7/KAWN3NzdqbRqmXRmNlYm22nJI5TKF89UjLS3uzoimB66AiU9ezBEu9PtbGV39FHlS/Qh5ksK463MglllOTI3
P+cB0DvM19ToZLlF89aJ1wJsviucp5lymEr11H5bfeV0R05hoBVcD1bFuITrgRHvHGwmI6y256bUtayHYE2ZLrSOo5XLgSy11lo41xjJ8p2pW5OUhlaeKKDp
1SAZPrkIrsxbky5J32M2I15ixTIwgRcA4yps6vh+eq0IrkzngahMLHYfx0VRGVzj6amsaMA5Ic8nBlSBDT5srIbNaVgcPUFPfB6ufDxRHMcqJ8ws8mCB+H2c
Ycqyb0HXpyvfScLIBE64lJUU68gAzDkYF5w2l/TJAxZM3GB6vnCiTxk2ursQ8yapuCTg8B0QiPM0bRZegrtaLDlkGa6S2Ju5Kr4ErD/N026bdGcZKWWAi5E9
KhEVzTyYSTKSMCOWjT9YbP9Q4qs308zILsrSlgtDYoxIRquuHIphoAc0caaf6CS3jLHOnZWfZLgw4WuLFIB4+5MBWD0w9BSXdQKKVkr2XTkwfbmapImmVMpi
xBGSzohDJL0Do3/L/GpWzColVFWpnZ4/f41sDpL0/lVXJFdhmqloGVLC2fyUioXMwpSdZVezEYkX+97UjQ/T6BLm7MCQKkc2cO5l6ORReVTZHPaVl2tTIOR0
Pg9s3IJUYvS3E0xdftWY+t5yeT0eJ2EI66Tg2naisxV9vdP8mBO4LXInE801vYzqdizuP8mnU5aJP8biyQeOr+P3TYVizmBSB0IJPzYUbq693lB5fUvd6w11
r2+py7fqYw7QcSG5slf5FnN56O+KKTdf+JxgU5/NM3eyOrNBkbhRcq9RUpi6t58HzTqH/jfYpcRb+q4dzhvgjDTr63mVdbrWhiqUYPqRaJeWHuDVkwTlDTsq
p8IbpaoKryl9oo0rw8YffyjyjMfPgjNQRo3dQnTR92WAGFNBczY1VnZp5sPm3wsN6KQsdL6BCYvPLgxKzHwW89+V/L2KmxgTVKm9dBE3WJiNTGybVZjXEtNa
Yr6Wv68Z85pQrFOc1/T7ehPGc0bg8Z+A0GRCYEiZNEgKiwiwBuWeKSRTJB3AhECH6gd8tNOnqX0ZejOjCjy+E/jqbthXd8O+vhv29d2wX98N+/XdsJ/fCdq7
E3SwDfTHw7z2N9nHKeTG/WFeaKAxdT0f5O0yJ4KDHmi5tCSgEwP0rtMs1qdcqp2KknbVUFh2K7Lb6y4N58AYjsnSKhtOAcj03rzAXSfiE29d0sLJBRPuqt0F
QFLENngsFq4Tr9CTzjlKCv9CpQPDfaU5HvVLUyRchVFyriJ8vJveb/cMq49eDGgukbgx+J+aA6MUbhbkzW2GDnrtprkKriLw95piT/RsvB/+wUPRG4kfBB46
Viu+1AX62wk2Iv2CGid0mazB/3SXsQ0jtMEh+h34qdbx+bTBqoOFPQUToXH9V0ypfJfcinpOcV1/cZu2e1FuljwwIMxu8SP73aZJi0W0up3BXbFpp0V1RL0y
ng2U2M2lVttt3q1y4ejUXavnD+vu1tEzz606enz/+fuMCltWucmqaGytGMO92kF8/1v7++Zdq0Tfb0UpH+ZwDCsn8CkasLwbj8EdcuKp5+WS1wt9BgltBnUs
W4V9bNpIt+kYhAtIbMxBbCc9G6NENp0xsHtte5o4cT6cCSrm7ZUbWGa/1Tb7j3H7HZaSD61ebywu9sN9dP1BheC6yV0vw9jlFZU1anUt3K2Oa48RRasggIV7
8QRR+loeHnLd0bw7PzBNpzsazQ+mtYeHsoqlc0NZEaq5bpsDSvhnIA93yA0IpBb4mfPYTeIGrJPa7CfrjrS+7G2sMH0y/NN8pGhGezsNxNMGZRdMUONZqPcQ
TcVrcGh5036vGLfgDM7gRGY+vHT1azqlhzVkNfHwoaxjxk6yihyMD6BH3bA0gb9/XwFVuOg5sFqvHM+L7fGRoWNQoNkxpPIZh4owz9JJzk1x+uZTY+6Al908
pQUpowvCaOH4mUUDCB8X+mPw/qfhzMV0g0Hr7Uhc0cl3cE85Q7Z+UIRIiXaZ4i8RntjyYsxVmNpHXtguVjFmsneXZETxzlrMGuG22NnIPiXf2dtulVtk6cVI
6rLyAhbGcz+NFsh3wNv7L5arV456obOf39StOTetN+vWmpvWmWmj+mKRzkqUloveXNz7ZFaG4JoC1k/3KqR8YUqBVSIOqweTha1ZkXQ+FZR8DnlsWar2e437
C/MKOrdc/UpyMR6jpL0PHfFZmKa4uQNaXJnQtMbpO8MFzUKGR9MFo9o2vBXjYQGlXLtRYhBevcFjE9uo0khtnSxf3CQuDKlBXBpuaq7Miju2CTgxLKrvw9lH
r1+/+fD6ybOnY95kQw9mPH4D00rtYGZvxuPAvUq3EqvRmLjitkOAxsW51je3Int9bg9oPg94+wfVz2+7Y7ndInPaty5Gv+0av+0yEXDzhZ8oqz2RI9v+CT/h
c1CxO6NIZ5Tdixv18MncIgQ2zce6SBTNi9icwRwFHzsToyIMz+kcXFwFl+FS8rEZkxSeQvyrIuRViHJVBLZKsSztRQUBVXF6ngTEL4lWrjRCtCx6TGpghsFP
sFQinItTDOKeqoAn/uBEgpQlLqHi9ikBLk4NOhV3GpwyMvrYCtTi6fVJwPtmH0FW1/jDAxGCVRImLmybeFdWew/LDTH++L+PT00yb6334ZGyToxQJX+fyF7S
6q/BIDJBLh0ZhjmBJ1ypfWJC85BSyaMNjCnczdFny+oZ3R6efRz1jEE7DT+j8QAT6VcdzlD/fTK1T7foeJkhlnN7HdAfDPfAHxX/aQTgspAm4H2t0vGbdOu6
/PqmeVg+p0H77yiTIE7gVVyM+AReDSSusdHhhG4BA6h7t0HKlNUPs1FU9Hh57sTuvUZin60M8f7YfvHBAJK1qgZGUxVZ3Cg3nEaZ1WCg2ZL9r2UEz/B6Pm0o
u+871+DfXtmydaO67zJuUYtGpmdVW5a1OFg86vGw3GwsVxN/EwwOZkMvbukoiuZtHGxt5kVrAy9afwEvtqX3HfnSuoUvrT/JFxyTuJ0roo4rlfMGfLf81Plc
DSfXpC8D1OJ0xp8V+tQJghDe+J5DH2ngHicemKZ1SgsWohvR0RF1cGuVYg7Cq1Y4iUNQJS5pXLV+Qa9/Snth8UaE0IkZ5bydJvR9xL5MSzqN6HiR2h44AzlE
A/TPd0evzHqEOZ2HXTm8BVbXeqtlDfRNLSfubc2K29RhjftSM6k2lpcn1kbw9i3lhUm1EVZOrNtgcs7NLRPsNjhWfhuhtuh87XT7z+bdXXjzH87HW8f6Z7hY
42XV+zeun5CD8+zn41oPB+Y/L9/VqTJt+f75L1He6rCDOnJwFYWJq13nRzGh/FGDjfgm16T1DkFtCzoKARU8p6C5XcxBHtcoWXnud/PQN7jQVdcgbHD1tnD3
cmqAnfxh1xiBj9/ty89wtnbxN/l8GCTcaLG3c8bu4FrcZuWMrWzb5t4uo/Dff85Jkd89DYwhkHzYsYxO9bJqR/ztZDo/a2D0vflxJ7dhRtRAlyJewTJuPP5c
EWQx0lT1KHMg/Yb6jBG/YrRxgvC2Gh/ktJfOdazCKzrqjGh/rpFycE7jR9oDI43zCNUR9Gzc8Xiyms+xQ0/dy/d4xqgMsUDSjcdpZA4/6mRq9ygu3xl2B0bn
tt1H2hiqClrRGdeDdrtpiNfgAH7RViNuYQCZMAAa2xzBVl/FcCi5eruxIrB5YA+xPz17hDsFo4NB9VZRRU1Z5QtqZm22219aE9tEIm5bEybTePzq6F+GyJ70
lvM74BzoSCJnPvemHHXnz+v4Q0SOgGBgPxH/xkI0DSB2+LmYKY70E33RwknE1TnuO0mf1/fAuODx0OyD5sxpF/zZMH6fTJ9YZqim505E2+FewjtYFAG6iE/l
5tbcxwPCEVsoefksxnW4C/wV7s5/AwUiH2ncvQAA"""

WAVE22_PATCH_SHA256 = "07c60277043e66874ddfdc36a0fcfa4057bdb6f8c7a73372064bd359411d6239"
WAVE22_PATCH_GZIP_B64 = """H4sIAAAAAAACCq1XbVMjNxL+7l/R2dQWUJ4Zv4CBXerqYA+WbF24vJC9vQ+pwvJIHuuYGU0kDcZJ3X+/p6Uxxgskm6r4gwFJ3ep++ulHjdTzOaVpoT2JgbD5
QnuV+9aqQVHmrRSpVU7x+mAp7tR4nM5bp2TqlrooW2w2QttUKqeLOi2EV1klafYXOerVaklzXSqqjFQ0Gg4PDw56upbqnobxk2XyjRJDlffSNKWBVHeDui3L
Xr/f/+vCOD2ldJgMqT9KDg/o9LTX/5qiN/oEJzQeU0rBD10v9eW3H+n7n/5D0RdFX8S+en1Yfk3vrflV1fRLq5zXpubVc6Mc+YXq/I2it7TzlotaagkHlJeI
nbR3pO41rOuCclM1gCjBfc60NldJr6/uRe7TRljtVwRjaqyRbc63hUAciblXtguRvZi6XIUAdFmqQpQktQVuhDB6fTOn6evcCy2z+ykJWNPULcopmUZZuP87
p/AQuUA04ICaG6u6i/kGQXk70/VbPjudTr26971+RPEmlGJ046qjSdb4+4RKXSs63j8hZa2xb+m6UblGVFYVSBuRC1u0lao91QbELUuzVLLXx5Wka+dtl+sO
otwJ120iHFO+EHUR8BZ+67hTqEmdK5pbU60DRTy9Pvxks/0xvbajUUJrLBI6PHnw7s2WRWXusvaJxckTV+H7kZufFsx2BGoVsI+kkEyUWoQYNwiUVgm5wmZe
CgvEuc6lhiWy4tR6/cbo2mf0AQhJ6QAV1aLCyY0PmKzRwC5TqfU4cCfKVmXbdM1F61ABl6Po60DXNb9VtlYl6UoUIGIp2jpfUKFMpbxdJeQWHGBaqcrYFbZX
pvUJXV2dkbFSWRBWAnlRe+1Ckglda9B+XY6Efjim7X04QKQPruZgwEzkt1S0wkrHDrVrhEcUpSmKQL/tLlgae1saAdRAUgOQ7VI7RbOVV6mWIJbORRkQeIfd
x5bCVg4Iop51l/+w631OSHjPxqbOCAChOpBXAB3aa7ZClS+//cfH87Ob9x+vL85vrj99QH//bTRdY33JzZmAlBEZXh1l9KPKTUfTwIfrb87SO2X1XOPWEMM+
1q1pi8VDSZxnPDhr0TSxt3t9IrjOreLOQS3XDRFlADGMM/rGOJ/mCwXjQD0tihpLOo8ICue43Qg6OI3NOwXpZlagsAjdu4zOwj0x3NbGpoU2uQCec8oGGNGq
My0jv00pH3SnkxTerxDRPvzBpprhEeCjEfiO7RtddNXN0YRfibYElZcaJeM2hFSld9OMLoDWKoT1qJDAAEFXrfO0YCB+VdZE2JLud+gqCO+hYy7ct1kNvpg/
nG+9WoswK+p/VXSP/qq4abseZolDPgecT4SWeUDSomMtZxvWHjLacWTyvG1Ena9iQ6OvgScQmozGKaqN5k9CGBxVKldobZ2nsdW6FsRd6BElkKGpgz3Lw6w0
KC5gpusr0szlX1qIvUR0E+Za+Itm2qfhGeHm47aIEMyhXA6tEQ44QuyudduV8UuTXl5cXUW8S2zjZcVLIFAUZDLH85weHuC9EkUBVRgdfdab6fjggCWjUQ4h
HWb0PZMuoAOUS+U76LrnzbWaCYB2AC9XsDhCEi0TrLXUGKeDz5koAaXa0gAmvXvoNUdng3cJvRucdV/8Z8gh0GmC20uZ4OcSCtA2IejRkCrA2zLkiMIGdXIB
WxzqeIfYAvmDr0C3KigTZ1SUiCcnb24VhyHwvlNAvYRsToaDyRAJHXNVgt4EGdFzXIq+rLdysWrObNUVFu9QrFmgTSz+5HVCatMCPJWg7hGbO9XthXXkgfeE
hy2pSi/42IOXlN1w1hx4jCB2ekwvrcQ9eAf5A2MftXx0UUFDqD95HYTujMkSZZKnlrb0axLyaOFY7fBIoYYYPLYVmZ97NTPmFtWo2UXG3p6OQB05kjWmaC80
hnwcF88pKBz3d7NpUTzEmIS1C3NF1EWuLQAuRY4Glo/H5SiAA2fzQXwB3eD5gQaj6Bef7eZbORT7kzdvskwMD+V8sr+efnnM/RM3d4Pwl5/nWfd4mBxRH9/H
POwSf9zCrseZMU8thwmNT+izz2AQ3sACL1Hz4TwagjDrcWe/M9x/yTCqWqoxebOLJzcfsANcP3nOwT9xJkib66XR7qWRbdtuDhHwa/IvtJQ865iyrSK76cVB
7plLumHuaXC/c0lMUvkmK1S8qNn4wvdJPHHKy4SHlt5ffrw5/+5fFyfPmd60DfSpYctJQscnf0hYvJkZtG/28l5HyOP90dEBSJLl8nguxuIPCdlZv0jAbp8J
N56MjpND6oefEFWsYTcOFPRbbwtN/Gt18+ns3xcg7fUV4gliqtzuzm87e1lu2trv7iVfaPK/jcnGYq+rbWBvmFe+2v3qcxcsP1Bkt/vqZZ692nvO1YueNkf5
8+oF3v1cv8y7VxsXfDV9eRY7Pw93/rSJfd7kcwv+X87t7mV4nncxFb59q92NcLnWbP1/vjqS2n8QAAA="""
WAVE23_PATCH_SHA256 = "90927231489e073259796131f3d04971f5281a2596fc843a116ad6c51fca2ad6"
WAVE23_PATCH_GZIP_B64 = """H4sIAAAAAAACCt19aXPbuLLod/8KxKeSkSKK1mZJpic542xzpjJZTuy5c15l8hhKhGyeSKTCxcskqbo/4v7C+0tud2MhQFKSPTO3
XtXzh4QCgUaj0egFaDTDaLFg3e55lLPgIEjnF1HO53mR8oPz5bwIg27KM47lB1fBJR8Mu5/6g2n3nK9W3ZBn0XncPQ9y7q5CNvtT
zfdifsUW0ZKzVRJy1u/1xqPRXhSH/Jr1xJ/rHgbDwaQ32+t2u+wg5JcHcbFc7nU6nT/b+Q8/sG7P6bFO3zmcsB9+2Ov8jQkY7Fdo
ygZD1mUvoTV70s3y4JyzH5+/esUEFIZQ9jrQ5m/sRZr8zmP2ueBZHiUxlj5LeMbO06RYR/E5WyRFylKeB1HMQ/ZyONAQAfk0wEYZ
i+I8YUnMRZfZRZDysLviqyS92etkxZqnoskKOk6jYLm8YdFqnSaAaX7BGTyFxRxBwSMHoi7ZOsgvACI749kyYGejvyNmNLT+mK14
kAHVQmqscVN4reDnMknWLMgZjOL8AnobjO8zfr1OMqiHtWBke50ghh8XyRUBgsofF4vYD5Or+CN0f5OxQ3fE8mgF1AgygBrfsFmQ
phFP4XWUYuleh9ogPf1i/dEVxO9PvsuQGt3ZMpl/AgTPoyzHVjA2ns8v2DxZLmHiod8kBuSG7rA7dKf39zozPg+KDEnLPvF1TuND
QGJgFl1ZNr/gYbHkrp7xHOYwE23gn3SvkwbxJ+hkxecXQRxlK49lax4Lsg2nzlF/0J3d5Jy9evXP7nwZZFmli1kRnvOcZQlxwV4H
Z5/GBGNPkfJZFPI4ZzO+SOD3PFmti5xYAoqAxplrcxkOLlgC5sma+O8M8JjDLEQhEBBGzNeZPaWCnCXLXfHo/CIH3oEfDns9Hh28
RoZbBkU8vwBO40hW6N5hSZEDLsDGUeiwj6tpPP3UH39k0DEQJcwcZFl+DpMSzOfFqlgGohm87uIoF8PBXifkn4sgzqNMDAnZBcrZ
i1cnLElDnrrsp5whac95JmZSs57kMQ9H2ZVUZScH19k8WPKDJwdX9ADVkhTHBavtKmOLNFmxI2cCI4LlJCaI4QRlxwilWGMxrcc5
LDo+L/IIyFOZFCJUqKYE16RkWoKB1CUIRgsEVqygTRQL5kkjwB3mKcjmwC0oBGi4x0gBBGIABWDLJeKODUlmqJ7zC1hsauTZMslF
Z7Dk06s0ynMeu3UWiICO67xLiNDSZR9//PnpL89OfBRfPkqXR31YZs8Byg0rAO/1OklhIe115sE6mEXLKL9xgB9uYP4Ba+x/rfhJ
IFllqU88jfnSZU9gzRiCCKRDuspkS7nCemxRoABBPucxcAWslFfADAGORrOI7mQwAHk7jzJ8kSd7nSXHUgHi9Cr68edfWAi8NVvy
UK2THwEm8aagOJb2XfaO4xTlKWBG0E//cdIFAkSLSC2RoaaX6hkGB6IH8QnWa+BMmLO9DmMAep7yFaAL86vkBsja+QWgMHDZWxQC
2MdFhLwZAZeC3AxWa2BWkDcKJAjo8Yh9FBrnI1tGszSACSEB5LJ/JFlOnQmcixSACNm0KrKcrSWfmRJIzJhDvCnFplxCDNeTQ+Dy
q0TxXYYrlRmaBdFCIYCLMQSBdnIgVy/L4H8ezzkMcOiykyzjKyC5WAmgjVm28ieHqMRBlrKrCLjg4zq/BonfvQROOzPFEXaimZVQ
AkICOnJgF0jP33maCOo78nmN+gypiYoEIBilyyQIUUiOELFPhBNyOwtTWNm4uFKWgHxaB/H8BtXZDJl0yc+BoELm8cxjg8MxwT3s
DwSZLlIeoIijjsKbOFhFc7UShWSH7nKAEwDSuJiDOUkSQXck7OkrWIoELAXqRdgQeucB6K5znqx4DiD2OofImvSazaK8C3wCDIoS
UgpfGEJWZM1KWiw7HCF1Y6lR1joCGXg9PRq3HVMpt6DkejQdj6CY2JCB8DwXVkkX4C8JVBgF5zFwYDR32TPADRdNtEJOAukSgPJJ
OQdJHydgQJqWB2AbRvMcxjWGcRWCR3GpL3kup2UdgOQCzVuA6QN8JS0Z4pnckmNCEdG6nghgJHTB/oiwr+4sWMKUAuam5UM2hVr6
GWiLJw57cnAi/8GfND7q7hBtCNBsh+wK5BRIXaRHv1daRoZxhhMKlZTUzHiWkYADWMS2K1z+QoAvAZ85CKtPHNEI5rAiaFaXIFQP
eweHPUeLOBBdJDdYUIRRXkJLZkCYS2kJklKUy72kDiKz15ki95B0FbUWgDzMXdxkDUpjMQNtiIwoOPfwvsM4Dok6R/LhDAsaX3L5
jsqBHiCJ0UwP+TIPiBEUlC6CwUGJ3oXcEiTqroJrWGcg3GH1gRUHCzjTOKwSQoEk9wnyt1QBUKlY5mrdEMeh+A5ZClyQLEifd8ls
IuEKbMhnSfIJtTCCAHgnYGgi34FvkToID1hnDuMR3OeoiQHZAEs67ErEsO8FrIECdS9PgeFRl+x1kPToTRDfoFZcRNddWHbAN8Ce
oelNCXF+IOV9Jp0QH50QH50QF7hydptat/KNjvhiMeuPGn2j2/RhuECDwZh8oIODe1qphWLpo0whOYp0qDtEmildai1AnIHyE8Ji
nqQIJQb6HqiJsCSIwaslC/BLnF/SOHsdNOezPPQ89CQ87ydQ5KCVjtUrMVLPmxWLBU897wkoDjC6ntDPY7uO0Aqe9wV/+sElzDZa
Dw57Cr+/VSoLAZt53kt6OOV5pULK19CX532e+j0/SwI/T/wZkYaQI5OD/Xry7tUvbz1WZNHvnD1ih8fqzU9nz9+dli8GPf3m3fO3
z0/OjHcjAriIyZkKv3/xuEUYsAeItwNiI5fazGMv2qz7GOQCrqHvF+ORg+iC5vS8H5fP0zRJH+91rsC3EdqXvYAW8asib5nNWu16
KxCdX0QT5AUfZRKwoBgdk2/wT+DRav/9WJR9E/8hvm52A6/SJIZBlRWW6B/lQZrDOOXcel6cXLXax/X+iGZ/qrs3n1rUm8uXwRoM
yVbbDTIfJEzmA7WADA9Zn4/ZgZge9FuhuL3X+SZnACZb+FBZC2SBH0YrOU1odBo/iZ6t/+Dz74vpY4fJh7bCHUf9OYMht2BUEhB0
LUC0y/G5q2Dd+hp9Za1W5F6lYIqCHvbB3WoNjtplQRCGrQiQPpq02+w+GDSHbUS9mJqgpNOsCXtwwJ4Li6M/ZoP/250AMtcDEC0w
/eAG5EveRe8FFMoy+sRtK+Q0ORFOpGvMInpk9TG1xKAAu+GgbeKzAM3g0/j8r+x977oHyhER+LAN6dZnsMtET8askP0ltHULPcEt
c4IzASbWYzUTiCw5j3ei/nAi6NwnOqPJ1gWf6NDttWGc/cEEHpoGITGWrgi6JWmoWSKY8SWs6e8y3AGYo+0v7BVWYTRRaI1MlsVg
eNglasL8AmiC4kAUkx6wi8AA9clU8cBITpaydAEmFUxSlJGN47GI6tMgwKRYVoYAM/HvLIlbD8B8WxC1T8G6B8PRWLGwnAHWvVZZ
gn/7X778tk++5m/73m/7X779tu/8ti8HDkVfvsFPMWT1Cwcrn/c6v+0bI8VSzx1iJTlQo4SMOvF7JJvqoSvQ9qip9Nu3fcdGGcfo
0qQ1vZCoN70Sw2h6g0NqKjfG1vRaDnJHS2DMXbU1IZpe2kQxarS16FXLUSj4VimLlaqS3Cc1K3ugVavzv7QGTLUmeBW8geT6e3Dq
hFHBUbt5nlByj03xjID8dRCiQMNn1mETXN1TkBRTQ3nFM6hhyDjjlSmsoFJddygB1TYb5WD3hT42FU8WgIql0XqA9R6IKg6rgjWV
7DUAMOWkGl8TDrRnhgNXuLhLmE9DpHUs3Opvr0URkGpkFmuaKmG74SXQtNJygKU4Cw9ZqVpqNfp+bzT1Dydjof7GI2NMaCWBkQij
skxEMDb4FdlTjhi2RbQrUtLQzAVPNZlX6CE7af/dDdd5ajbL5g3NTGI1NiWz5SJPwtYVTavqTaNkVMjmuoJkEGuyF2b/PiiolpyS
DT2KKguAeW0DqhCgVZvCzVS4tqlAXViTbDeRYsEVe8a/c//zVM4L4nWNFLnGUW9CAcwLE3Mt+tBna0KjZKU66tqtuVPreufQ9CtU
+2pqQDVO9MP81SqAcarVbIvdeSky1R/yRaUESGKXXNfqXNfq4F5hrYRWlaBk5aUUb43viBi1N0opNNH0bkQhp/X/T8rYNlrL5Nhy
ySuqtSyetEXCRs8Khd5FAnrbYMhLPr/3vgd8fFyRqB+aGpqztrMlIRPmyQUtkwe13h3WPMYNzXTfDmsePGJq2yWApdWjYYXjjp6p
pdzfo3Xrgd2R+VptNra+tlA3tL+ywEXFG4H+brN7IBTKnxWnwU/SVrfvMPQdImQBsJvbesLB8XqaFDGgo7cxoxhPAmkvOYovk7k6
AxSnmcEsS5YFTMHZiM3FuVOYRovccL5aSDTTTqQZlLZeWzhl6F+BY9LgV8sdB3NBmpZj55HYeZDi+OvXDTyr58XwMRoab2Lpv7L5
n0Jf7iKg0F+AJUvtLJ4Te7Ryexhoi7aHpGFb7hmU+w22j6TNXEMk1J2Fmo9QcQ0sv8428k3cnBo9PU3YjTUNT7Cytr5nPaNeo0Pw
zfDK8TC3trF0C+s7WrB79g5dq22Sbw0+Zb6M77X23xthHh9YnMjDH34Zzfkx7i9eoPep9vT32zaDFGmM8wOr1553UlYYCfKINgY9
b50mM0OwAn7EEuDJLRI3A30V/DtJHWaXRXGStoFkrYnDDtuVpYVdw6hb+2q3VW93i9M0WJtglvJ038XIkEYMpcYEJLUX5Xl4HtZ6
YPErUtPSrsQAPKZz09ZGxDLoQh0dv/vp2eDZo776/eT07OTH5+Vv42h5C756+/eRQt2Qmeb+tKroi2qWyAbDL7GoJmHK07Ai1iyz
by1YcUDn40nfI5onS15f+/K9ONT31zz1s1WrPDmAdiA7tyACFbri2NA4b/xciEP27Tgd9gd/CCdotx0nqHAnnIBRDDJ9z/ooKQ0k
sWQHF5fGwgVIQTo+NY9GM3k22sAlJp+guES+fl92JncUHggJ/kAykMP2jWPPfYcd+ZPB1GHTI5iuwQhcEueWIPB8dF82HPlT3Dev
tget/aISLgIqdE4nVGWs0oICP8pzdfcWCIhDWGD+cQ9w6OMI4BH+mWgETNMM/BAhn4FEkljStkEXpfVVlH2V78pNHUVtLT5L1LQg
Vez1geGGnECRduTOl/4G0xx36nAXTZ6aI/M0sq7aXFP1gKW21UOQ8ujdF0fvPrnn8Lpnb8aVTOsY7Cre32bMQnngeMVRsN5uBJpm
6kfK1zzIM2vDkUgPJe+/fPsAVXBe7K1EG09xWGIU0BGD8VtaEJaat6YXt6OFNeF5uNHaVjvLnvc97mv7jx9DrX8noHb3nf22SSPF
M45NF1QOJT/tWN3yRLAMVRBngwFuQmwQ/EK9oknQdFCapfMDuQxkEcz/5NBd59flGeaWSvJEdNjj46MRmLH9wexoMhyr81I8GL1N
X/aZ6baKeGQ6GII7N2Ed+r8/ZFD26tWJ/2vf26OHZ29eP4dHScTjPSBF172MsgiDZlyMd7lhzatpHa1xCwAETfev+yN4m1evp897
zcghGXRmSjGA8jMdtonIkTTKL1Yc92iFadjYg8teYNRGGQb3EqMCAZKUn9QPOT1U/UDEP3IrSlQfu1NoEgJRcUkihJDgnRLCHhvh
edMJMMRkgJuQIkYQtVmHPWHj/mgED1e6FN2i4VTHCLp/PelvN+9ig0M6nUEarJhbjEds7dOeRlM57lo0lNP+huBREIb9I2TSMTDp
WDGpwZtuys+ZOwNldj/1Z0EYggGLT1n5GOP2onhEkS3f4+OxBQP6vp+CeMbqUXjtiF+fZ0HG5XOGz82NPq/zVNUqH5PF4nivK9Wu
jAn26AjydbE6RapljwZlPDDR0WEXfElBmCpgOMPIW2ikIdnRuYskyUkxsK6KyklV7CU3bKYcebGLndxkGpQM67GPRYXc+A5YqVsb
6ls/cMT/s2PzNR4e3l+89a8zq3gGs3b/AipXimnC3voxv86tF2tcsffXb30YGi9PeOXq9szgVysYnUiXsSWaNkA6K8oNw3QSEYdR
xeClL2mOj3PczThuqhQYta4z48fMelZREw0AFOdh+/J5ZjxKjuzUaPESzLrYZjv5BhqhEFAElIMG8wmD+4Hw7myKjljwHiXJh2NJ
S5hHEdlF58XXbDQFqbKG9YJxaBHwYSO4kYJ2nb0HoSOgSXA49+WZjDwq2oHU7D0KMo0URuTIqO4/htasghYCxIgACVTi1NlBqP5g
MJ0SEAAxAiyaiNUIxaRPvzcoRyahNNOos4NGg9HhZGzi00yn7RjNGjBqJg8TTLYMhUAmeQxrve+w9yTGPxw3VxiICtkcKpDgPuxN
HVBXg8N+3+lPG+T2Krl0C1ocg6EjkJSwYcbdTMpzvVSwkim2tWB4i1dlUnl/hNZft0fR+9JlxBhzlKAoGLEYbOM1wzdQSFG9GlJ5
XSS8xN1MEbBKAgV1iYj5C+ZpkmVC/oRRhjZqLqLaHawea2hQX0T6QmkmoycZqo+sDIznDMWfiEajmHkpoEramJJnoE5rzdeGMBr2
G96X8kkox8YqRp1sQyUtv4bDRiz061FTD6XaNWScjUNNM3f2mJb96bpbxBGGXLBVF9UzOy8CcCg8Or5lK4ypzBha5tMV+Pe4yeju
ySP/fO0uc9HRGvh4SlgeN70doL8qXwsmHhw5Y+Li0cg57Fe4uMbHuC99zJr+YBCfZqz1UkZWk7nfBghKCaOVpcwDDCVcJucF95gw
CkSbnrhbJW5JSXOhtA/ITOtqfAxN3Tsuy0sVDeULGYBpVtANZ3bDQivxspxoB7axoJ3Q144kw/20P5LVflDv2CwNiHxvf3r73H9L
VFR17sHUDKsV/nUqX4OwOV8ms2BpDew9ip0eyJuu1cYzQPaqIJ/UIJokIYiDCsQn3kYkZqKJsP0+GGORannLiA1oBmnfa/OxgoTg
uK5wzF7+/ObNW89gYD0JRzb9maT/kUbk13fgqx9vYjwSA566oYDB2OLaA11OkLGzUSb58QpsTBZlWcFDDY2OfVS8OQvOE8eUdcC2
l1FSSADf4U26/MJhVxcRsHIkpO3VRbIszdw1SddkwVTssmsweoVpaC+35Jksl8qQpus9Ss8P0mwNJG1Vi2aOEW+f1MAhw7xHYYvg
lLlrNPA29C/E74eK6dzELRLQybv6UIBX3mspjbCUYW21kyiA90kHqrRiN4kaIUk8pfPERCHZlc6kKZZz/qnTZ0+ev3jz7rkGh/VX
dEVzoSo56K1EeaYj5pd40Y4VIPRS0+GOYpBduKm7gEcNcJEmYs6j3EXlHkbzgC5IgnBPxAWCJYbrU2fUlToeoe04MBiz3PKVzKuC
Jv+QmdHgvYnHUe9ofNxQMau6eWBzll5KSdzBUF6BFWuKLi8Yros2Hch1QfNBokaQsmKmNSP5JGJFi9WttGcU1+voxyYtLLU8AiAx
gmcUxDOe7NWSJcLpcLa5SD+oWpp1S5iCfzsNC9Wog6u10yhcByNDwHe2LujBqDYgWtSdhkVt4lfrWqiCkaEHOtuWfkO3T7wdw9Gq
orNTRgxGBvmqIsLo8/Xzf53VqUhKpV/RKJ2d0qRfHxPC9wzDXA6o56j/MFKw9nrgqP9G5ioynH6lqvo7zImy9i6T4sUuk+LFHzAp
Xmw3KV7c3aR48deZFC/+sEnxQpoUndtKws5dJGE514OpI//FHYja62Hfkf82NTa388QT7hg0VMtq235N4Cw5Jh+Vy1RfWA2rGzVr
p6pZt4lZ+9Vw6JSuVP3tyNwdqr22doo21Mjqm0lMO1GkmrQDrHaoW9mnaL0G9UqGF95oS4q8myy6Kd6fxTuF66wtNZMmx9M3r97+
cqYceVoW/ZKCp//n9dNyl0DJGsENY0dIHETuw6ZKE6NSpz9WOwqTwyMHll+HHgaHyhmz2FnEc6wCN43lMhwcoqxGLlygawe/iSga
U0/bCXQpEpVZSx71YuxCkrfZinMwZnBPtea5iwtk6io7dxsML7VNoW2qJY7rRhlcIdLccuqCGcA7Fnchxc1TCrHCmaa9fR15RZf1
S1x0EgZlUugTh0AdRAhz3kpdYOxXrOhqPF0DNg86pFkXIBwMP3L0ljFdp5HX6N3q3o3QEFLC32El/lVWyI/v3vzy1hfMWJU6Q0f+
2yyURo78t1ko2ZsZm4VSZUfDBGdhKpcThdU1oO+Vt5wUhwqOxB1tFiyQN/SdRivzgkwWolMupBxTDrg7ZZgQ2mrrqS6p+tY+eIMg
UuL4ZXmPsCarzDqyErNIg16uXqvkuHo7D19XSWjdTa2/k0et83C6CAaB684PB3wwne48apWtN56wyvcoqfrA9FPWEf9Bgbwl9WoV
vBK5Bb7syfCvZAaugYh6cpSgVsdE4yfGhuRMpCURTUjesI+CtB9Zq7znTkKjLZeiOoy1e+navUy8EpC+vk7i5JM8Q1VyiYI7wX3R
e05K2JVnvmUvHasXPLbZeGCjxVb10EbxqXnG20yrQc+zUnHgtjam4/jnS5lkg0amcs4kixyvc6Nve/IfSmzleYyHmaONXfQ9dYEc
gwUOYJFBD45M3yHujf9z6vdUxgO6zM7Z07MTV/DE4QS3w/E/4ol1MVN8oaPeFF/QcSUNV1xgqzDGAG8FCbkvNpXX4gIv5SOhtDao
vbpPdN4MmdjE5Arrbpw1TzIHjJgvWw/ICcPQJ/NgW86TDsnbgPbhCe7xWBk0Qr4IimV+LLaVZBASnUMmVyoSbob5M0pYerPoCvM0
BTFmQ2DBOWquHJMlxZxjHg0RtZBcZT5BDRVO4kx5IsIejhxMmMXo/l9tEnSIK0ZmSzHGHjEMoDjnub8oYrrs3WoOKKJgtAY4Yv3e
AgxV3AQF7O3bwjGFAIHrNoKDxXsXaFB934pSNoEhD9wBGFbfNFC9LDfB0xV8WugYdr8JlqiQXUXnywLBDfrb0TPr12HyMhBLnhQQ
Tx2ObsNT5jpUvOXUq0jJLbnG2QTD1/UkXzj2JNeEtJrxyi2PmrBVk9nQsyEvjVlqqGhS0avMQqX6t7Yg5ZQk5WBK1v5WWuLEnn8O
fGH0PJKh2PGl510GqZ9krX0V2PvPE2FQ7bfdKPOzZMXxIrYFSEld5A1wIHQ19uDBZsCvz376+Tkx8Ea4pei4NVQRl2zD7FiDVrK2
CtaeUegD054NwlqxQKpWvJmAKjJ6y0ANcbttLk7Ozl777978eroFFK3oMpnWXWhH4EE1j7aAr4oCi4LEg0dHaMENe4dCW29fzj57
9Jj1DW7+ZvRnCAprVe6/l5keMd1LCmpcRG6SRsdVAuNWIZuaxXUBTaqO6AS3orxLLphY/RbzrH7RQjWmyehg4s8vUIlaVXEa9FVy
g2ZlTGi3utZRckINcNM01o5kQoelFNqqcHSYOps2UKJ2ChmnwgiONXOVi/f/D+ipl+H/HoE7fwmBNaJ/mNYah3Yjbyvqf1BZxrrS
lFd5m8rFLC9s7JuA3nxqmasLl+Cw30c1MOxPbqNRS1rY5ZoUew1qrkLdkkh2ZZNitV41+ew3FVoKF2A0GpAPMBqBatsxKJHGQEhp
IVPKkJcDdpYWeFTKYyPH4NBOalTeYLgKMsx4uozmEW7aGBnJFLifoOplFBZGMjkQsDhtmInTjgykdKx4FBzF6HKBr4FhoS8lNHRr
FjGrX88pM2mgIW4GaNNIdYPKPQoarDFEEX1KPp3cNhOpkDDNqn01xEZo27WcErU3a5yt78WEPK4hiXoiyKD9QoawfxW5Ar/KnIGu
YT2VuSRK36WcNelSgoOP6cTWy2COCcnijHfVelFO5npZZAzXu/bOS3jSH0VPFzOUiESS8ER+qMpmeUp5L0W0jNzjw608kYdtucTA
Y2YQylz0myeP2HkMbIzsPB5PnNFwOzu3G5hYzyk5seKwtAzUboi4VtHWNNM3JaAyDMvKMUrR3UZSU729oXnZSA5ZQqNd0dpGr95A
fVvm+dKJ78SSytD/pF1blcuQ4L1kuLhE9DTwLAzzWBya8/wiwQyaWcJk9lLKEXeBaUxTef9GWLZZUoITGffo/oKx7zPHbHmUXg+z
M4b6us7f3uNl/6vWfBmt1zeelyeJj/mL/SA9LzD5Z9b+UF+5W6+tEzMYorOeFgX/REKHK/9z5rGnv4iLjGudBMWsIQIPN9W63gLh
ekfbm00vyhws1oV2nYXFKpV5WHRZ9R6oSixmiouQz4pzP8gynub3WraEc9j+yzJVcXlfsm5pHzcD9PnneyrvCruP8Wy9rVXlzf77
bDioVEVbWN5iQotdyjhfFpW5XZAEZjuZMQjxr/AKrrHdKQ5K5tjwQqaBqaQ4qNe/bq5505z4oDHjQUOqgypSMotqpRSEeWPuoNrG
odyDtOX7ixevZaTLRQDEJCWHWW2V+pYhOLawz1QcqrirgmJR6QncMctAxfPsWG9BlmdNKDRQK5TgjEBkCowpxSTIqDLTqty8nPQG
zmAAsn5yONnplRuyvr7ZjJhYeRd3bTq7GkgVGm6Q3zL9O+brTopYZ3xnVyBkL0qADx8ODt2j+w8fsiK+iMIQkzqqTcWBO7pPBo9O
OIzgBu7wvs4miTnCNSy+jih6k/33f/6XvLahE86L8/kuxtPp4CgZ+LYEicyhuwzQKyPgUN5Twlop2uWmKD1DQ0pdqwOqXJWvMito
HxXTbovQPcz5rCEqGvMgxWyu9o7q01/Ofj45Pf1O3d8GLpzqvM+UMz4rIX3Ud1gw7+NH6KZpts5UXDaLMnUMoMJW13R2RXHaaYDX
VnBocZkqVWj0EpZIjukoukqFX95/QZrj9BBXRcLMGYvxZl08/D59VQKrXIzBtlG24z4Me5YUsyVOiez7KimWBoKU+TlMEzpTRwQS
NhSnA3IX/Cle4ZxTwn8u+sWFhibZsHe/iXwnen4dlbRe2ihk4RhJ61Ux2C0FyLECl49TuTl0YEiHhw/xBqTOQAvcD9i+/3gK0t3z
6obAxw+Gy/BU2v6UZx3kTBRkejnQ9ywuddL+ibww7IKfIexOifElL+Gp713IdOXlvfgiI14G+oSYcC+2zB465cZuUWEe20ZStFoV
OdqvGg2dYVf4NmmZTD2jcEWd7l4dmEgTKkzmLSEVwFhitzar2HazSlySZBWzSojbYZ8iHfqTyc79ZMtoYQ1GC9thtHRvb2N0b29j
dP+gjdG9s41RWlcbLyPapgjbZIqwbaaImJjRiE52J9ORM969K6HtDtZkd7C63dFttDu6dbujSdkKDI8GgnWmw962vctvKjLoVqxc
Snzk2iWKxI/gb6AzVn7nROlsFI6XoJVgrcqPKagMH3kg0q+W8LJghfJC9uQIoY5luKOlM4xTiXU2fCw/iLK8KWFhbABPMd2y0GZi
dyRKpdaVqjDQUeuW6ruxdeQFX6JqQrdK2uk8QvVkSMCPyGAWFQbD7zLr9rHhzTbTodNMBxo4UqCMAoJhqFSS0smVH32Rw1ZnnreX
TAs1tMbl1SiXpr0jvCgOzDU6lDfG/6xgMhhdFmtu16xmHVarFEDWKfMW8caafTIyoHl2r6W/kYC3q79irpF27TTCEFzlBhAlKfFh
Onnr69fqypcIeN5zvOrNW/t1ASW2X0IdpGV+fUInITDOFKzDWzrFwHSdi5JM7At7sHIN7cK+MUTPLNd57CpA6CzJbE8Fze0t2lC6
LkrySA+Y1hAfrlXJtSq5abNHNo1ahuytilvh7JX+HTRv6jgRsCPxX0zpwSqpQ7VmEce0kymd0477UrECveUXSSozuOF0ubXv1OvV
Du531pJpEqsnKFvO21vVw4Atx+m37P6WuOKhglVJBWSOeyNBzFFvCzHpDDTI1HYr5Xo5bzhwb+1Xp5jQ29VMDaL52HVnazURVZNF
nJDfsrmcoOrWCu023xKCmLb6IG5HA3s+awezmMlmFwgxy2ZTw7wz5s+VkrO1f38OzlHo3uxj9hZwOFDw9tVCGw/ADhkhcxxOMYHD
Ju4wO7EGW/ZjBCqaXQ024boBDEYRnjzxX795/vYnC+VxHVANCh4q4nZAa3/tUzAe32+bLFNucejPgP36j+evhWdPdsdFAC4hfisC
P0YifG0wPGQYduhZoMgaAHwx6TI+qp0H4UjGiWnRwAwvcvVlFXFeZIISlkgLv9NErjcGEYf1m6Rtt2xl07JcBSUpUQ/ijIhr5niW
VyHnHWBtmd07QGm4SWFCG9Wh3Wvds2GVU9xwe8ee7Z0QLCbpVJkE/NryC3fCthWmnXE3bpaE+NW9nAVg7uZRJgPj8qvEglfuSRnX
0itRe3QaAgxifkYO79fJoHsjt1JJZkN03WHabw1py6R3qkQ2AZQUNnI12ETe2UxlZ7hjM5WJ4a7N/kBvO2723QXUrrD1u8Bqillv
bn+vGYBclndoYa2jZmFvaPndqqkJhKEdtwPo7YxSMlRro+kIADYFsKdFHIN7WY1d18UybD0YTqeLo7nr8hkfLHqDjWHrZcNaxHr5
iuIsehRuh/+BcIAS8AslVdEm95PFIuM5fY2kJzw007Mzj81bBWYbxATq2u86xZAqsvN7Khk++hUPaZuooRi/qyIOiQ8ahaUVP15+
OA6PQ3BTVW4Ugtcei33BlbxPTuAaPXORAqThRFXsEfJrns619H179i+GX5dzKSsssawkFX3xo1UljBnqIN5hnNkKvNRoTaRtiUgB
OjWqHr4LkNZRJRO7kBumQfZmpfp+9Ei2cbMAP7+Y66/uGK7lgweqUhN2yCSjPh31jwaHDn1gS29pYmxMa49t+9vETDiAWdvl12v8
osw+RShgXDjefrLs0Oof7SjKDwqwR+LR/ZxR7nrWYZ8z2c2tQOhv/Qgw8ssGEhT90tA6m6Ch/9yYkRYI28QkMytXbdOftAdI0p68
fv3ml9dPnz/zRNAj6k3PexNTAtpKifgGRPt4O3QbrIv7ED7eJRL7GFubViIad9a1IvOQRCIqD4OIKBunoExXpd+85Ydy6pFxm/6Q
P4nRwPDe3WIX4b7tqiCTTX66Zfb/pr+G4/KmP/1tldtWbTwob/q7vh3U69tDvLlFHZqo3dVwJnfXinfU2TaN3/b+yOzukIENBw+b
J1Ts49MGVac/UXsrtNH647p4lYScxHzNnabvC6Ji8zzDwW4Qv4467M9wAJhLlfElfQr4ClSdj9ahSFHrNGgkY8eKLCf8Pno1O7sp
7xyzmhUdjfjOQSXx8puHz/jlKQYx1Gvg5yOXngcE+JXSeRyrk7IxxaBNxrTdcKtdhro6oqicI/pe3Osk5kr56Gi1v71HiB/0t8HM
aEIVBOarQB4f7QxffS7W/zQcyLTAdopyy9a2FAR+7XajVW7VpATPW+xpq3J/3LttVQMBHdCiaMBMGvSt+xw+HlYAP2G4VeaL6EQL
sCBBba+lweyR+a9lBmtJkf8B09Wv0vGBAAA="""

HF_REPO = "Qwen/Qwen2.5-0.5B-Instruct-GGUF"
HF_REVISION = "9217f5db79a29953eb74d5343926648285ec7e67"
HF_FILENAME = "qwen2.5-0.5b-instruct-q8_0.gguf"
HF_EXPECTED_BYTES = 675710816
HF_EXPECTED_SHA256 = "ca59ca7f13d0e15a8cfa77bd17e65d24f6844b554a7b6c12e07a5f89ff76844e"

# Frozen Wave 28 production protocol.
PRODUCTION_REPEATS = 6
COLD_ITERS = 5
WARMUP_ITERS = 5
MEASURE_ITERS = 10
RETAIN_MEDIAN = 0.05
MEASURABLE_MEDIAN = 0.02
DIRECT_HARD_STOP = 1.10

WAVE24_PATCH_SHA256 = "b161faf1e0bc88688c320e423ef1b3f4ade1ebdf1850895910cb195e6d0e8ee8"
WAVE24_PATCH_GZIP_B64 = """H4sIAAAAAAAEAO09a3PbOJLf8ysQbyVDjSlZpKiH5SQ7TuKZmsommUsyN7vl8fEoEbK5lkiFpGx5k/z36248CL5kO7tXNbUb7U4sgUCj0S90N0AgjBYL1u2eRzkLDs6X800YHPBtsFoveXZwHVxx1/NXq8D1z/lq1UszNrtLrQcxv2aLaMnZKgk5c/r9kec9iOKQb1lffHq9YX8C//MedLtddhDyq4N4s1w+2N/fv2MfP/zAun27z/Yd2x0M2A8/PNg/OHjIfoOqzPVYGKV8nrPzIOdskaQsv+Asv06684sgitnzbpYH55z9dPL6NZsHcRiFULFHIAScDxdRxuD/AZsnKYKKeZYdpDxLNumcs2yech7bLE5ytk6TcDPPoyRmKc95TN/4VRTyeI4wH+xvMmiRh9NpHq34dPpzDL3H+ZF6JIY7nc42iwVPp9PnwfySx+Fz+nlUrhOm0RXW+YQ//eAqiJbBbMlt9gJ+f6lUvuRpzJfZdPqKvrzneaVCytfQ13T6ceL3/SwJ/DzxZ0QaQm6eAKbst+N3r3/9Zco2WfQPzp6y4ZF68vOHk3fviwduXz95d/LLyfEH45lHABcxQxKET358ZhEG7DHibbPVJmfLYBPPL6bsxw7rPmPveLZZ5k8WI89GdJMUCPfT8iRNk/TZg/3rC57yB/sMPj9Ci/j1JrfMZlan3sp+sP9JNEGB8BnIAcihGB2TT/Aj8LA6fz4SZV/EH8S3l93AozSJYVBFhSXPgb1BmsM4JW+n0zi5tjpH9f6IZv9Ud28vLeqtx5fBOuOh1ekFmZ/xeeYDtYAM3zOHj9iBYA8LMgbFnQf7XyQHgNnXPDq/yDMr2eR+GK0km2xA0fhJ9LT+m8+fbCbPbCa/dBTuOOqPGQzZglFJQNC1ANEpxtdbBWvrc/SZWVbUu06D9TqKz/3VZmm5h52iIAhDKwKkD8edDnvE3OGwg6hvJiaoebJcgi5qwh4csJNtAHq+cEbM/Z/uGJDZumBfgP3LKM+XvAuKFAUx/LrkZARARcEC8JC9T45ZtAJR7xlcnAdgb2pjssSgALuB2zHxWSyD3Kfx+Z/ZaX/b79sMETjbhbT1MbNlTwZXYBTRVYC2I7PS5DrbwRPkxGLgPlOcQGSxyf2oPxgLOjtEZ4DHusxxh71+B8bpuGP40jQIiXGWp2DzQN1Ax0ItEsGML0GnvwP5zKM5KEVqiycVQROFpZHJsjhPLsslimH+BmiC5kAU02RQLppFuc9RHKZsliRLWbqI0gyYFGWrIEfzElF9GkQE00tlCMCJv2dJbD3O+HJB1H6fp0AyU2NBnQHWQ6sowc/ep0+/72UXwZr/vjf9fe/Tl9/37N/35MCh6NMX+CmGrH7hYOX3B/u/7xkjxdJpb4CV5ECNkhTFRPz2ZFM9dAW6PGoq/fJlzy6jjGPsEdOaHkjUmx6JYTQ9wSE1lRtja3osB3lLSxDM22prQjQ9LBPFqNHRplepo5jgrcIWq6lKlKiZlT3WU6v9/6QD5rQmZNVmz5Ptk/AmFk4Fx9ltOhWT3DPTPCMgfx2EaNDwO9tnY9TuCViKiTF5xTOoYdg445FprKBSfe5QBqpjNsrB+Qt9bCq+lQBUPA3rMdZ7LKrYrArWnGS3AMC0k2p8TTjMbnJhyRUuvSXw0zBp+yXc6k+3oghI5ZnFmqbK2LY8BJpWWrpYilz4nhVTS62G4/e9iT8cj8T0N/KMMaGXBE4ijKrkIoKzwa/Jn7LFsEtEu6ZJGpr1guUymVfoITvp/LkXrvPUbJbNG5qZxGpsSm7LRZ6E1jWxVfWmUTIqZHNdQQpIidkLs38fJihLsqSlR1FlATC3ZUAVAlg1FrZTYVumAnVRYnK5iTQLvY8b8AFBff2PE8kXxGuLFNniqNtQAPfCxFybPpCXRjQKUaqjrsOae7Wudw5NP0O1z+YMqMaJwRiGZTBOpc1lszsvTKb6oFxUSoAk5ZJtrc62VgdwqpeQVglKVh5K89b4jIhRe6ImhSaa3o8oFLn+e1Km7KNZpsQWKq+oZpVksmwSWiMrNHoXCczbhkBe8fnD0z7I8VHFop41NTS5dmtLQibMkwtSk8e13m3WPMaWZrpvmzUPHjEt+yWAZalHwwuPcp6as1TvH9HaelzuyHy8TrIIp0rrs4VzQ+czC3o48UYwf3fYQzAKxc9K0OAnqdV1bIaxQ4QiAH5zRzMcAq8XySYGdGbBMojnwJUozqKQsyTm8PUqmdMczWZ8HmDGIZhlyXIDLPjgsTkYocuMhWm0yI3gy0KimX4icVD6eh0RlGF8BYFJQ1wtMw6mQpqe4/5TkXmQ5vjz5xaZ1XzBj+y7oXGbSP8rm/9T6MssAhr9BXiy1K4kc8ks46l0pJC26HtIGnZkzqDIN5RjJAJRiRrqwUItRqiEBqW4ruzkm7jZNXpONWFbaxqRYEW3nrC+Ua8xIPhiROUrwKqWWLqD9x0t2MNyhs7qmORbQ0yZL+OH1t6pSG52cUxnLE7Yi19fHrOQX0VzfoT5xQuMPlc8yDYpD/c6ZQHZpDHyB7S3zHearKB/4OwLyvSt02RmGFbAj0QCIrlF0stgvgr+nqQ2K5dFcZJ2gGTW2GbDTkW1sGsYtbWnUq4p/7iJUnC6oel4yEA3wS3l6R7Ay5NGDOWMCUjqKGo6XSZBaD0uyStSszq7uj6Pka6h1YpYBl389Bekp//Tu59fui+fOur38/cfjn86KX5jIth//frYferswFenf58q1A2baSapVUVfVCuZbHD8khLVJExRlW1iLTJ7JYWlwIf77nAE3SN5SvZ668vnMzKu/pqnfrayFHSbQTuwnTsQgQrd/CLl4JQm8/lmDVb9hn3c8PTmVpyGjvtVOEG73ThBhXvhBIJikOkJ89BSGkg+gRCrWVhacjoKkaL3POIpNDyHUWQ8nBr9Pf1UfP9i9KrL4fuXvZoXReOuiZwpdGh7UUlOiwYyPfFYTAePpTTabG+xiH1c9PA36z2bHfpjd2KzySHw3vUgvrHvCCJMruM92dDzJ5iEr7YHF+BdcH4OMz/YdR4fwAywhvl6nlzxFNdW1ssNLp/ESdw1VkheMUqR9e6AR0rQ/Utn1AdUHBwIfIV/xhoP092D2EbYfKCUpJn0lzDssT6Lss/yWZEoUkTXJrlATRtnJbJnDJN8AkXK8p0v/RZ3H7N/mJkT4puhUDSqg0rYqXogJLvqIUiYe4JVNPeBkjAl+BTyw+N+OcFXCKNtCKN4fpcxiwkJx3sdpKvNWqcwgaaZ+pHyNQ/yrJTEJNJDyemnL2dQBflSTk+W8RQLMEYBLVsYv6VXUnIdqI+yrtacYyrE9LfwXqZTTOxWn8u89nT6BLPq/rNnNQh/T8AF2LP3OiZtlazZZXriRFXI4S3zpVyiBNp0RX2xWBlgQqRlEhJTPbonYcPKbZbOD6T6yCKQm/Gwt863xaLqjkpyidZxZ4fjwajXO/SCw0V/rhZwcaX2Ln2VF3F3VcQ1XGfiDuwR26e/w77LoBAmYv/l2zcn0weKeEcPgAQMl2ZZu8pN9eovDmTN4Z84R2CMVn0zWt9lH8RKMHvljFjO4wz8lARUTHrCGKuEUYZMz1k2cNk6AB/RxqUjXN4FM4bhDT4R0F7jyt9NxkZej73BBeORB740TD5SnzvoB4HtYhbOP6qwR1iA7IXddZBfECSRV2XhZr2M5igJAOzVwO2SZ8YgTGBvAA7UD0MeHjFKMYl0n03LX5RCQ2lCRxgxg3HTwOIQaoRcNABF4ODnzYI0hYks60KF7iubwENonENPGfwDThyYFkZiKcb5nmzNlB2zQX/sYnpU9IdD3WfP2cjxPPhyXZQ+ZYcw+QgMBYzuv+7zYL93FWURTP6sB0wGf6BdLKRx662DNFix3gY4tPaLXEulXGdTyuXblvrblvo31VIUJb/IzJTLo7ixGCOmB/sdvW7dA78DqgAj2KP1E8d7dmSWz0AIHl08qZYCpEfpE29SLsb1vkeLJwO3UhuQf5SGTzwNBfhGWjWYMph1oGeoF2U4B2AUi/sfgmUE8SKurcab1YwjdhcoT5RsJLXq1RHyz2/8HCZz8S2eqW8pXzUhhI+2ia2+ZsXXm6SCKKiZ2t4BygyCHAs5NpSp0DGpdOY6cAnLWTwLMk7IzTBTLb5lG4HvLE2uG4jtz0BHU1m1+BoXANAKyOf4tXHE1F8UbsVIZx8VIvA9w+/NjT6u81TVKr4mi8VRgwhBHVRX9Uj4Ewzcpeg8Bl8LwE4wmAtOUefPjiSVoSvy+BitOW+ZNwH9F2aJrSOIoxvBeQraNjsF8yCgSXAojMW6jlxuugWp2SmaHI0UmljJzK9Da1ZBCwGiWZVAFU7S3wyFopKyA4Udm52STdEOaaWCKypk87YKA6qwbYfgiQrtEIZU4ab+mGRSYAj2p+W5QDCKWx4L9NAenZk5QGUZVuD0kwXAVOR3Gct4GgVLYG0XmMEyiNLYMknWDJ0adp5GYe+mx95xEOMoPtfQaI8WzJBcyFeX3BHkHk7GaHDQqsyXwQr3MIj07Sq4BBXnEHLcaDAw09PmBJF+XCICXM5lyxvqJEmj8ygGBLH7Je8SgvME5hEx3yl8+mIH2GqzzKM1TDXJgqZ3RCPHDUgQNHI26dgsSwgu34JlRNzSZBOHE1pk6WhwhYTTJI71VJ9gVDHTBITbLBbRPEK/BZObAUtliAVeIXvx4VgiB/SWjNF2dA5VgKySfdnFstHQ4rfRUeHdilFC7C5as+9hgLKLzbJ3HYVc9mNY4AKieyQJnk9A4ZDVYGgyngsIMMZepqVb/ysAaTxTYyBiGgBRHBooKjylfyNyx+C2iB7bkc0qyIoOTPoUU0tWnlrcI6NfNE6i7x0D9OziXwHlqB2zmwpmTjNSN+Yk14CUDLh3YDW0i39LcyVMYL1Mkx2EzxY6rrBSyAfbejX9VdeKYllrUHo+8spcBLTJNCfXuIgzQ1UE9QKhVgalkGokZA4Cua2JCQ3Ga5AQgg+RKhgxuWICaip1YCQbDZxaK2i0DGJeRWAMLeJmDCa2eN6OgRBPktQq3EOtqVtN47C3TCQBHZIHrCS6UX1Q2JPMwGIhfNBXyXt0ESjY2VbV3nFsCW/QoEsx6jwlv41JE80bmWAAqo3Wm4TxIAW7CbYN91UFKa3UsZjzUDh91xeJ0sseWzsOQCbGEqYXQaZBgaezNIQ2OwLMRWJCUg1s53LJhOu2z4wFQWn1QMTXvWUuSLmWI3SUBlX5hLlFYr3bxPRzMNHrn18Cssh+dqA2YhRC4wxk+xoByWpG56rtI9XW6NtrM2REfbHdhrbaMOuVtGkdQ33FlOxKrRwfVXETj/Df/vZH8ZkUoqI2F0wwE1aaiQSc+VWOy4A9IVLKfyAlCZUtaq4zpjruzjoTYet31jkU5nJnHaEL4dBgrRZuwhUZVOOtZI3eWKN3iK7FXmewPM9JwqfsFONp+n52CrGBYMPZKYgm/Ka49azwBY7N/DjYLJy6s80KnA9c8JSOzkWwXBzBpG2krc2qJUdnsQFZRwRQFUQ+YEE+idDpjP2Dp4nyX2fCb8GEARPj6BXTC1iPTTVaESak7gyY8Yz8OqhPxUaoIyvVRdCIgmQdxx23TXpmLFNAd7yGia8U6OhmjpI5Y3Yzq47MKKkJatYEddIEtKg5roRbhUSto2VyviGGbFaxSJbgLu4gRmdtkSar8l5jkerrPpeLPL2aQI+VQDtHVTPgTAw7N64/JlZjJcdE8T1YUfQwgyyDMGcFDuVUZp9YxHCJLWMyN4NmH4XXig68DgtETknDEZ4FPHzkdb6fqClivklT9FEhosba7BJc6AhXdDAricCeFEaoVxUvV83hjYZZmH+EgUveYZNldvV8XhNwlwjpNlltDZswluOi/YEhja1xjqE5QCDslhwwAep8E6Rhs8y7faNpXc7EY2ks3X5hCdH8aRjEdHfc2lz9MfyawlUQKK7zFCKW8Ao3aWRsf+AeXM46DcbDnSh8vUlNyOTDSQmdwqtxQQYxT9DW7lC0LnEEaSgicYEn5kmYtYgg4CtCJtx+lwlJVuKb4WaP0YBk2PDLUVTUez5KOoVP0sjXvpQgvcmhkK6BfqbFq9SWBiMqua7RWuZUCLT897A6YsTyCcaQx29eVvWkRYxciU2zuRSP6U+TkImnh7JOOe6TUoIkrsiIV4hIoVkDR2ttlf0D1xaJnRr/RaOBcGScFv4jBmXua/6/SGRqHnzE54UkULyvzBnmHsQsTBkdAIgaxIwcvAZHa5PsDQYmmI2XP2ESFvOvNxHWTOJFSSMWXCVRiL5pClzpTjSsWRBf4sy+AMOXg7eL6wPkDqPbuwxyWiVGcY3g6WoSTy6dEYxhAQE9WuNelVSUTCylF91h63QK1ssu2gyaJ8ci/WdOqUUqsGJsNNQme1MG15RYrHgiKu3ZYlDMFKmuWjcsKlOK+bmdMIqEKpY1RANG+nVcyHBZs2U6VCucbNPQcZVV3h051cgoI0+bNTLK9N90PtkrfAWTXllBsLo6ltLRmZGELhENhPsXnnZJkKUqvDs5fskQt2zKjg/k0g84pa/RiB2x5wfXuugNFjV5pwOaSzEq82ozgYrGwG2f1GwItZOtR9WWuIv//HuvuZGYeAaDmuFZ8RV49GQ+hOVZSQVng4lXN32ewtytWzjPsHBeY0do3xq7cms9ackaFs5No9zJL9otrmhfwVqq2KiARh35DahbFShn2Kp8utWwgNIgsPcc0LhJ5ivDqcu9O9gh8qq1OzAEv5zFTtfdTRzhHiC26lIARp4dSDv9WLF0A14+OreTlZyzm/0KiClIEhrti4sbWNofYzjrtT9Gf9dtf4x2tt/+GPfx7EAN81katZIZkPQQSzTzDQhakCdpxqyXGP1kLJ5Dr/Hc6bAtE9F0ZiSpxTIjekP9hXz1u39kPnFKT2oN3daGg90NvdaGw90NR60Nx7sbTlobHu5s6LYSx91NHLeVOO5u4ritxHGrxKkpGiJbT3xKebmc6XSWyEqSL4e7OV795e3bX6aG5J1z050WcYx2cH+AYjZLA9oI8tu7nz+cmFKJ6/8ge4VnSAGEMTHRzjNezfqoOUpsH6BuHmKEpzuiLaH+X98XK2DlhBSS7ZSCLrVIluVyNZGqnGKMcyZ8bRyJCXNq9Niv9Pi81qFghurOrXeHFU7RpcbuFuXOnk9vwV84cK1jEC6UMQ6FuHSPqtgfv6sTjHYiOKq7rLk7qHSqvQ/s78IpD+T4nRyKzjr3a3KwVp7KPFmt8f0CK7uM1mtguMxL32DKuZssuikmZ0SmuVMVAkcP6tX7v715UQxI4arzDqdqqjtrqzQ2Ku07o4Z6BnXkbKRfPgGXHHjbw4Vn0EZFk6b2rtl+322GgDPehWsS7d0mjjEvJGB1wT9JIJRCGBy3OaUcsyJ5oqbBPuMBBEGXpVjaiPyGDQ6WNgfHFO2Q21NrOGpwmHRDFRFWma2RsoLlNe66EssHU7EW/Owpc0zO1jgj5H8wbGXdUFYw2GagPDGWvIrSw6ZSr99Y6hilK/HKU08GrT0ZIuKL/LgpEd2XXjah/w/krk38fBKIYMdfbPzlevLvSPzVT//5joQ/4Tmyg6H8OxZ/9dO6TyxQoH+9fv2x8MoPRfMaL0zjNxiZcp3GJNoEZTEWfbQ9llQw3GPxGB6hzJJuLZR1WwCN9GNK3RB4UdPpNwFB+SX1agGCiQ9CQtR0nJrlksLsVG1RoykyqCdUDv3piVd/KvRqZCQFvqnBNzX4GjVwK2rgfo0aDCpqMGhTA7eiBu43NfimBn8ENfAqauB9jRoMK2owbFODQUUNBt/U4Jsa/BHUYFRRg9HXqMG4ogbjNjXwKmrgfVODb2rwR1CDSUUNJl+jBocVNThsU4NhRQ2G39Tgmxr8AdTArYTI7teEyG4lRHZbQ+RRRQ1G39Tgmxr8EdSgEiK7XxMiu5UQ2W0NkccVNRh/U4NvavBHUINKiOx+TYjsVkJkV4TIWrqnWiVOcPMXbvey5MsDeNBMknfYivM8Y/heFpuJV7PonVS+zeUWQ9zudZ1GuXqfoLSspEneuuHJ6x+OxAtPBNPcUI7bx4vXJlp399BXd0ibWOpgxOpaIxhju+fArW0yEcufBO1YbcWtgyh2CtbfyzFB4OKL2NdclUO5pbVf7MDQBgiXdzXDaMW2YNjP5bc8LvCdQXWuUZ4wkyOVxTg8A+BIw/kgD4K5jjP2t9P87DSe98/onUD5yznDXYx/D+Y8BnGg3aDsKe7Y2FcbjOoLWGLhub4TWWzXhOZqa5KlFp/0/sTyOrbYUjoobcH4obTa+5vet2K+8iO3SAo86B8zpjA3k6ltlDbz6vIqHqqXJrRJz3K1JnvlqlVjeHqGBkmudDhkkASC05rp0aPSLxPcc9QGD/8w4yZb5gwaDPG/5XDJPjvD/5ThytzSf8pwJyKH8B8yXBF6uk1O1L/lcF0RE/ynDNcTvt8X7UuIY4CoJR0DRCczsrsd/iLOPBRnCDWc/tJUjqe53Ha00ioJS1fh1J/Jg5TmQ5e7k0mvNxuNXXcW3nqQkmzden6SfE7HJoFk4KlJ8GeCRybJ+xher4LXSbgBynwS4zNJMZXHLMqhH6gDHNzBFLylTVq8r4b+LDXMcEdQROco4alHYje42DTUK3Vw6biTAv5+Gb43pROXzFOZ8ESHc57SKUXGvlJ5UhO9QP2Kzlgy+hBnPLWMoT+lLWA8zoKch13c/4Sw/+sVW2wy3A4W5RcMz6LFN6OSRb4KtuQ7Hv+3HEiQ5zF24bV24UzpNCgAhid1HWzW2IPN3l9HP/3lV3GG0n9N/L56xxkIhmEKndpALBseEstGjmDZejNTbNPHX5bZpu6xqLNLvlIq2BZkc7zlBNxpg2/0Cro45YmwlqMkNRHMMu7IKLEq5fo4D8Wv+Q2EVPk1l++4qOOzYPgZDpP2iBXs0kd0NuPvDI/x/VYtbEi3kC+CzTI/Eq+Og+uOLyLRK5TJtToZcwZhkgHr+iKCXvFF84sgx1N+jg+es+AcxQdon4s31OWo8R1bn6CGCifkiAshOXDE9Tx7jByh+0BqvFAfOqlZaBLEBXiG2TnP/cUmpqNGrOpZgFSRjqZsgAKx4F3hmBrcDg5Zeg9oWH2vdNqwCQx5dw9gWL0NM61VbfB0BZ/0FI/PboMlKmTX0flyg+BcZzd6Zv06TF4cfkiyMBySLIz6d5EF/EibKkXCrlco217N9PaaQi8VOytHsRv1hG4pTjXAM2yZwYKGiiaJphUSV6p/6Qg6HdJxfe7hoQ2+xK2EevyYzvtxw1qxGEutWByqHF9Np1dB6ieZtWce0fuK5LYXZX6WrLhlnoaMMqItD4rHKijqlUlZIFUtlkhVi3cjhecGl5HSbekA18L4AFqtoI4/fHjjv3v72/sdoEhPgJ/ykrnKIHdiSuABU28H+KqClSiIzB/0hzB37Q+coZjCdvPeZ0+fMccQoy9Gf4b6dc1Ge6fC8emqs5HEGaS0nR/FE8atDh89/xj4lJ7RBcRUfTapOxwVNy1FSw6yo34LPut2alJUBaQyBt+MHse+cFFKVZEv+uYlg4jFcaelMWIVNFBQg9tMD8OWUmmzlE5tVUjbUiztYva2TakiEAovuyIkdomrZcn+Q9Ba6+z/H/H3/9XE1zh/NR80Op1GnVCcOWPysN+ueik55Qs8JacwAvIY9D0T0NtLy9RKUl1ngnZ7AOjfYX7Toy8XS1KUCwu6lAldEKlc36RYBZJBvvKTCi2FP+3BlO044FF7o5HtOrtHRReAaVRFOR0pi1/Qn/yQbnjxGnfDSbJdsJ7y/tDiRpEID7vDE4R5KL1fdOsXMaufU19cKYceqHk6cIEbNhDlFOdq3DRGAyYiXopp5LEw4k7Q77LqGenS+ZUI0bHOAxp97Xz6ArW3ayTwE0HCZzUC4pQQZNB+YXXE1YIrCjc/M/G3Z3gynSYiG6S9z0D2qwNpOWh/x0D2v2og2EuniSNaWmRsyFYBvkezXgZzjJvijHeVrqpokd5LQ7MjglEYcQFPBpYYsuKdgxRS4jcKKOWpMXgI71q9miqPDMQDW+jMi2C55GmZ46bBaZdCUqVx36PYdOwO7aG3W5Oa+Kp5Sm/dFRftZvMLTuRkJxQjyqic7i/NxEE5dM4RDqSAVjuA+YjGSEcx01k74iDmUJ7EbC73lS/NOaBDH2IQKpzLKNLElJM8DDnK5Hmj8FPe1YuN/nSKJ9deW/NltF7fTKd5kvirIL7xg/R8g2c3ZJ2zRlVvv/CJiG6Yx/qFgvgRV6Fd+x+zKXvxq7gCZK2vDzRriNNI22ptd0DY3tL2pu1BcXth6SoofX9hqVTeYKjLqjeoqCt5TbUM+Wxz7gdZxtP8oVU2iTbbQydb3pVb3DRS98KPmgH6/ONDdWMhe4Qvafd3VpV3Yj2iV6771TBDnUTzVNsSXxYVtyLSiWuVdnijHYlCeaJEAJUSZZoqxXQ9hs+XGbc+f1ZUnE5P4nOQeUvQSN5pDeZAJL3w9jdxI4uwbPpo+c6fK9AN2a0gTlcz0YV+9AWvsMMvW1WyVSU3Hfa0DNUqpNquyK9NkmprmbSheVPHiYAdiT8xXQVVuSaymd5YnfK7ldsz8PNYDgnvOvoev/v629zHM1/spvr4YuI96m/vCX97T/g396qd3Kt2dK/a8V1qn1X4E/u0pP5UK9SBXkjAD11GJK/HLnf5/aKCgzXn0RIE4cpQQAIOxkY/oguo2MiDMqdTBSBRgEcNT/tNQxaS1XiLbC1zLJPQZb/gxx/f4BQI8xweHNjTNxOo48vlucdlJyGjN2N5ljG6swqzvsq/EGctg+XOjnQOGqovcWDCwUJvogBnHLlNB6CrrDL5Y/+rroH5357Mj03GIkEG0ZHI/IA9YTnPYCKvRA+/fPir//71eNjDuBK9eWvvTis2Fl6yUYK0JywXAMShrKIsowE3NWaWOv59Xjs1inf2ShGXZpmabsrs/Vr8xbS/V5We5kHcHmVQT+aFQUaEpxH/JwmPwe5Xkh2bMuuXiyBDMotzZLspxxs0cA4CkpPcjPojiDhBbsaHIvJsFBs6DixaQBjFEnDF2AaIkbIVhN0o0dg7rqXg3QMgrXRuF56rFWOnM4gAwDdOCodaAhS1hccI0EAcQgZiccHxmLgALRZJO94RB46hOFSMiIQHeqd4EGIJnLpMUaz0zEG7aN2DTh7LxXIEwpBxzSJBJxKP/+oVyR9UsJyo45+8+fDu55P3U3b6OMvBwR2fleeoHVUnomqZY2XmWHsVjtLr+HgXE7kHAsUMSWiu68gbaeR6C44Sa1DS+qgGLorny03IcaklwFPSJW9meOBsgNATA2ZXwSSxRa5I4XA8YVTAuLQLhxpgw1pBbaCNyzC31lL2p6q5u9T8TiApEr5r3bvhKlRWrmCN5BLW2PF2EJAuxAY9lWEy3ZB13rBkYu1VU8KE0m3NFOLVxooEt7TWxr/qvmGEf7fGyvJWEKBkxx0hCEbVh3A3CpQ5WFsCwNu/bgMhTfGRZOzYEYz1YLo9bOWsmgeMkRrGX+/PxKcv3r7+5dcPJ3udpnnkYTOAR2n4i3+vFms/mOHxh3sdk51GikCckosnNsrbGIxF5vM0WF/gIbNkWdVeThxCCRStZ8sridSpkGSu1IK3WFbfpHQ20A101ZAbkLBeUYohv7DRgsM0Ka/7gFJKMohboebJagYtCYq5iyFM8qxX9SYogDRkt0eOGgfaVLeW73WAcJs4x5DYGdX9knZIxtZlE4h7Hxit+89NiJOvhFjesn4LxDK4QpaKu2fK0nRrK3nHzD1byStl7ttqV18PdwzMcd3J5H7t7qLLxJ6S5S0YpG4JKQleCwjDZu0G0L91ldIweI1TuLZ5kwm5iuPxaLeriHMZ2xeTE14/1lWXBiDSvcbhtDjKemTVfWnGCMuLdxPoRv03cNWPst/QXIfq6TqNNrXBgS8QM3bLCcYLqo37jqDa5BYHG4nzXSb2DB1ruwleIM++mmbF2WGtBHNg3Pgf0sGTf90yvRqrULWd5CpLeasu36udoc1tGwHTTRzztLYHUBfL7X98xt1F3+31Jv1gMQ+91u1/RcPazr/ikVjDo70X+MehixIXsXBx5IDoalXLSMmaeX6igHiGq/rqUiM/WVhisUZesFjM1Jecr7PyQfZK06gnfX2RLYMBmoblNQav8N0ohCaOA6QL4WJ9bwleZ5TyNR43FufFAlDxMgzdzSCvSeqJq7jRIpYGWkpJs5Zhk5zJmtVxT2jLRDNN6HRWSuVUV1KaOr+IQnAIdmPCDEzY06eyTS8LwFEJ8Noo7N9yO4W8AnKyUhPDUCI8l3YteQPHdkdSIsinxEXWiv5WP5KWmCXzxXH4mQXN+jiAWafHt2sI3MBo43ITbtObQdhd8myrH0qNU4b3Y0an98PX3sesF+L54vvsYya7uRMIkRLWYMRPBYp+aWi1bVLqEy3YZeOF4UDYujghycTYzWWJpg+0yqM5uqWuf/zmzdtf37w4eTkVe2/QK5tO38Z0S3ilZDqN+XVp01LTpwy2h6l8H3MimPK/BTH88IZLfXd99CYQJJTYAIL+MF1vLOjTVfcZ70np1VcBx+YvTK+27Llo+2iS2yy+vcVthPtyWwV5C28hE+0rdm2feXnJru2j1ODOVeUKyO3Vt3eDur07xJs71CFG3V4NOXl7rfiWOrvY+KXddhTaTnNiSdubZknS852ISj0nj/ur9Hwn9DLYqp6LZXkPd6PuO+PRyNi189N68zoJOc0pNV8P06/ZZs3T6dQYXYOtt/W9RKgFNIvzJcf17eso4z4uGvjqeuHa9Ge4eDXSih1U/jq4MUWmoVrNBJdaCnYfFaOawywJpJ1tFgsc3Ut+9R4XO+o1Vkic6RTI9BtF80eSmOO+2ONwOLKd4a0ZlYd1mTkcaXdS73740ylCkBsCKrtTwihbo6/skyflg4fjK0+qBNhXjpRVmnpKoWcZEX+M29AmAqE7NICaNvP8yci7YwPa5uaM+jui00p97571Eb7nmTecC8oqgjKToE5pZ7KPuSMQWdyjkPli60wJfGZ1mlja4MZJShJpNEH/D8KlsjkXmwAA"""
WAVE50_PATCH_SHA256 = "915ea7ede79038970018ae10d2c7851d0e01e64f3a73d38e1c0eef764b6dc6c9"
WAVE50_PATCH_GZIP_B64 = """..."""
WORK = Path("/kaggle/working") if Path("/kaggle/working").is_dir() else Path("/content")
RUN_ID = dt.datetime.now(dt.timezone.utc).strftime("%Y%m%dT%H%M%SZ")
ROOT = WORK / f"glcuda-wave58-{RUN_ID}"
META_REPO = ROOT / "meta"
RESULTS = ROOT / "results"
TREE = ROOT / "wave50"
TARGET = ROOT / "target-wave50"
FINAL_ZIP = WORK / "glcuda_t4_wave62_n32_production_results.zip"
ROOT.mkdir(parents=True, exist_ok=False)
RESULTS.mkdir(parents=True)

def run(cmd, cwd=None, env=None, timeout=1800, check=True):
    merged = os.environ.copy()
    for key in [
        "GLCUDA_FORCE_Q8", "GLCUDA_GRID2D", "GLCUDA_R256", "GLCUDA_NO_MMA",
        "GLCUDA_FUSE_Q8_GLUE", "GLCUDA_GQA_GROUP", "GLCUDA_NTILE128",
        "GLCUDA_BSTAGE", "GLCUDA_TELEMETRY", "GLCUDA_CACHE", "GLCUDA_ATTN_ROWS",
        "GLCUDA_GQA7_CHAINS", "GLCUDA_ATTN_MMA4", "GLCUDA_FUSED_SWIGLU",
        "GLCUDA_GEMM_K128", "GLCUDA_GEMM_MMA2", "GLCUDA_GEMM_N16",
    ]:
        merged.pop(key, None)
    if env:
        merged.update({str(k): str(v) for k, v in env.items()})
    p = subprocess.run(
        [str(x) for x in cmd], cwd=str(cwd) if cwd else None, env=merged,
        text=True, stdout=subprocess.PIPE, stderr=subprocess.PIPE, timeout=timeout,
    )
    if check and p.returncode:
        raise RuntimeError(f"command failed ({p.returncode}): {cmd}\nSTDOUT:\n{p.stdout}\nSTDERR:\n{p.stderr}")
    return p

def save_log(name, proc):
    (RESULTS / name).write_text(
        f"returncode={proc.returncode}\n\nSTDOUT\n{proc.stdout}\n\nSTDERR\n{proc.stderr}",
        encoding="utf-8",
    )

def archive():
    if FINAL_ZIP.exists():
        FINAL_ZIP.unlink()
    with zipfile.ZipFile(FINAL_ZIP, "w", zipfile.ZIP_DEFLATED) as z:
        for path in sorted(RESULTS.rglob("*")):
            if path.is_file():
                z.write(path, path.relative_to(RESULTS))
    return FINAL_ZIP

def sha256_file(path):
    h = hashlib.sha256()
    with Path(path).open("rb") as f:
        for chunk in iter(lambda: f.read(8 << 20), b""):
            h.update(chunk)
    return h.hexdigest()


def fail_phase(phase):
    text = traceback.format_exc()
    (RESULTS / "FAILED.json").write_text(
        json.dumps({"phase": phase, "traceback": text}, indent=2), encoding="utf-8"
    )
    archive()
    raise RuntimeError(f"Wave 62 {phase} failed; partial archive: {FINAL_ZIP}")


def decode_patch(encoded, expected, label):
    data = gzip.decompress(base64.b64decode(encoded))
    got = hashlib.sha256(data).hexdigest()
    if got != expected:
        raise RuntimeError(f"{label} patch hash mismatch: {got} != {expected}")
    path = RESULTS / f"{label}.patch"
    path.write_bytes(data)
    return path

def ensure_cargo():
    cargo_home = WORK / ".wave27-cargo"
    rustup_home = WORK / ".wave27-rustup"
    candidates = [
        shutil.which("cargo"), cargo_home / "bin/cargo", Path.home() / ".cargo/bin/cargo",
        Path("/usr/local/cargo/bin/cargo"),
    ]
    cargo = next((Path(x) for x in candidates if x and Path(x).is_file()), None)
    if cargo is None:
        installer = ROOT / "rustup-init"
        url = "https://static.rust-lang.org/rustup/dist/x86_64-unknown-linux-gnu/rustup-init"
        last_error = None
        for attempt in range(1, 6):
            try:
                request = urllib.request.Request(
                    url, headers={"User-Agent": "GwenLand-glcuda-Wave27/1.0", "Accept-Encoding": "identity"},
                )
                with urllib.request.urlopen(request, timeout=120) as response:
                    payload = response.read()
                if len(payload) < (1 << 20):
                    raise RuntimeError(f"rustup-init download is unexpectedly small: {len(payload)} bytes")
                installer.write_bytes(payload)
                installer.chmod(0o755)
                break
            except Exception as exc:
                last_error = exc
                if attempt == 5:
                    raise RuntimeError(f"cannot download rustup-init: {last_error}") from exc
                time.sleep(min(30, 2 ** attempt))
        install_env = {
            "CARGO_HOME": str(cargo_home), "RUSTUP_HOME": str(rustup_home),
            "PATH": f"{cargo_home / 'bin'}:{os.environ.get('PATH', '')}",
        }
        install = run(
            [installer, "-y", "--profile", "minimal", "--default-toolchain", "stable", "--no-modify-path"],
            env=install_env, timeout=1800,
        )
        save_log("rustup-init.log", install)
        cargo = cargo_home / "bin/cargo"
    cargo_bin = cargo.parent
    os.environ["PATH"] = f"{cargo_bin}:{os.environ.get('PATH', '')}"
    if cargo_home in cargo.parents:
        os.environ["CARGO_HOME"] = str(cargo_home)
        os.environ["RUSTUP_HOME"] = str(rustup_home)
    cargo = Path(shutil.which("cargo") or cargo)
    rustc = shutil.which("rustc")
    if not cargo.is_file() or not rustc:
        raise RuntimeError(f"Rust toolchain bootstrap incomplete: cargo={cargo}, rustc={rustc}")
    cargo_version = run([cargo, "--version"])
    rustc_version = run([rustc, "--version", "--verbose"])
    save_log("rust-toolchain.log", cargo_version)
    with (RESULTS / "rust-toolchain.log").open("a", encoding="utf-8") as f:
        f.write(f"\n\nRUSTC\n{rustc_version.stdout}\n{rustc_version.stderr}")
    print(f"Rust toolchain: {cargo_version.stdout.strip()} / {rustc_version.stdout.splitlines()[0]}")
    return str(cargo)

PATCH_ORDER = [
    "wave3", "wave4", "wave11", "wave12", "wave13", "wave13b",
    "wave15", "wave19", "wave20", "wave21", "wave22", "wave23",
    "wave24", "wave50",
]

def ptx_region(text, entry):
    begin = text.index(f".visible .entry {entry}(")
    body_open = text.index("{", begin)
    finish = text.find("\n}", body_open)
    if finish < 0:
        raise ValueError(f"unterminated PTX entry: {entry}")
    return text[begin:finish + 2]

def arithmetic_suffix(kernel):
    begin = kernel.index("MMA_STAGE_BAR:")
    finish = kernel.rfind("\n}") + 2
    return kernel[begin:finish]

def compile_phase():
    global PATCHES, GPU_INFO, CARGO, TREE, PTXAS_RESOURCES, DIRECT_RESOURCE
    global DIRECT_DIAGNOSTIC, PARITY, LIB_TESTS, GLBENCH, STACK_OK, MODEL_OK, PTXAS

    patch_rows = [
        ("wave3", WAVE3_PATCH_GZIP_B64, WAVE3_PATCH_SHA256),
        ("wave4", WAVE4_PATCH_GZIP_B64, WAVE4_PATCH_SHA256),
        ("wave11", WAVE11_PATCH_GZIP_B64, WAVE11_PATCH_SHA256),
        ("wave12", WAVE12_PATCH_GZIP_B64, WAVE12_PATCH_SHA256),
        ("wave13", WAVE13_PATCH_GZIP_B64, WAVE13_PATCH_SHA256),
        ("wave13b", WAVE13B_PATCH_GZIP_B64, WAVE13B_PATCH_SHA256),
        ("wave15", WAVE15_PATCH_GZIP_B64, WAVE15_PATCH_SHA256),
        ("wave19", WAVE19_PATCH_GZIP_B64, WAVE19_PATCH_SHA256),
        ("wave20", WAVE20_PATCH_GZIP_B64, WAVE20_PATCH_SHA256),
        ("wave21", WAVE21_PATCH_GZIP_B64, WAVE21_PATCH_SHA256),
        ("wave22", WAVE22_PATCH_GZIP_B64, WAVE22_PATCH_SHA256),
        ("wave23", WAVE23_PATCH_GZIP_B64, WAVE23_PATCH_SHA256),
        ("wave24", WAVE24_PATCH_GZIP_B64, WAVE24_PATCH_SHA256),
        ("wave50", WAVE50_PATCH_GZIP_B64, WAVE50_PATCH_SHA256),
    ]
    PATCHES = {name: decode_patch(b64, sha, name) for name, b64, sha in patch_rows}
    (RESULTS / "patch-manifest.json").write_text(
        json.dumps([
            {"name": name, "sha256": sha, "bytes": PATCHES[name].stat().st_size}
            for name, _, sha in patch_rows
        ], indent=2), encoding="utf-8"
    )

    gpu = run(
        ["nvidia-smi", "--query-gpu=index,name,compute_cap,memory.total,driver_version",
         "--format=csv,noheader,nounits"], timeout=60, check=False,
    )
    save_log("nvidia-smi.log", gpu)
    rows = [x.strip() for x in gpu.stdout.splitlines() if x.strip()]
    fields = [x.strip() for x in rows[0].split(",")] if rows else []
    if gpu.returncode or len(fields) < 5 or fields[1] != "Tesla T4" or fields[2] != "7.5":
        raise RuntimeError(f"Wave 50 requires Tesla T4 sm_75, got {rows}")
    GPU_INFO = {"raw": rows[0], "name": fields[1], "compute_cap": fields[2], "driver": fields[4]}
    (RESULTS / "gpu.json").write_text(json.dumps(GPU_INFO, indent=2), encoding="utf-8")
    CARGO = ensure_cargo()

    stack_log = []
    commands = [
        (["git", "clone", "--filter=blob:none", "--no-checkout", REPO_URL, META_REPO], None, 1800),
    ]
    for cmd, cwd, timeout in commands:
        p = run(cmd, cwd=cwd, timeout=timeout, check=False)
        stack_log.append({"cmd": [str(x) for x in cmd], "returncode": p.returncode,
                          "stdout": p.stdout, "stderr": p.stderr})
        if p.returncode:
            raise RuntimeError(f"stack command failed: {cmd}")
    cat = run(["git", "cat-file", "-t", BASE_REV], cwd=META_REPO, check=False)
    if cat.returncode:
        p = run(["git", "fetch", "--depth", "1", "origin", BASE_REV],
                cwd=META_REPO, timeout=1800, check=False)
        stack_log.append({"cmd": ["git", "fetch", BASE_REV], "returncode": p.returncode,
                          "stdout": p.stdout, "stderr": p.stderr})
        if p.returncode:
            raise RuntimeError("cannot fetch base revision")
    p = run(["git", "worktree", "add", "--detach", TREE, BASE_REV],
            cwd=META_REPO, check=False)
    stack_log.append({"cmd": ["git", "worktree", "add", str(TREE)], "returncode": p.returncode,
                      "stdout": p.stdout, "stderr": p.stderr})
    if p.returncode:
        raise RuntimeError("cannot create reconstruction worktree")
    for name in PATCH_ORDER:
        p = run(["git", "apply", "--whitespace=error", PATCHES[name]], cwd=TREE, check=False)
        stack_log.append({"cmd": ["git", "apply", name], "returncode": p.returncode,
                          "stdout": p.stdout, "stderr": p.stderr})
        if p.returncode:
            raise RuntimeError(f"historical stack apply failed: {name}")
    global CANDIDATE_SNAPSHOT_MANIFEST
    candidate_snapshot = {'glcuda/src/kernels/mod.rs': {'sha256': 'aa7498a532030f09b36b87e5f1f2a20b4198cfdc99cb59d0ad1934f934b090a1', 'base64': '...'}, 'glcuda/src/runner.rs': {'sha256': 'b74df9bdd488acfa947334a1751e5ef4b75ad3c0c06ad7ab0a82a5a21a17c398', 'base64': '...'}, 'glcuda/src/kernels/glcuda_sm75.ptx': {'sha256': '2a87248b17bbd3aef50209295e1a4c2f71d9fa559f14cb84564a99fd7f583e55', 'base64': '...'}, 'glcuda/src/kernels/glcuda_sm75_wave59.ptx': {'sha256': '0f3a390c1bc9509836aed1ccd450dd341b5f6d91cf5c5c18916f0b7913ca86f6', 'base64': '...'}, 'glcuda/tests/wave59_n32.rs': {'sha256': '4f812b4d234ab05ca7b9a4823ef38d9169dc9382ff8605b13347afb2f91beaae', 'base64': 'Ly8hIFdhdmUgNTkgZGV2aWNlIHBhcml0eSBnYXRlLiBSdW4gc2VyaWFsbHkgYmVjYXVzZSBpdCBvd25zIHByb2Nlc3MgZmxhZ3M6Ci8vISBgY2FyZ28gdGVzdCAtcCBnbGN1ZGEgLS10ZXN0IHdhdmU1OV9uMzIgLS0gLS10ZXN0LXRocmVhZHM9MWAuCgp1c2UgZ2xjdWRhOjpidWZmZXI6OkJhY2tlbmRCdWZmZXI7CnVzZSBnbGN1ZGE6OmRyaXZlcjo6e2N1ZGFfYXZhaWxhYmxlLCBDdWRhfTsKdXNlIGdsY3VkYTo6a2VybmVsczo6S2VybmVsU2V0Owp1c2UgZ2xjdWRhOjpyZXBhY2s6OnE4XzBfc29hX3RvX2JzdGFnZTsKCiNbdGVzdF0KZm4gbjMyX20zMl9pc19iaXRfZXhhY3RfdG9fcmV0YWluZWRfbjE2X20zMigpIHsKICAgIGlmICFjdWRhX2F2YWlsYWJsZSgpIHsKICAgICAgICBlcHJpbnRsbiEoIlNLSVA6IG5vIENVREEgZHJpdmVyL2RldmljZSBvbiB0aGlzIG1hY2hpbmUiKTsKICAgICAgICByZXR1cm47CiAgICB9CiAgICBmb3IgZmxhZyBpbiBbCiAgICAgICAgIkdMQ1VEQV9HUklEMkQiLAogICAgICAgICJHTENVREFfTlRJTEUxMjgiLAogICAgICAgICJHTENVREFfQlNUQUdFIiwKICAgICAgICAiR0xDVURBX0dFTU1fTjE2IiwKICAgICAgICAiR0xDVURBX0dFTU1fTjMyIiwKICAgIF0gewogICAgICAgIHN0ZDo6ZW52OjpzZXRfdmFyKGZsYWcsICIxIik7CiAgICB9CiAgICBsZXQgY3VkYSA9IEN1ZGE6OnByb2JlKCkuZXhwZWN0KCJDVURBIHByb2JlIik7CiAgICBpZiAoY3VkYS5pbmZvLnNtX21ham9yLCBjdWRhLmluZm8uc21fbWlub3IpIDwgKDcsIDUpIHsKICAgICAgICBlcHJpbnRsbiEoIlNLSVA6IFdhdmUgNTkgcmVxdWlyZXMgc21fNzUgb3IgbmV3ZXIiKTsKICAgICAgICByZXR1cm47CiAgICB9CiAgICBsZXQga2VybmVscyA9IEtlcm5lbFNldDo6bG9hZCgmY3VkYSkuZXhwZWN0KCJXYXZlIDU5IFBUWCBtdXN0IEpJVCIpOwogICAgYXNzZXJ0IShrZXJuZWxzLmdlbW1fbjMyX2VuYWJsZWQoKSk7CgogICAgZm9yIChvdXRfZGltLCBpbl9kaW0sIG50b2spIGluIFsoMTM2dXNpemUsIDE2MHVzaXplLCAxN3VzaXplKSwgKDg5NiwgMjU2LCAzMyldIHsKICAgICAgICBleGFjdF9jYXNlKCZjdWRhLCAma2VybmVscywgb3V0X2RpbSwgaW5fZGltLCBudG9rKTsKICAgIH0KfQoKZm4gZXhhY3RfY2FzZShjdWRhOiAmQ3VkYSwga2VybmVsczogJktlcm5lbFNldCwgb3V0X2RpbTogdXNpemUsIGluX2RpbTogdXNpemUsIG50b2s6IHVzaXplKSB7CiAgICBsZXQgbnRva19wYWQgPSBudG9rLmRpdl9jZWlsKDgpICogODsKICAgIGxldCBuYiA9IGluX2RpbSAvIDMyOwogICAgbGV0IHFzOiBWZWM8dTg+ID0gKDAuLm91dF9kaW0gKiBpbl9kaW0pCiAgICAgICAgLm1hcCh8aXwgKChpLndyYXBwaW5nX211bCgyOSkud3JhcHBpbmdfYWRkKGkgLyA5NykpICUgMjU1KSBhcyB1OCkKICAgICAgICAuY29sbGVjdCgpOwogICAgbGV0IHNjYWxlczogVmVjPHU4PiA9ICgwLi5vdXRfZGltICogbmIpLmZsYXRfbWFwKHxffCBbMHgwMCwgMHgyMF0pLmNvbGxlY3QoKTsKICAgIGxldCAodGlsZWRfcXMsIHRpbGVkX3NjYWxlcykgPQogICAgICAgIHE4XzBfc29hX3RvX2JzdGFnZSgmcXMsICZzY2FsZXMsIG91dF9kaW0sIGluX2RpbSkuZXhwZWN0KCJCLXN0YWdlIHJlcGFjayIpOwogICAgbGV0IHg6IFZlYzxmMzI+ID0gKDAuLm50b2tfcGFkICogaW5fZGltKQogICAgICAgIC5tYXAofGl8ICgoaS53cmFwcGluZ19tdWwoMzcpICUgMjUxKSBhcyBmMzIgLSAxMjUuMCkgLyAxMjcuMCkKICAgICAgICAuY29sbGVjdCgpOwogICAgbGV0IGJ5dGVzID0gKHRpbGVkX3FzLmxlbigpCiAgICAgICAgKyB0aWxlZF9zY2FsZXMubGVuKCkKICAgICAgICArIHgubGVuKCkgKiA0CiAgICAgICAgKyBudG9rX3BhZCAqIGluX2RpbQogICAgICAgICsgbnRva19wYWQgKiBuYiAqIDQKICAgICAgICArIDIgKiBudG9rICogb3V0X2RpbSAqIDQKICAgICAgICArIDY0ICogMTAyNCkgYXMgdTY0OwogICAgbGV0IG11dCBidWYgPSBCYWNrZW5kQnVmZmVyOjpuZXcoY3VkYSwgYnl0ZXMpLmV4cGVjdCgiZGV2aWNlIGJ1ZmZlciIpOwogICAgbGV0IHdxcyA9IGJ1Zi5hbGxvYyh0aWxlZF9xcy5sZW4oKSBhcyB1NjQpLnVud3JhcCgpLmRwdHI7CiAgICBsZXQgd3NjID0gYnVmLmFsbG9jKHRpbGVkX3NjYWxlcy5sZW4oKSBhcyB1NjQpLnVud3JhcCgpLmRwdHI7CiAgICBjdWRhLmh0b2Qod3FzLCAmdGlsZWRfcXMpLnVud3JhcCgpOwogICAgY3VkYS5odG9kKHdzYywgJnRpbGVkX3NjYWxlcykudW53cmFwKCk7CiAgICBsZXQgeGYgPSBidWYuYWxsb2NfZjMyKHgubGVuKCkpLnVud3JhcCgpLmRwdHI7CiAgICBjdWRhLmh0b2RfZjMyKHhmLCAmeCkudW53cmFwKCk7CiAgICBsZXQgeHFzID0gYnVmLmFsbG9jKChudG9rX3BhZCAqIGluX2RpbSkgYXMgdTY0KS51bndyYXAoKS5kcHRyOwogICAgbGV0IHhzYyA9IGJ1Zi5hbGxvY19mMzIobnRva19wYWQgKiBuYikudW53cmFwKCkuZHB0cjsKICAgIGtlcm5lbHMKICAgICAgICAucXVhbnRpemVfcTgoY3VkYSwgeGYsIHhxcywgeHNjLCAobnRva19wYWQgKiBpbl9kaW0pIGFzIHUzMikKICAgICAgICAudW53cmFwKCk7CiAgICBsZXQgcmV0YWluZWQgPSBidWYuYWxsb2NfZjMyKG50b2sgKiBvdXRfZGltKS51bndyYXAoKS5kcHRyOwogICAgbGV0IGNhbmRpZGF0ZSA9IGJ1Zi5hbGxvY19mMzIobnRvayAqIG91dF9kaW0pLnVud3JhcCgpLmRwdHI7CgogICAga2VybmVscwogICAgICAgIC5nZW1tX21tYV9xOF9ic3RhZ2VfbjE2KAogICAgICAgICAgICBjdWRhLAogICAgICAgICAgICB3cXMsCiAgICAgICAgICAgIHdzYywKICAgICAgICAgICAgeHFzLAogICAgICAgICAgICB4c2MsCiAgICAgICAgICAgIHJldGFpbmVkLAogICAgICAgICAgICBvdXRfZGltIGFzIHUzMiwKICAgICAgICAgICAgaW5fZGltIGFzIHUzMiwKICAgICAgICAgICAgbnRvayBhcyB1MzIsCiAgICAgICAgKQogICAgICAgIC51bndyYXAoKTsKICAgIGtlcm5lbHMKICAgICAgICAuZ2VtbV9tbWFfcThfYnN0YWdlX24zMl9tMzIoCiAgICAgICAgICAgIGN1ZGEsCiAgICAgICAgICAgIHdxcywKICAgICAgICAgICAgd3NjLAogICAgICAgICAgICB4cXMsCiAgICAgICAgICAgIHhzYywKICAgICAgICAgICAgY2FuZGlkYXRlLAogICAgICAgICAgICBvdXRfZGltIGFzIHUzMiwKICAgICAgICAgICAgaW5fZGltIGFzIHUzMiwKICAgICAgICAgICAgbnRvayBhcyB1MzIsCiAgICAgICAgKQogICAgICAgIC51bndyYXAoKTsKICAgIGN1ZGEuc3luY2hyb25pemUoKS51bndyYXAoKTsKCiAgICBsZXQgbXV0IGEgPSB2ZWMhWzBmMzI7IG50b2sgKiBvdXRfZGltXTsKICAgIGxldCBtdXQgYiA9IHZlYyFbMGYzMjsgbnRvayAqIG91dF9kaW1dOwogICAgY3VkYS5kdG9oX2YzMigmbXV0IGEsIHJldGFpbmVkKS51bndyYXAoKTsKICAgIGN1ZGEuZHRvaF9mMzIoJm11dCBiLCBjYW5kaWRhdGUpLnVud3JhcCgpOwogICAgYnVmLmZyZWUoY3VkYSkudW53cmFwKCk7CiAgICBsZXQgbWlzbWF0Y2ggPSBhCiAgICAgICAgLml0ZXIoKQogICAgICAgIC56aXAoJmIpCiAgICAgICAgLnBvc2l0aW9uKHwobGVmdCwgcmlnaHQpfCBsZWZ0LnRvX2JpdHMoKSAhPSByaWdodC50b19iaXRzKCkpOwogICAgYXNzZXJ0X2VxISgKICAgICAgICBtaXNtYXRjaCwgTm9uZSwKICAgICAgICAiV2F2ZSA1OSBtaXNtYXRjaCBhdCB7bWlzbWF0Y2g6P306IG91dD17b3V0X2RpbX0gaW49e2luX2RpbX0gbnRvaz17bnRva30iCiAgICApOwp9Cg=='}, 'glcuda/examples/wave59_n32_gemm.rs': {'sha256': '08b26f5c87491b63f693e703a243375fe050b8e422a82ce1095d4a964012d4d2', 'base64': 'Ly8hIFdhdmUgNTkgZGlyZWN0IFQ0IHNjcmVlbiBmb3IgdGhlIGlzb2xhdGVkIE4zMi9NMzIgbmFycm93LWdyaWQga2VybmVsLgovLyEgQ29ycmVjdG5lc3MgYW5kIHJlc291cmNlIGNoZWNrcyBhcmUgaGFyZCBnYXRlczsgdGltaW5nIGlzIGludGVybGVhdmVkCi8vISBkaWFnbm9zdGljIGV2aWRlbmNlIGFuZCBkb2VzIG5vdCBieSBpdHNlbGYgcmV0YWluIHRoZSBwcm9kdWN0aW9uIHBhdGguCgp1c2Ugc3RkOjp0aW1lOjpJbnN0YW50OwoKdXNlIGdsY3VkYTo6YnVmZmVyOjpCYWNrZW5kQnVmZmVyOwp1c2UgZ2xjdWRhOjpkcml2ZXI6OntjdWRhX2F2YWlsYWJsZSwgQ3VkYX07CnVzZSBnbGN1ZGE6Omtlcm5lbHM6Oktlcm5lbFNldDsKdXNlIGdsY3VkYTo6cmVwYWNrOjpxOF8wX3NvYV90b19ic3RhZ2U7Cgpjb25zdCBXQVJNVVA6IHVzaXplID0gMTA7CmNvbnN0IElURVJTOiB1c2l6ZSA9IDEwMDsKY29uc3QgUkVQRUFUUzogdXNpemUgPSA1OwoKZm4gdGltZWQ8Rj4oY3VkYTogJkN1ZGEsIG11dCBsYXVuY2g6IEYpIC0+IFJlc3VsdDxmNjQsIGdsY29yZTo6R2xFcnJvcj4Kd2hlcmUKICAgIEY6IEZuTXV0KCkgLT4gUmVzdWx0PCgpLCBnbGNvcmU6OkdsRXJyb3I+LAp7CiAgICBmb3IgXyBpbiAwLi5XQVJNVVAgewogICAgICAgIGxhdW5jaCgpPzsKICAgIH0KICAgIGN1ZGEuc3luY2hyb25pemUoKT87CiAgICBsZXQgc3RhcnQgPSBJbnN0YW50Ojpub3coKTsKICAgIGZvciBfIGluIDAuLklURVJTIHsKICAgICAgICBsYXVuY2goKT87CiAgICB9CiAgICBjdWRhLnN5bmNocm9uaXplKCk/OwogICAgT2soc3RhcnQuZWxhcHNlZCgpLmFzX3NlY3NfZjY0KCkgKiAxZTYgLyBJVEVSUyBhcyBmNjQpCn0KCmZuIG1lZGlhbih2YWx1ZXM6ICZtdXQgW2Y2NF0pIC0+IGY2NCB7CiAgICB2YWx1ZXMuc29ydF9ieShmNjQ6OnRvdGFsX2NtcCk7CiAgICB2YWx1ZXNbdmFsdWVzLmxlbigpIC8gMl0KfQoKZm4gcThfd2VpZ2h0cyhvdXRfZGltOiB1c2l6ZSwgaW5fZGltOiB1c2l6ZSkgLT4gKFZlYzx1OD4sIFZlYzx1OD4pIHsKICAgIGxldCBxcyA9ICgwLi5vdXRfZGltICogaW5fZGltKQogICAgICAgIC5tYXAofGl8ICgoaS53cmFwcGluZ19tdWwoMjkpLndyYXBwaW5nX2FkZChpIC8gOTcpKSAlIDI1NSkgYXMgdTgpCiAgICAgICAgLmNvbGxlY3QoKTsKICAgIGxldCBzY2FsZXMgPSAoMC4ub3V0X2RpbSAqIChpbl9kaW0gLyAzMikpCiAgICAgICAgLmZsYXRfbWFwKHxffCBbMHgwMCwgMHgyMF0pCiAgICAgICAgLmNvbGxlY3QoKTsKICAgIChxcywgc2NhbGVzKQp9CgpmbiBhY3RpdmF0aW9ucyhyb3dzOiB1c2l6ZSwgaW5fZGltOiB1c2l6ZSkgLT4gVmVjPGYzMj4gewogICAgKDAuLnJvd3MgKiBpbl9kaW0pCiAgICAgICAgLm1hcCh8aXwgKChpLndyYXBwaW5nX211bCgzNykgJSAyNTEpIGFzIGYzMiAtIDEyNS4wKSAvIDEyNy4wKQogICAgICAgIC5jb2xsZWN0KCkKfQoKc3RydWN0IFJlY29yZCB7CiAgICBsYWJlbDogJidzdGF0aWMgc3RyLAogICAgb3V0X2RpbTogdXNpemUsCiAgICBpbl9kaW06IHVzaXplLAogICAgbnRvazogdXNpemUsCiAgICByZXRhaW5lZF91czogZjY0LAogICAgY2FuZGlkYXRlX3VzOiBmNjQsCiAgICBiaXRfZXhhY3Q6IGJvb2wsCiAgICBmaXJzdF9taXNtYXRjaDogaTY0LAp9CgppbXBsIFJlY29yZCB7CiAgICBmbiBqc29uKCZzZWxmKSAtPiBTdHJpbmcgewogICAgICAgIGZvcm1hdCEoCiAgICAgICAgICAgICJ7e1wic2hhcGVcIjpcInt9XCIsXCJvdXRfZGltXCI6e30sXCJpbl9kaW1cIjp7fSxcIm50b2tcIjp7fSxcCiAgICAgICAgICAgICBcInJldGFpbmVkX24xNl9tMzJfdXNcIjp7Oi4zfSxcImNhbmRpZGF0ZV9uMzJfbTMyX3VzXCI6ezouM30sXAogICAgICAgICAgICAgXCJzcGVlZHVwXCI6ezouNH0sXCJiaXRfZXhhY3RcIjp7fSxcImZpcnN0X21pc21hdGNoXCI6e319fSIsCiAgICAgICAgICAgIHNlbGYubGFiZWwsCiAgICAgICAgICAgIHNlbGYub3V0X2RpbSwKICAgICAgICAgICAgc2VsZi5pbl9kaW0sCiAgICAgICAgICAgIHNlbGYubnRvaywKICAgICAgICAgICAgc2VsZi5yZXRhaW5lZF91cywKICAgICAgICAgICAgc2VsZi5jYW5kaWRhdGVfdXMsCiAgICAgICAgICAgIHNlbGYucmV0YWluZWRfdXMgLyBzZWxmLmNhbmRpZGF0ZV91cywKICAgICAgICAgICAgc2VsZi5iaXRfZXhhY3QsCiAgICAgICAgICAgIHNlbGYuZmlyc3RfbWlzbWF0Y2gsCiAgICAgICAgKQogICAgfQp9CgpmbiBzY3JlZW4oCiAgICBjdWRhOiAmQ3VkYSwKICAgIGtlcm5lbHM6ICZLZXJuZWxTZXQsCiAgICBsYWJlbDogJidzdGF0aWMgc3RyLAogICAgb3V0X2RpbTogdXNpemUsCiAgICBpbl9kaW06IHVzaXplLAogICAgbnRvazogdXNpemUsCikgLT4gUmVzdWx0PFJlY29yZCwgQm94PGR5biBzdGQ6OmVycm9yOjpFcnJvcj4+IHsKICAgIGlmICFrZXJuZWxzLmdlbW1fbjE2X3VzZXNfbTMyKG91dF9kaW0gYXMgdTMyLCBudG9rIGFzIHUzMikgewogICAgICAgIHJldHVybiBFcnIoZm9ybWF0ISgie2xhYmVsfSBkaWQgbm90IHNlbGVjdCB0aGUgcmV0YWluZWQgTjE2L00zMiBjb250cm9sIikuaW50bygpKTsKICAgIH0KICAgIGxldCBudG9rX3BhZCA9IG50b2suZGl2X2NlaWwoOCkgKiA4OwogICAgbGV0IG5iID0gaW5fZGltIC8gMzI7CiAgICBsZXQgKHFzLCBzY2FsZXMpID0gcThfd2VpZ2h0cyhvdXRfZGltLCBpbl9kaW0pOwogICAgbGV0ICh0aWxlZF9xcywgdGlsZWRfc2NhbGVzKSA9IHE4XzBfc29hX3RvX2JzdGFnZSgmcXMsICZzY2FsZXMsIG91dF9kaW0sIGluX2RpbSk/OwogICAgbGV0IHggPSBhY3RpdmF0aW9ucyhudG9rX3BhZCwgaW5fZGltKTsKICAgIGxldCBieXRlcyA9ICh0aWxlZF9xcy5sZW4oKQogICAgICAgICsgdGlsZWRfc2NhbGVzLmxlbigpCiAgICAgICAgKyB4LmxlbigpICogNAogICAgICAgICsgbnRva19wYWQgKiBpbl9kaW0KICAgICAgICArIG50b2tfcGFkICogbmIgKiA0CiAgICAgICAgKyAyICogbnRvayAqIG91dF9kaW0gKiA0CiAgICAgICAgKyAyICogMV8wNDhfNTc2KSBhcyB1NjQ7CiAgICBsZXQgbXV0IGJ1ZiA9IEJhY2tlbmRCdWZmZXI6Om5ldyhjdWRhLCBieXRlcyk/OwogICAgbGV0IHdxcyA9IGJ1Zi5hbGxvYyh0aWxlZF9xcy5sZW4oKSBhcyB1NjQpPy5kcHRyOwogICAgbGV0IHdzYyA9IGJ1Zi5hbGxvYyh0aWxlZF9zY2FsZXMubGVuKCkgYXMgdTY0KT8uZHB0cjsKICAgIGN1ZGEuaHRvZCh3cXMsICZ0aWxlZF9xcyk/OwogICAgY3VkYS5odG9kKHdzYywgJnRpbGVkX3NjYWxlcyk/OwogICAgbGV0IHhmID0gYnVmLmFsbG9jX2YzMih4LmxlbigpKT8uZHB0cjsKICAgIGN1ZGEuaHRvZF9mMzIoeGYsICZ4KT87CiAgICBsZXQgeHFzID0gYnVmLmFsbG9jKChudG9rX3BhZCAqIGluX2RpbSkgYXMgdTY0KT8uZHB0cjsKICAgIGxldCB4c2MgPSBidWYuYWxsb2NfZjMyKG50b2tfcGFkICogbmIpPy5kcHRyOwogICAga2VybmVscy5xdWFudGl6ZV9xOChjdWRhLCB4ZiwgeHFzLCB4c2MsIChudG9rX3BhZCAqIGluX2RpbSkgYXMgdTMyKT87CiAgICBsZXQgcmV0YWluZWRfb3V0ID0gYnVmLmFsbG9jX2YzMihudG9rICogb3V0X2RpbSk/LmRwdHI7CiAgICBsZXQgY2FuZGlkYXRlX291dCA9IGJ1Zi5hbGxvY19mMzIobnRvayAqIG91dF9kaW0pPy5kcHRyOwoKICAgIGxldCByZXRhaW5lZCA9IHxvdXR8IHsKICAgICAgICBrZXJuZWxzLmdlbW1fbW1hX3E4X2JzdGFnZV9uMTYoCiAgICAgICAgICAgIGN1ZGEsCiAgICAgICAgICAgIHdxcywKICAgICAgICAgICAgd3NjLAogICAgICAgICAgICB4cXMsCiAgICAgICAgICAgIHhzYywKICAgICAgICAgICAgb3V0LAogICAgICAgICAgICBvdXRfZGltIGFzIHUzMiwKICAgICAgICAgICAgaW5fZGltIGFzIHUzMiwKICAgICAgICAgICAgbnRvayBhcyB1MzIsCiAgICAgICAgKQogICAgfTsKICAgIGxldCBjYW5kaWRhdGUgPSB8b3V0fCB7CiAgICAgICAga2VybmVscy5nZW1tX21tYV9xOF9ic3RhZ2VfbjMyX20zMigKICAgICAgICAgICAgY3VkYSwKICAgICAgICAgICAgd3FzLAogICAgICAgICAgICB3c2MsCiAgICAgICAgICAgIHhxcywKICAgICAgICAgICAgeHNjLAogICAgICAgICAgICBvdXQsCiAgICAgICAgICAgIG91dF9kaW0gYXMgdTMyLAogICAgICAgICAgICBpbl9kaW0gYXMgdTMyLAogICAgICAgICAgICBudG9rIGFzIHUzMiwKICAgICAgICApCiAgICB9OwoKICAgIHJldGFpbmVkKHJldGFpbmVkX291dCk/OwogICAgY2FuZGlkYXRlKGNhbmRpZGF0ZV9vdXQpPzsKICAgIGN1ZGEuc3luY2hyb25pemUoKT87CiAgICBsZXQgbXV0IGhvc3RfcmV0YWluZWQgPSB2ZWMhWzBmMzI7IG50b2sgKiBvdXRfZGltXTsKICAgIGxldCBtdXQgaG9zdF9jYW5kaWRhdGUgPSB2ZWMhWzBmMzI7IG50b2sgKiBvdXRfZGltXTsKICAgIGN1ZGEuZHRvaF9mMzIoJm11dCBob3N0X3JldGFpbmVkLCByZXRhaW5lZF9vdXQpPzsKICAgIGN1ZGEuZHRvaF9mMzIoJm11dCBob3N0X2NhbmRpZGF0ZSwgY2FuZGlkYXRlX291dCk/OwogICAgbGV0IGZpcnN0X21pc21hdGNoID0gaG9zdF9yZXRhaW5lZAogICAgICAgIC5pdGVyKCkKICAgICAgICAuemlwKCZob3N0X2NhbmRpZGF0ZSkKICAgICAgICAucG9zaXRpb24ofChhLCBiKXwgYS50b19iaXRzKCkgIT0gYi50b19iaXRzKCkpCiAgICAgICAgLm1hcF9vcigtMSwgfGl8IGkgYXMgaTY0KTsKCiAgICBsZXQgKG11dCByZXRhaW5lZF9zYW1wbGVzLCBtdXQgY2FuZGlkYXRlX3NhbXBsZXMpID0gKAogICAgICAgIFZlYzo6d2l0aF9jYXBhY2l0eSgyICogUkVQRUFUUyksCiAgICAgICAgVmVjOjp3aXRoX2NhcGFjaXR5KDIgKiBSRVBFQVRTKSwKICAgICk7CiAgICBmb3IgXyBpbiAwLi5SRVBFQVRTIHsKICAgICAgICByZXRhaW5lZF9zYW1wbGVzLnB1c2godGltZWQoY3VkYSwgfHwgcmV0YWluZWQocmV0YWluZWRfb3V0KSk/KTsKICAgICAgICBjYW5kaWRhdGVfc2FtcGxlcy5wdXNoKHRpbWVkKGN1ZGEsIHx8IGNhbmRpZGF0ZShjYW5kaWRhdGVfb3V0KSk/KTsKICAgICAgICBjYW5kaWRhdGVfc2FtcGxlcy5wdXNoKHRpbWVkKGN1ZGEsIHx8IGNhbmRpZGF0ZShjYW5kaWRhdGVfb3V0KSk/KTsKICAgICAgICByZXRhaW5lZF9zYW1wbGVzLnB1c2godGltZWQoY3VkYSwgfHwgcmV0YWluZWQocmV0YWluZWRfb3V0KSk/KTsKICAgIH0KICAgIGJ1Zi5mcmVlKGN1ZGEpPzsKICAgIE9rKFJlY29yZCB7CiAgICAgICAgbGFiZWwsCiAgICAgICAgb3V0X2RpbSwKICAgICAgICBpbl9kaW0sCiAgICAgICAgbnRvaywKICAgICAgICByZXRhaW5lZF91czogbWVkaWFuKCZtdXQgcmV0YWluZWRfc2FtcGxlcyksCiAgICAgICAgY2FuZGlkYXRlX3VzOiBtZWRpYW4oJm11dCBjYW5kaWRhdGVfc2FtcGxlcyksCiAgICAgICAgYml0X2V4YWN0OiBmaXJzdF9taXNtYXRjaCA8IDAsCiAgICAgICAgZmlyc3RfbWlzbWF0Y2gsCiAgICB9KQp9CgpmbiBtYWluKCkgLT4gUmVzdWx0PCgpLCBCb3g8ZHluIHN0ZDo6ZXJyb3I6OkVycm9yPj4gewogICAgaWYgIWN1ZGFfYXZhaWxhYmxlKCkgewogICAgICAgIHByaW50bG4hKCJbd2F2ZTU5LW4zMl0gbm8gQ1VEQSBkZXZpY2U7IG5vdGhpbmcgbWVhc3VyZWQiKTsKICAgICAgICByZXR1cm4gT2soKCkpOwogICAgfQogICAgbGV0IGN1ZGEgPSBDdWRhOjpwcm9iZSgpPzsKICAgIGlmIChjdWRhLmluZm8uc21fbWFqb3IsIGN1ZGEuaW5mby5zbV9taW5vcikgIT0gKDcsIDUpIHsKICAgICAgICByZXR1cm4gRXJyKCJXYXZlIDU5IHByb2R1Y3Rpb24gZ2F0ZSBpcyBwaW5uZWQgdG8gc21fNzUiLmludG8oKSk7CiAgICB9CiAgICBsZXQga2VybmVscyA9IEtlcm5lbFNldDo6bG9hZCgmY3VkYSk/OwogICAgaWYgIWtlcm5lbHMuZ2VtbV9uMzJfZW5hYmxlZCgpIHsKICAgICAgICByZXR1cm4gRXJyKCJzZXQgR0xDVURBX0dSSUQyRD0xIEdMQ1VEQV9OVElMRTEyOD0xIEdMQ1VEQV9CU1RBR0U9MSBcCiAgICAgICAgICAgICBHTENVREFfR0VNTV9OMTY9MSBHTENVREFfR0VNTV9OMzI9MSIKICAgICAgICAgICAgLmludG8oKSk7CiAgICB9CiAgICBsZXQgcmV0YWluZWRfcmVzb3VyY2UgPSBrZXJuZWxzCiAgICAgICAgLndhdmUyN19uMTZfbTMyX3Jlc291cmNlX2tlcm5lbCgpCiAgICAgICAgLm9rX29yKCJyZXRhaW5lZCBOMTYvTTMyIHJlc291cmNlIGtlcm5lbCB1bmF2YWlsYWJsZSIpPzsKICAgIGxldCBjYW5kaWRhdGVfcmVzb3VyY2UgPSBrZXJuZWxzCiAgICAgICAgLndhdmU1OV9uMzJfbTMyX3Jlc291cmNlX2tlcm5lbCgpCiAgICAgICAgLm9rX29yKCJXYXZlIDU5IHJlc291cmNlIGtlcm5lbCB1bmF2YWlsYWJsZSIpPzsKICAgIGxldCByZXRhaW5lZF9hY3RpdmUgPSBjdWRhCiAgICAgICAgLm1heF9hY3RpdmVfYmxvY2tzX3Blcl9zbShyZXRhaW5lZF9yZXNvdXJjZSwgMjU2LCAwKQogICAgICAgIC5va19vcigicmV0YWluZWQgb2NjdXBhbmN5IHF1ZXJ5IHVuYXZhaWxhYmxlIik/OwogICAgbGV0IGNhbmRpZGF0ZV9hY3RpdmUgPSBjdWRhCiAgICAgICAgLm1heF9hY3RpdmVfYmxvY2tzX3Blcl9zbShjYW5kaWRhdGVfcmVzb3VyY2UsIDEyOCwgMCkKICAgICAgICAub2tfb3IoIldhdmUgNTkgb2NjdXBhbmN5IHF1ZXJ5IHVuYXZhaWxhYmxlIik/OwogICAgaWYgY2FuZGlkYXRlX2FjdGl2ZSA8IDYgewogICAgICAgIHJldHVybiBFcnIoZm9ybWF0ISgKICAgICAgICAgICAgIldhdmUgNTkgb2NjdXBhbmN5IHRpZXIgZmFpbGVkOiBjYW5kaWRhdGVfYWN0aXZlX2Jsb2Nrc19wZXJfc209e2NhbmRpZGF0ZV9hY3RpdmV9IgogICAgICAgICkKICAgICAgICAuaW50bygpKTsKICAgIH0KCiAgICBsZXQgcmVjb3JkcyA9IFsKICAgICAgICBzY3JlZW4oJmN1ZGEsICZrZXJuZWxzLCAicWt2IiwgMV8xNTIsIDg5NiwgMjQ0KT8sCiAgICAgICAgc2NyZWVuKCZjdWRhLCAma2VybmVscywgImZmbl9kb3duIiwgODk2LCA0Xzg2NCwgMjQ0KT8sCiAgICAgICAgc2NyZWVuKCZjdWRhLCAma2VybmVscywgInJhZ2dlZCIsIDEzNiwgMTYwLCAxNyk/LAogICAgXTsKICAgIGxldCBhbGxfZXhhY3QgPSByZWNvcmRzLml0ZXIoKS5hbGwofHJlY29yZHwgcmVjb3JkLmJpdF9leGFjdCk7CiAgICBwcmludGxuISgKICAgICAgICAiW3dhdmU1OS1yZXNvdXJjZV0ge3tcInJldGFpbmVkX3RocmVhZHNcIjoyNTYsXCJyZXRhaW5lZF9hY3RpdmVfYmxvY2tzX3Blcl9zbVwiOnt9LFwKICAgICAgICAgXCJjYW5kaWRhdGVfdGhyZWFkc1wiOjEyOCxcImNhbmRpZGF0ZV9hY3RpdmVfYmxvY2tzX3Blcl9zbVwiOnt9LFwKICAgICAgICAgXCJjYW5kaWRhdGVfc3RhdGljX3NoYXJlZF9ieXRlc1wiOjgwNjQsXCJjYW5kaWRhdGVfbWF4bnJlZ1wiOjgwfX0iLAogICAgICAgIHJldGFpbmVkX2FjdGl2ZSwgY2FuZGlkYXRlX2FjdGl2ZQogICAgKTsKICAgIHByaW50bG4hKAogICAgICAgICJbd2F2ZTU5LW4zMl0ge3tcIndhcm11cFwiOnt9LFwiaXRlcnNcIjp7fSxcInJlcGVhdHNcIjp7fSxcInJlY29yZHNcIjpbe31dLFwKICAgICAgICAgXCJhbGxfYml0X2V4YWN0XCI6e319fSIsCiAgICAgICAgV0FSTVVQLAogICAgICAgIElURVJTLAogICAgICAgIFJFUEVBVFMsCiAgICAgICAgcmVjb3JkcwogICAgICAgICAgICAuaXRlcigpCiAgICAgICAgICAgIC5tYXAoUmVjb3JkOjpqc29uKQogICAgICAgICAgICAuY29sbGVjdDo6PFZlYzxfPj4oKQogICAgICAgICAgICAuam9pbigiLCIpLAogICAgICAgIGFsbF9leGFjdCwKICAgICk7CiAgICBpZiAhYWxsX2V4YWN0IHsKICAgICAgICByZXR1cm4gRXJyKCJXYXZlIDU5IGRpcmVjdCBiaXQtZXhhY3QgZ2F0ZSBmYWlsZWQiLmludG8oKSk7CiAgICB9CiAgICBPaygoKSkKfQo='}}
    CANDIDATE_SNAPSHOT_MANIFEST = {}
    for relative, metadata in candidate_snapshot.items():
        data = base64.b64decode(metadata["base64"])
        got = hashlib.sha256(data).hexdigest()
        if got != metadata["sha256"]:
            raise RuntimeError(
                f"candidate snapshot hash mismatch for {relative}: {got} != {metadata['sha256']}"
            )
        destination = TREE / relative
        destination.parent.mkdir(parents=True, exist_ok=True)
        destination.write_bytes(data)
        written = sha256_file(destination)
        if written != metadata["sha256"]:
            raise RuntimeError(
                f"candidate snapshot write mismatch for {relative}: {written}"
            )
        CANDIDATE_SNAPSHOT_MANIFEST[relative] = {
            "sha256": written,
            "bytes": len(data),
        }
        stack_log.append({
            "cmd": ["snapshot-write", relative],
            "returncode": 0,
            "sha256": written,
            "bytes": len(data),
        })
    (RESULTS / "candidate-snapshot.json").write_text(
        json.dumps(CANDIDATE_SNAPSHOT_MANIFEST, indent=2), encoding="utf-8"
    )

    p = run(["git", "diff", "--check"], cwd=TREE, check=False)
    stack_log.append({"cmd": ["git", "diff", "--check"], "returncode": p.returncode,
                      "stdout": p.stdout, "stderr": p.stderr})
    (RESULTS / "git-stack.json").write_text(json.dumps(stack_log, indent=2), encoding="utf-8")
    if p.returncode:
        raise RuntimeError("reconstructed stack fails git diff --check")

    sm75_path = TREE / "glcuda/src/kernels/glcuda_sm75.ptx"
    sm75_ptx = sm75_path.read_text(encoding="ascii")
    kernels_rs = (TREE / "glcuda/src/kernels/mod.rs").read_text(encoding="utf-8")
    attention_rs = (TREE / "glcuda/src/attention/mod.rs").read_text(encoding="utf-8")
    parity_rs = (TREE / "glcuda/tests/parity.rs").read_text(encoding="utf-8")
    example_rs = (TREE / "glcuda/examples/wave48_regq_attention.rs").read_text(encoding="utf-8")
    retained_region = ptx_region(sm75_ptx, "gl_attn_mma4_fused_f32")
    candidate_region = ptx_region(sm75_ptx, "gl_attn_mma4_regq_fused_f32")
    n16_region = ptx_region(sm75_ptx, "gl_gemm_mma_q8_bstage_n16")
    n16_m32_region = ptx_region(sm75_ptx, "gl_gemm_mma_q8_bstage_n16_m32")
    STRUCTURAL = {
        "sm75_ascii_lf": "\r" not in sm75_ptx and "\0" not in sm75_ptx and sm75_ptx.isascii(),
        "retained_mma4_contract":
            retained_region.count("mma.sync.aligned.m16n8k8") == 4 and
            retained_region.count("bar.sync 0;") == 2 and
            "wave20_q_smem[4096]" in retained_region,
        "candidate_four_product_unroll":
            candidate_region.count("mma.sync.aligned.m16n8k8") == 32 and
            "W48_K_CHUNK:" not in candidate_region,
        "candidate_register_snapshot":
            candidate_region.count("ld.shared.u32 %qa_") == 32 and
            "%qa_hi00, %qa_hi01" in candidate_region and
            "%qa_hi0<8>" not in candidate_region and
            candidate_region.count("bar.sync 0;") == 3,
        "candidate_aliases_dynamic_scores":
            candidate_region.count("mov.u32 %r14, sm_wave20_scores;") == 1 and
            candidate_region.count("mov.u32 %r15, sm_wave20_scores;") == 1 and
            "wave20_q_smem[4096]" not in candidate_region,
        "opt_in_dispatch": all(x in kernels_rs + attention_rs for x in [
            "GLCUDA_ATTN_MMA4_REGQ", "Mma4RegQ", "attn_mma4_regq_fused",
            "mma4_regq_attention_capacity_supported",
        ]),
        "direct_gate_contract": all(x in example_rs for x in [
            "MIN_DIRECT_SPEEDUP: f64 = 1.10", "retained_blocks != 3",
            "candidate_blocks < 4", "bit-exact gate failed",
        ]),
        "parity_bit_exact": all(x in parity_rs for x in [
            "attn_mma4_regq_fused", "assert_bits_eq", "Wave48 register-Q vs Wave20",
        ]),
        "n16_baseline_contract":
            n16_region.count("mma.sync.aligned") == 32 and
            n16_m32_region.count("mma.sync.aligned") == 16 and
            n16_region.count(".maxnreg 72") == 1 and
            n16_m32_region.count(".maxnreg 72") == 1,
    }
    if not all(STRUCTURAL.values()):
        raise RuntimeError(f"Wave 50 structural gate failed: {STRUCTURAL}")

    PTXAS = shutil.which("ptxas") or "/usr/local/cuda/bin/ptxas"
    if not Path(PTXAS).is_file():
        raise RuntimeError(f"ptxas not found: {PTXAS}")
    p_sm75 = run([PTXAS, "-v", "-arch=sm_75", sm75_path,
                  "-o", ROOT / "wave50-sm75.cubin"], cwd=TREE,
                 timeout=1800, check=False)
    save_log("ptxas-sm75-v.log", p_sm75)
    if p_sm75.returncode:
        raise RuntimeError("ptxas rejected the Wave 50 sm_75 module")

    def ptxas_resource(log, entry):
        marker = re.search(r"Compiling entry function ['\"]" + re.escape(entry) +
                           r"['\"].*?(?=Compiling entry function|\Z)", log, re.S)
        segment = marker.group(0) if marker else ""
        regs = re.search(r"Used (\d+) registers", segment)
        smem = re.search(r"(\d+) bytes smem", segment)
        spill_stores = [int(x) for x in re.findall(r"(\d+) bytes spill stores", segment)]
        spill_loads = [int(x) for x in re.findall(r"(\d+) bytes spill loads", segment)]
        stack = [int(x) for x in re.findall(r"(\d+) bytes stack frame", segment)]
        return {
            "entry": entry, "found": bool(marker),
            "registers": int(regs.group(1)) if regs else None,
            "static_shared_bytes": int(smem.group(1)) if smem else 0,
            "spill_store_bytes": max(spill_stores, default=0),
            "spill_load_bytes": max(spill_loads, default=0),
            "stack_frame_bytes": max(stack, default=0),
        }

    ptxas_log = p_sm75.stdout + "\n" + p_sm75.stderr
    PTXAS_RESOURCES = {
        "mma4_retained": ptxas_resource(ptxas_log, "gl_attn_mma4_fused_f32"),
        "mma4_regq": ptxas_resource(ptxas_log, "gl_attn_mma4_regq_fused_f32"),
        "n16_bstage": ptxas_resource(ptxas_log, "gl_gemm_mma_q8_bstage_n16"),
        "n16_m32": ptxas_resource(ptxas_log, "gl_gemm_mma_q8_bstage_n16_m32"),
    }
    if not all(x["found"] for x in PTXAS_RESOURCES.values()):
        raise RuntimeError(f"entry missing from ptxas report: {PTXAS_RESOURCES}")
    if any(x["spill_store_bytes"] or x["spill_load_bytes"] or x["stack_frame_bytes"]
           for x in PTXAS_RESOURCES.values()):
        raise RuntimeError(f"Wave 50 kernel spills/stacks: {PTXAS_RESOURCES}")
    if PTXAS_RESOURCES["mma4_retained"]["static_shared_bytes"] != 4_096:
        raise RuntimeError(f"retained MMA4 resource drift: {PTXAS_RESOURCES}")
    if (PTXAS_RESOURCES["mma4_regq"]["static_shared_bytes"] != 0 or
            PTXAS_RESOURCES["mma4_regq"]["registers"] > 128):
        raise RuntimeError(f"Wave 50 resource gate failed: {PTXAS_RESOURCES}")
    for name in ("n16_bstage", "n16_m32"):
        if (PTXAS_RESOURCES[name]["static_shared_bytes"] != 9_728 or
                PTXAS_RESOURCES[name]["registers"] > 72):
            raise RuntimeError(f"N16 resource drift: {PTXAS_RESOURCES}")

    common_build_env = {"CARGO_TARGET_DIR": str(TARGET), "CUDA_VISIBLE_DEVICES": "0"}
    candidate_env = {
        **common_build_env,
        "GLCUDA_GRID2D": "1", "GLCUDA_BSTAGE": "1", "GLCUDA_NTILE128": "1",
        "GLCUDA_GEMM_N16": "1", "GLCUDA_ATTN_MMA4": "1",
        "GLCUDA_ATTN_MMA4_REGQ": "1",
    }
    p = run([CARGO, "test", "-p", "glcuda", "--lib", "--locked"],
            cwd=TREE, env=common_build_env, timeout=3600, check=False)
    save_log("cargo-lib-tests.log", p)
    LIB_TESTS = next((line for line in (p.stdout + p.stderr).splitlines()
                      if line.startswith("test result:")), "")
    if p.returncode or "64 passed" not in LIB_TESTS or "0 failed" not in LIB_TESTS:
        raise RuntimeError(f"unexpected Wave 50 host-test result: {LIB_TESTS}")

    parity_env = {**candidate_env}
    parity_env.pop("GLCUDA_ATTN_MMA4", None)
    parity_env.pop("GLCUDA_ATTN_MMA4_REGQ", None)
    if any(key in parity_env for key in ("GLCUDA_ATTN_MMA4", "GLCUDA_ATTN_MMA4_REGQ")):
        raise RuntimeError(f"Wave 50 parity environment leaked attention flags: {parity_env}")
    p = run([CARGO, "test", "--release", "-p", "glcuda", "--test", "parity", "--locked",
             "--", "--nocapture", "--test-threads=1"],
            cwd=TREE, env=parity_env, timeout=7200, check=False)
    save_log("cargo-test-parity.log", p)
    PARITY = {
        "returncode": p.returncode,
        "summary": next((line for line in p.stdout.splitlines()
                         if line.startswith("test result:")), ""),
    }
    if p.returncode or "33 passed" not in PARITY["summary"] or "0 failed" not in PARITY["summary"]:
        raise RuntimeError(f"Wave 50 full device parity gate failed: {PARITY}")

    p = run([CARGO, "run", "--release", "-p", "glcuda", "--example",
             "wave48_regq_attention", "--locked"], cwd=TREE, env=candidate_env,
            timeout=7200, check=False)
    save_log("wave50-direct-exact.log", p)
    direct_line = next((x for x in p.stdout.splitlines()
                        if x.startswith("[wave48-direct] ")), "")
    DIRECT_DIAGNOSTIC = json.loads(direct_line.split("] ", 1)[1]) if direct_line else {}
    DIRECT_RESOURCE = {
        "retained": PTXAS_RESOURCES["mma4_retained"],
        "candidate": PTXAS_RESOURCES["mma4_regq"],
        "retained_blocks_per_sm": DIRECT_DIAGNOSTIC.get("retained_blocks_per_sm"),
        "candidate_blocks_per_sm": DIRECT_DIAGNOSTIC.get("candidate_blocks_per_sm"),
    }
    if (p.returncode or
            DIRECT_DIAGNOSTIC.get("bit_exact") is not True or
            DIRECT_DIAGNOSTIC.get("retained_dynamic_shared") != 15_616 or
            DIRECT_DIAGNOSTIC.get("candidate_dynamic_shared") != 15_616 or
            DIRECT_DIAGNOSTIC.get("retained_blocks_per_sm") != 3 or
            int(DIRECT_DIAGNOSTIC.get("candidate_blocks_per_sm", 0)) < 4 or
            not math.isfinite(float(DIRECT_DIAGNOSTIC.get("retained_us", 0))) or
            not math.isfinite(float(DIRECT_DIAGNOSTIC.get("candidate_us", 0))) or
            float(DIRECT_DIAGNOSTIC.get("retained_us", 0)) <= 0 or
            float(DIRECT_DIAGNOSTIC.get("candidate_us", 0)) <= 0 or
            float(DIRECT_DIAGNOSTIC.get("speedup", 0)) < DIRECT_HARD_STOP):
        raise RuntimeError(
            f"Wave 50 direct gate failed: resource={DIRECT_RESOURCE} "
            f"diagnostic={DIRECT_DIAGNOSTIC} stderr={p.stderr[-4000:]}"
        )

    p = run([CARGO, "build", "--release", "-p", "glbench", "--locked"],
            cwd=TREE, env=common_build_env, timeout=7200, check=False)
    save_log("cargo-build-glbench.log", p)
    if p.returncode:
        raise RuntimeError("Wave 28 glbench release build failed")
    GLBENCH = str(TARGET / "release/glbench")

    STACK_OK = True
    MODEL_OK = False
    print(json.dumps({
        "gpu": GPU_INFO, "structural": STRUCTURAL,
        "patch_sha256": WAVE50_PATCH_SHA256,
        "ptxas": PTXAS_RESOURCES, "driver_resource": DIRECT_RESOURCE,
        "direct_exact": DIRECT_DIAGNOSTIC,
        "parity": PARITY, "lib_tests": LIB_TESTS,
    }, indent=2))

try:
    compile_phase()
except Exception:
    fail_phase("compile-resource-correctness")


def wave59_resource(log, entry):
    marker = re.search(r"Compiling entry function ['\"]" + re.escape(entry) +
                       r"['\"].*?(?=Compiling entry function|\Z)", log, re.S)
    segment = marker.group(0) if marker else ""
    def largest(pattern):
        return max((int(x) for x in re.findall(pattern, segment)), default=0)
    registers = re.search(r"Used (\d+) registers", segment)
    shared = re.search(r"(\d+) bytes smem", segment)
    return {
        "entry": entry,
        "found": bool(marker),
        "registers": int(registers.group(1)) if registers else None,
        "static_shared_bytes": int(shared.group(1)) if shared else 0,
        "spill_store_bytes": largest(r"(\d+) bytes spill stores"),
        "spill_load_bytes": largest(r"(\d+) bytes spill loads"),
        "stack_frame_bytes": largest(r"(\d+) bytes stack frame"),
    }

try:
    if not globals().get("STACK_OK"):
        raise RuntimeError("retained production reconstruction did not pass")
    wave59_path = TREE / "glcuda/src/kernels/glcuda_sm75_wave59.ptx"
    wave59_source = wave59_path.read_text(encoding="ascii")
    WAVE59_STRUCTURAL = {
        "ascii_lf": wave59_source.isascii() and "\r" not in wave59_source and "\0" not in wave59_source,
        "one_entry": wave59_source.count(".visible .entry") == 1,
        "mma_32": wave59_source.count("mma.sync.aligned.m8n8k16") == 32,
        "a_ldmatrix_4": wave59_source.count("ldmatrix.sync.aligned.x2") == 4,
        "b_ldmatrix_2": wave59_source.count("ldmatrix.sync.aligned.x4") == 2,
        "barriers_2": wave59_source.count("bar.sync 0;") == 2,
        "shared_8064": all(x in wave59_source for x in [
            "sm_a[1536]", "sm_xs[128]", "sm_b[6144]", "sm_bs[256]",
        ]),
        "own_ptx": "wmma." not in wave59_source and "cutlass" not in wave59_source.lower(),
    }
    if not all(WAVE59_STRUCTURAL.values()):
        raise RuntimeError(f"Wave 62 structural gate failed: {WAVE59_STRUCTURAL}")

    p = run([PTXAS, "-v", "-arch=sm_75", wave59_path,
             "-o", ROOT / "wave59-n32.cubin"], cwd=TREE, timeout=1800, check=False)
    save_log("ptxas-wave59-v.log", p)
    if p.returncode:
        raise RuntimeError("ptxas rejected Wave 59 PTX")
    WAVE59_PTXAS = wave59_resource(
        p.stdout + "\n" + p.stderr, "gl_gemm_mma_q8_bstage_n32_m32"
    )
    if (not WAVE59_PTXAS["found"] or
            WAVE59_PTXAS["registers"] is None or
            WAVE59_PTXAS["registers"] > 80 or
            WAVE59_PTXAS["static_shared_bytes"] != 8_064 or
            WAVE59_PTXAS["spill_store_bytes"] or
            WAVE59_PTXAS["spill_load_bytes"] or
            WAVE59_PTXAS["stack_frame_bytes"]):
        raise RuntimeError(f"Wave 62 resource gate failed: {WAVE59_PTXAS}")

    wave59_env = {
        "CARGO_TARGET_DIR": str(TARGET), "CUDA_VISIBLE_DEVICES": "0",
        "GLCUDA_GRID2D": "1", "GLCUDA_NTILE128": "1", "GLCUDA_BSTAGE": "1",
        "GLCUDA_GEMM_N16": "1", "GLCUDA_GEMM_N32": "1",
    }
    p = run([CARGO, "test", "--release", "-p", "glcuda", "--test", "wave59_n32",
             "--locked", "--", "--nocapture", "--test-threads=1"],
            cwd=TREE, env=wave59_env, timeout=7200, check=False)
    save_log("wave59-device-parity.log", p)
    WAVE59_PARITY = next((line for line in p.stdout.splitlines()
                          if line.startswith("test result:")), "")
    if p.returncode or "1 passed" not in WAVE59_PARITY or "0 failed" not in WAVE59_PARITY:
        raise RuntimeError(f"Wave 62 device parity failed: {WAVE59_PARITY}")

    p = run([CARGO, "run", "--release", "-p", "glcuda", "--example",
             "wave59_n32_gemm", "--locked"], cwd=TREE, env=wave59_env,
            timeout=7200, check=False)
    save_log("wave59-direct.log", p)
    resource_line = next((line for line in p.stdout.splitlines()
                          if line.startswith("[wave59-resource] ")), "")
    direct_line = next((line for line in p.stdout.splitlines()
                        if line.startswith("[wave59-n32] ")), "")
    WAVE59_DRIVER_RESOURCE = json.loads(resource_line.split("] ", 1)[1]) if resource_line else {}
    WAVE59_DIRECT = json.loads(direct_line.split("] ", 1)[1]) if direct_line else {}
    production_records = [r for r in WAVE59_DIRECT.get("records", [])
                          if r.get("shape") in ("qkv", "ffn_down")]
    if (p.returncode or
            WAVE59_DIRECT.get("all_bit_exact") is not True or
            len(production_records) != 2 or
            int(WAVE59_DRIVER_RESOURCE.get("candidate_active_blocks_per_sm", 0)) < 6 or
            any(float(r.get("speedup", 0)) < 1.10 for r in production_records)):
        raise RuntimeError(
            f"Wave 62 direct gate failed: resource={WAVE59_DRIVER_RESOURCE} "
            f"direct={WAVE59_DIRECT} stderr={p.stderr[-4000:]}"
        )

    WAVE59_SUMMARY = {
        "notebook_build": NOTEBOOK_BUILD,
        "gpu": GPU_INFO,
        "candidate_snapshot": CANDIDATE_SNAPSHOT_MANIFEST,
        "structural": WAVE59_STRUCTURAL,
        "ptxas": WAVE59_PTXAS,
        "driver_resource": WAVE59_DRIVER_RESOURCE,
        "device_parity": WAVE59_PARITY,
        "direct": WAVE59_DIRECT,
        "direct_hard_stop": 1.10,
        "decision": "kernel_gates_passed",
    }
    (RESULTS / "wave62-kernel-screen.json").write_text(
        json.dumps(WAVE59_SUMMARY, indent=2), encoding="utf-8"
    )
    (RESULTS / "SCREEN_SUCCESS.json").write_text(json.dumps({
        "notebook_build": NOTEBOOK_BUILD,
        "decision": "kernel_gates_passed",
    }, indent=2), encoding="utf-8")
    archive()
    print(json.dumps(WAVE59_SUMMARY, indent=2))
except Exception:
    fail_phase("wave62-n32-screen")


## 2 - Fetch the pinned Q8 production model


In [ ]:
if not globals().get("STACK_OK"):
    raise RuntimeError("Bootstrap gate did not pass")

def fetch_pinned_model():
    model = WORK / HF_FILENAME
    part = WORK / f"{HF_FILENAME}.part"
    if model.is_file() and model.stat().st_size == HF_EXPECTED_BYTES and sha256_file(model) == HF_EXPECTED_SHA256:
        return model
    if model.exists():
        model.unlink()
    if part.exists():
        part_size = part.stat().st_size
        if part_size == HF_EXPECTED_BYTES:
            if sha256_file(part) == HF_EXPECTED_SHA256:
                part.replace(model)
                return model
            part.unlink()
        elif part_size > HF_EXPECTED_BYTES:
            part.unlink()
    url = f"https://huggingface.co/{HF_REPO}/resolve/{HF_REVISION}/{HF_FILENAME}?download=true"
    for attempt in range(1, 6):
        start = part.stat().st_size if part.exists() else 0
        headers = {"User-Agent": "GwenLand-glcuda-Wave50/1.0", "Accept-Encoding": "identity"}
        if start:
            headers["Range"] = f"bytes={start}-"
        request = urllib.request.Request(url, headers=headers)
        try:
            response = urllib.request.urlopen(request, timeout=120)
            status = getattr(response, "status", response.getcode())
            if start and status != 206:
                response.close()
                part.unlink(missing_ok=True)
                start = 0
                response = urllib.request.urlopen(
                    urllib.request.Request(url, headers={"User-Agent": "GwenLand-glcuda-Wave50/1.0", "Accept-Encoding": "identity"}),
                    timeout=120,
                )
                status = getattr(response, "status", response.getcode())
            if status not in (200, 206):
                raise RuntimeError(f"HTTP {status}")
            mode = "ab" if start and status == 206 else "wb"
            downloaded = start
            last_print = time.monotonic()
            with response, part.open(mode) as output:
                while True:
                    chunk = response.read(8 << 20)
                    if not chunk:
                        break
                    output.write(chunk)
                    downloaded += len(chunk)
                    if time.monotonic() - last_print >= 20:
                        print(f"model fetch {downloaded / (1 << 20):.1f}/{HF_EXPECTED_BYTES / (1 << 20):.1f} MiB")
                        last_print = time.monotonic()
            if part.stat().st_size != HF_EXPECTED_BYTES:
                raise RuntimeError(f"truncated model: {part.stat().st_size}/{HF_EXPECTED_BYTES}")
            digest = sha256_file(part)
            if digest != HF_EXPECTED_SHA256:
                part.unlink(missing_ok=True)
                raise RuntimeError(f"model SHA mismatch: {digest}")
            part.replace(model)
            return model
        except Exception as exc:
            print(f"fetch attempt {attempt}/5 failed: {exc}")
            if attempt == 5:
                raise
            time.sleep(min(30, 2 ** attempt))

try:
    MODEL_PATH = fetch_pinned_model()
    MODEL_META = {"repo": HF_REPO, "revision": HF_REVISION, "filename": HF_FILENAME, "bytes": MODEL_PATH.stat().st_size, "sha256": sha256_file(MODEL_PATH)}
    (RESULTS / "model.json").write_text(json.dumps(MODEL_META, indent=2), encoding="utf-8")
    MODEL_OK = True
    print(json.dumps(MODEL_META, indent=2))
except Exception:
    fail_phase("model-fetch")

## 3 - Ten-pair retained N16 vs candidate N32 production gate


In [ ]:
if not globals().get("MODEL_OK"):
    raise RuntimeError("Model gate did not pass")

PRODUCTION_REPEATS = 10
COLD_ITERS = 1
WARMUP_ITERS = 5
MEASURE_ITERS = 10
FIXED_PROMPT = (
    "Measure this deterministic systems prompt carefully. Explain how token-parallel "
    "integer matrix multiplication uses shared memory, Tensor Cores, and fixed launch geometry. "
) * 8
COMMON_ENV = {
    "CUDA_VISIBLE_DEVICES": "0",
    "GLCUDA_FORCE_Q8": "1",
    "GLCUDA_GRID2D": "1",
    "GLCUDA_FUSE_Q8_GLUE": "1",
    "GLCUDA_NTILE128": "1",
    "GLCUDA_BSTAGE": "1",
    "GLCUDA_GEMM_N16": "1",
    "GLCUDA_ATTN_MMA4": "1",
    "GLCUDA_ATTN_MMA4_REGQ": "1",
    "CARGO_TARGET_DIR": str(TARGET),
}
ARM_ENV = {
    "retained_n16": {},
    "candidate_n32": {"GLCUDA_GEMM_N32": "1"},
}
ORDERS = [
    ["retained_n16", "candidate_n32"] if repeat % 2 == 0
    else ["candidate_n32", "retained_n16"]
    for repeat in range(PRODUCTION_REPEATS)
]


def percentile(values, q):
    values = sorted(values)
    index = (len(values) - 1) * q
    lo, hi = math.floor(index), math.ceil(index)
    return values[lo] if lo == hi else values[lo] * (hi - index) + values[hi] * (index - lo)


def json_lines(hay, prefix):
    return [json.loads(x) for x in re.findall(re.escape(prefix) + r"\s*(\{[^\n]+\})", hay)]


def last_json_line(hay, prefix):
    matches = json_lines(hay, prefix)
    if not matches:
        raise RuntimeError(f"dispatch line missing: {prefix}")
    return matches[-1]


def check_dispatch(arm, hay):
    contract = last_json_line(hay, "[glcuda-contract]")
    expected_n32 = arm == "candidate_n32"
    required = {
        "exact_fusion": True,
        "grid2d": True,
        "ntile128": True,
        "bstage": True,
        "gemm_n16": True,
        "gemm_n32": expected_n32,
        "attn_mma4": True,
        "attn_mma4_regq": True,
    }
    bad = {key: (contract.get(key), value) for key, value in required.items()
           if contract.get(key) is not value}
    if bad:
        raise RuntimeError(f"{arm} contract mismatch: {bad}; full={contract}")
    attention = last_json_line(hay, "[glcuda-attn]")
    if attention.get("path") != "mma4-regq":
        raise RuntimeError(f"{arm} attention path drift: {attention}")
    gemm = json_lines(hay, "[glcuda-gemm]")
    paths = {row.get("path") for row in gemm}
    expected_paths = (
        {"bstage-n32-m32", "bstage-n16"}
        if expected_n32 else {"bstage-n16-m32", "bstage-n16"}
    )
    if paths != expected_paths or any(row.get("ntok") != 244 for row in gemm):
        raise RuntimeError(f"{arm} GEMM dispatch drift: {gemm}")
    return {"contract": contract, "attention": attention, "gemm": gemm}


def timing_rows(rows, expected, label):
    if len(rows) != expected:
        raise RuntimeError(f"{label} count {len(rows)} != {expected}")
    counts = [int(row.get("prompt_tokens", 0)) for row in rows]
    prefill = [float(row.get("prefill_ms", 0)) for row in rows]
    decode = [float(row.get("decode_ms", 0)) for row in rows]
    if len(set(counts)) != 1 or counts[0] != 244:
        raise RuntimeError(f"{label} prompt-token drift: {counts}")
    if not all(math.isfinite(value) and value > 0 for value in prefill + decode):
        raise RuntimeError(f"{label} malformed timings: {rows}")
    return counts[0], prefill, decode


def session_stats(path):
    data = json.loads(Path(path).read_text(encoding="utf-8"))
    engine = data.get("engine") or {}
    workload = data.get("workload") or {}
    if engine.get("name") != "glcuda" or engine.get("backend") != "cuda" or not engine.get("available"):
        raise RuntimeError(f"wrong engine contract: {engine}")
    expected = {
        "engine": "glcuda",
        "kind": "prefill",
        "prompt": FIXED_PROMPT,
        "seed": 42,
        "temperature": 0.0,
        "max_new_tokens": 1,
        "cold_iters": COLD_ITERS,
        "warmup_iters": WARMUP_ITERS,
        "measure_iters": MEASURE_ITERS,
        "verify_against": "glproc",
    }
    for key, value in expected.items():
        if workload.get(key) != value:
            raise RuntimeError(f"workload {key} mismatch: {workload.get(key)!r} != {value!r}")
    validation = data.get("validation") or {}
    if validation.get("passed") is not True:
        raise RuntimeError(f"session validation failed: {validation}")
    parity = [finding for finding in validation.get("findings", [])
              if finding.get("check") == "parity"]
    match = re.search(r"(\d+)/(\d+) tokens match oracle", parity[-1].get("message", "")) if parity else None
    if not match or match.group(1) != match.group(2) or int(match.group(2)) != 50:
        raise RuntimeError(f"exact 50/50 oracle failed: {parity[-1] if parity else None}")
    measurements = data.get("measurements") or {}
    count, prefill_ms, decode_ms = timing_rows(
        measurements.get("iterations") or [], MEASURE_ITERS, "measured"
    )
    _, cold_prefill_ms, cold_decode_ms = timing_rows(
        measurements.get("cold") or [], COLD_ITERS, "cold"
    )
    median_latency = statistics.median(prefill_ms)
    return {
        "prefill_p50": percentile([count * 1000.0 / value for value in prefill_ms], 0.5),
        "prefill_p90": percentile([count * 1000.0 / value for value in prefill_ms], 0.9),
        "prefill_p99": percentile([count * 1000.0 / value for value in prefill_ms], 0.99),
        "latency_p50_ms": percentile(prefill_ms, 0.5),
        "latency_p90_ms": percentile(prefill_ms, 0.9),
        "latency_p99_ms": percentile(prefill_ms, 0.99),
        "latency_mad_ms": statistics.median([abs(value - median_latency) for value in prefill_ms]),
        "latency_max_ms": max(prefill_ms),
        "decode_p50": percentile([1000.0 / value for value in decode_ms], 0.5),
        "cold_prefill_ms": cold_prefill_ms[0],
        "cold_decode_ms": cold_decode_ms[0],
        "samples_ms": [round(value, 3) for value in prefill_ms],
        "oracle": "50/50",
    }


def run_arm(arm, out, cold, warmup, iterations):
    env = {**COMMON_ENV, **ARM_ENV[arm]}
    command = [
        GLBENCH, "run", "--engine", "glcuda", "--model", str(MODEL_PATH),
        "--prompt", FIXED_PROMPT, "--tokens", "1", "--cold-iters", str(cold),
        "--warmup", str(warmup), "--iters", str(iterations), "--temperature", "0",
        "--seed", "42", "--kind", "prefill", "--verify-against", "glproc",
        "--out", str(out),
    ]
    return run(command, cwd=TREE, env=env, timeout=14400, check=False)


try:
    screen_success = RESULTS / "SCREEN_SUCCESS.json"
    if screen_success.exists():
        screen_success.unlink()

    DISPATCH = {}
    for arm in ARM_ENV:
        process = run_arm(arm, RESULTS / f"stabilize-{arm}.json", 0, 0, 1)
        save_log(f"stabilize-{arm}.log", process)
        if process.returncode:
            raise RuntimeError(f"stabilization failed for {arm}")
        DISPATCH[arm] = check_dispatch(arm, process.stdout + "\n" + process.stderr)

    PRODUCTION_RECORDS = []
    for repeat, order in enumerate(ORDERS):
        for position, arm in enumerate(order):
            out = RESULTS / f"glbench-r{repeat}-p{position}-{arm}.json"
            process = run_arm(arm, out, COLD_ITERS, WARMUP_ITERS, MEASURE_ITERS)
            save_log(f"glbench-r{repeat}-p{position}-{arm}.log", process)
            if process.returncode:
                raise RuntimeError(f"glbench failed: {arm} repeat {repeat}")
            check_dispatch(arm, process.stdout + "\n" + process.stderr)
            stats = session_stats(out)
            PRODUCTION_RECORDS.append({
                "repeat": repeat,
                "position": position,
                "arm": arm,
                **stats,
            })
            print(
                f"{arm:14s} r{repeat} p{position}: {stats['prefill_p50']:8.1f} tok/s | "
                f"P50/P90/P99 {stats['latency_p50_ms']:.3f}/"
                f"{stats['latency_p90_ms']:.3f}/{stats['latency_p99_ms']:.3f} ms",
                flush=True,
            )

    SUMMARY = {}
    for arm in ARM_ENV:
        rows = [row for row in PRODUCTION_RECORDS if row["arm"] == arm]
        SUMMARY[arm] = {
            "prefill_p50_median": statistics.median(row["prefill_p50"] for row in rows),
            "prefill_p90_median": statistics.median(row["prefill_p90"] for row in rows),
            "prefill_p99_median": statistics.median(row["prefill_p99"] for row in rows),
            "latency_p50_median_ms": statistics.median(row["latency_p50_ms"] for row in rows),
            "latency_p90_median_ms": statistics.median(row["latency_p90_ms"] for row in rows),
            "latency_p99_median_ms": statistics.median(row["latency_p99_ms"] for row in rows),
            "latency_mad_median_ms": statistics.median(row["latency_mad_ms"] for row in rows),
            "latency_max_median_ms": statistics.median(row["latency_max_ms"] for row in rows),
            "decode_p50_median": statistics.median(row["decode_p50"] for row in rows),
            "sessions": len(rows),
            "positions": [row["position"] for row in rows],
        }

    PAIRED = []
    for repeat in range(PRODUCTION_REPEATS):
        retained = next(row for row in PRODUCTION_RECORDS
                        if row["repeat"] == repeat and row["arm"] == "retained_n16")
        candidate = next(row for row in PRODUCTION_RECORDS
                         if row["repeat"] == repeat and row["arm"] == "candidate_n32")
        PAIRED.append({
            "repeat": repeat,
            "throughput_delta": candidate["prefill_p50"] / retained["prefill_p50"] - 1.0,
            "absolute_tps": candidate["prefill_p50"] - retained["prefill_p50"],
            "tail_max_delta": candidate["latency_max_ms"] / retained["latency_max_ms"] - 1.0,
            "decode_delta": candidate["decode_p50"] / retained["decode_p50"] - 1.0,
        })

    retained_summary = SUMMARY["retained_n16"]
    candidate_summary = SUMMARY["candidate_n32"]
    median_ratio = (
        candidate_summary["prefill_p50_median"] / retained_summary["prefill_p50_median"] - 1.0
    )
    absolute_gain = (
        candidate_summary["prefill_p50_median"] - retained_summary["prefill_p50_median"]
    )
    tail_delta = (
        candidate_summary["latency_max_median_ms"] / retained_summary["latency_max_median_ms"] - 1.0
    )
    oracle_ok = all(row["oracle"] == "50/50" for row in PRODUCTION_RECORDS)
    all_positive = all(row["throughput_delta"] > 0 for row in PAIRED)
    decode_ok = all(row["decode_delta"] >= -0.05 for row in PAIRED)
    tail_ok = tail_delta <= 0.05
    retention_ok = median_ratio >= 0.01 and all_positive

    if not oracle_ok:
        DECISION = "REJECT"
        VERDICT = "REJECT - exact production oracle failed"
    elif not decode_ok:
        DECISION = "REJECT"
        VERDICT = "REJECT - decode control regressed by more than 5%"
    elif not tail_ok:
        DECISION = "REJECT"
        VERDICT = f"REJECT - session-max tail delta {tail_delta:+.2%} exceeds +5%"
    elif not retention_ok:
        DECISION = "REJECT"
        VERDICT = (
            "REJECT - N32 missed the +1% and all-ten-positive production retention gate"
        )
    elif candidate_summary["prefill_p50_median"] >= 15_000.0:
        DECISION = "GOAL_REACHED"
        VERDICT = "RETAIN - N32 passes production and reaches 15,000 tok/s"
    else:
        DECISION = "RETAIN_BELOW_GOAL"
        VERDICT = "RETAIN - N32 passes production, still below 15,000 tok/s"

    RESULT = {
        "notebook_build": NOTEBOOK_BUILD,
        "gpu": GPU_INFO,
        "model": MODEL_META,
        "kernel_screen": WAVE59_SUMMARY,
        "method": {
            "repeats": PRODUCTION_REPEATS,
            "sessions": len(PRODUCTION_RECORDS),
            "cold_iters": COLD_ITERS,
            "warmup_iters": WARMUP_ITERS,
            "measure_iters": MEASURE_ITERS,
            "order": ORDERS,
            "quant": "Q8_0",
            "prompt_tokens": 244,
            "seed": 42,
            "temperature": 0.0,
            "oracle": "glproc exact 50/50 tokens",
        },
        "summary": SUMMARY,
        "paired": PAIRED,
        "comparison": {
            "ratio_of_session_p50_medians": median_ratio,
            "absolute_gain_tps": absolute_gain,
            "median_paired_delta": statistics.median(row["throughput_delta"] for row in PAIRED),
            "worst_paired_delta": min(row["throughput_delta"] for row in PAIRED),
            "all_positive": all_positive,
            "tail_session_max_delta": tail_delta,
            "oracle_ok": oracle_ok,
            "decode_ok": decode_ok,
        },
        "dispatch": DISPATCH,
        "decision": DECISION,
        "verdict": VERDICT,
    }
    (RESULTS / "wave62-production.json").write_text(
        json.dumps(RESULT, indent=2), encoding="utf-8"
    )
    (RESULTS / "production-records.json").write_text(
        json.dumps(PRODUCTION_RECORDS, indent=2), encoding="utf-8"
    )
    (RESULTS / "SUCCESS.json").write_text(json.dumps({
        "notebook_build": NOTEBOOK_BUILD,
        "decision": DECISION,
        "verdict": VERDICT,
    }, indent=2), encoding="utf-8")
    archive()
    print(json.dumps(RESULT, indent=2))
    print(f"Download archive: {FINAL_ZIP} ({FINAL_ZIP.stat().st_size / 1024:.1f} KB)")
    if DECISION == "REJECT":
        raise RuntimeError(VERDICT)
except Exception:
    fail_phase("wave62-production")
